# R2S2E Pipeline: Rural Bengali Dialect to English via Two-Stage Neural Translation
### Run each cell in order. Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU (or A100 GPU)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["ACCELERATE_DISABLE_RICH"] = "1"
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# INSTALLS
# If dependency errors are present, restart the session and run again.

!pip install sentence-transformers faiss-cpu --quiet
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install -q \
    "numpy==1.26.4" \
    "pandas==2.2.2" \
    "bitsandbytes>=0.46.1" \
    accelerate \
    transformers \
    peft \
    trl \
    sentencepiece \
    sacrebleu \
    rouge-score \
    openpyxl

In [ ]:
# MOUNTING
# Any files to fetch or create will be in your Google Drive after mounting.

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# UPLOADING
# Upload all Vashantor dataset files in JSON format.

from google.colab import files
import os

# run this cell multiple times, once per folder
# each time it will upload whatever files you select
# files accumulate in /content/ across multiple runs

uploaded = files.upload()
for filename in uploaded.keys():
    print(f'Uploaded: {filename} ({len(uploaded[filename])} bytes)')

print("\nAll JSON files in /content/:")
for f in os.listdir('/content'):
    if f.endswith('.json'):
        print(f'  {f}')

Saving Barishal  Validation Translation.json to Barishal  Validation Translation.json
Saving Chittagong Validation Translation.json to Chittagong Validation Translation.json
Saving Mymensingh Validation Translation.json to Mymensingh Validation Translation.json
Saving Noakhali Validation Translation.json to Noakhali Validation Translation.json
Saving Sylhet Validation Translation.json to Sylhet Validation Translation.json
Uploaded: Barishal  Validation Translation.json (106373 bytes)
Uploaded: Chittagong Validation Translation.json (107592 bytes)
Uploaded: Mymensingh Validation Translation.json (107058 bytes)
Uploaded: Noakhali Validation Translation.json (105193 bytes)
Uploaded: Sylhet Validation Translation.json (104335 bytes)

All JSON files in /content/:
  Sylhet Validation Translation.json
  Noakhali Validation Translation.json
  Mymensingh Test Translation.json
  Mymensingh Train Translation.json
  Mymensingh Validation Translation.json
  Chittagong Test Translation.json
  Barish

In [ ]:
# CONFIG
# If there is a dependency error, restart the session and run Cell 2 again.

import torch
import random
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

MAX_LEN    = 128
BATCH_SIZE = 8
EPOCHS     = 50
LR         = 2e-5
PATIENCE   = 3
DRIVE_PATH = '/content/drive/MyDrive/bengali_translation'

# All finished models will be saved at the DRIVE_PATH
import os
os.makedirs(DRIVE_PATH, exist_ok=True)

# Due to the nature of machine learning, using a GPU would be much faster for training.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


In [ ]:
# CREATING DATASETS

import json
import pandas as pd

dialect_keys = {
    'Barishal'   : 'barishal_bangla_speech',
    'Chittagong' : 'chittagong_bangla_speech',
    'Mymensingh' : 'mymensingh_bangla_speech',
    'Noakhali'   : 'noakhali_bangla_speech',
    'Sylhet'     : 'sylhet_bangla_speech',
}

splits = ['Train','Test','Validation']
rows   = []

for dialect,key in dialect_keys.items():
    for split in splits:
        filename=f'{dialect} {split} Translation.json'
        if filename == "Barishal Validation Translation.json":
          filename = "Barishal  Validation Translation.json"
        if filename == "Sylhet Train Translation.json" or filename == "Chittagong Test Translation.json" or filename == "Chittagong Validation Translation.json":
          space = True
        else:
          space = False
        try:
            with open(filename,'r',encoding='utf-8') as f:
                data=json.load(f)
        except FileNotFoundError:
            print(f"Missing: {filename}")
            continue

        loaded=0
        skipped=0
        for item in data:
            try:
                if space == True:
                  rural   = item.get(key+" ",'').strip()
                  standard= item.get('bangla_speech ','').strip()
                  english = item.get('english_speech','').strip()
                  if rural and standard and english:
                      rows.append({
                          'rural'   : rural,
                          'standard': standard,
                          'english' : english,
                          'dialect' : dialect,
                          'split'   : split.lower()
                      })
                      loaded+=1
                  else:
                      skipped+=1
                else:
                  rural   = item.get(key,'').strip()
                  standard= item.get('bangla_speech','').strip()
                  english = item.get('english_speech','').strip()
                  if rural and standard and english:
                      rows.append({
                          'rural'   : rural,
                          'standard': standard,
                          'english' : english,
                          'dialect' : dialect,
                          'split'   : split.lower()
                      })
                      loaded+=1
                  else:
                      skipped+=1
            except Exception:
                skipped+=1
        print(f"Loaded {filename}: {loaded} pairs ({skipped} skipped)")

df=pd.DataFrame(rows)
print(f"\nTotal pairs    : {len(df)}")
print(f"Train          : {len(df[df['split']=='train'])}")
print(f"Validation     : {len(df[df['split']=='validation'])}")
print(f"Test           : {len(df[df['split']=='test'])}")
print(f"\nDialects:")
print(df['dialect'].value_counts())

# save splits
df[df['split']=='train']     .to_csv('vashantor_train.csv',     index=False)
df[df['split']=='validation'].to_csv('vashantor_validation.csv',index=False)
df[df['split']=='test']      .to_csv('vashantor_test.csv',      index=False)
print("\nSaved vashantor_train.csv, vashantor_validation.csv, vashantor_test.csv")

# sample
print("\nSample pairs:")
for _,row in df.head(3).iterrows():
    print(f"  [{row['dialect']}] Rural:    {row['rural']}")
    print(f"            Standard: {row['standard']}")
    print(f"            English:  {row['english']}")
    print()
    print()

Loaded Barishal Train Translation.json: 1875 pairs (0 skipped)
Loaded Barishal Test Translation.json: 374 pairs (2 skipped)
Loaded Barishal  Validation Translation.json: 250 pairs (0 skipped)
Loaded Chittagong Train Translation.json: 1875 pairs (0 skipped)
Loaded Chittagong Test Translation.json: 375 pairs (0 skipped)
Loaded Chittagong Validation Translation.json: 250 pairs (0 skipped)
Loaded Mymensingh Train Translation.json: 1875 pairs (0 skipped)
Loaded Mymensingh Test Translation.json: 374 pairs (2 skipped)
Loaded Mymensingh Validation Translation.json: 250 pairs (0 skipped)
Loaded Noakhali Train Translation.json: 1875 pairs (0 skipped)
Loaded Noakhali Test Translation.json: 374 pairs (2 skipped)
Loaded Noakhali Validation Translation.json: 250 pairs (0 skipped)
Loaded Sylhet Train Translation.json: 1875 pairs (0 skipped)
Loaded Sylhet Test Translation.json: 374 pairs (2 skipped)
Loaded Sylhet Validation Translation.json: 250 pairs (0 skipped)

Total pairs    : 12496
Train         

## Section 1 — Rural Bengali → Standard Bengali: LLM Baseline
Qwen2.5-7B + LoRA. 27.52 BLEU — included as a baseline. mBART-based approaches outperform this.

## Rural Bengali to Standard - LLM Approach

Run all cells to create and test a basic Qwen2.5 Model for the R2S task.




In [ ]:
# CONFIG

DRIVE_PATH  = '/content/drive/MyDrive/bengali_translation'
BASE_MODEL  = 'Qwen/Qwen2.5-7B-Instruct'
SAVE_PATH   = f'{DRIVE_PATH}/qwen_r2s_v3'
MBART_PATH  = f'{DRIVE_PATH}/mbart_r2s'

import pandas as pd

train_df = pd.read_csv('vashantor_train.csv').sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = pd.read_csv('vashantor_validation.csv').sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = pd.read_csv('vashantor_test.csv')

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 9375 | Val: 1250 | Test: 1871


In [ ]:
# MODEL SETUP

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,                    # lower rank = less overfitting
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.1,        # higher dropout = less overfitting
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.eos_token_id
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [ ]:
# PROMPT AND DATASET SETUP

from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a Bengali dialect normalizer. "
    "Convert the input regional Bengali dialect into standard Bengali. "
    "Rules:\n"
    "- Output exactly one sentence\n"
    "- Preserve all meaning, names, numbers, and places\n"
    "- Do not add or remove any information\n"
    "- Do not explain or add commentary"
)

def format_chat(row):
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": row["rural"]},
            {"role": "assistant", "content": row["standard"]},
        ],
        tokenize=False,
    )

train_dataset = Dataset.from_dict({"text": [format_chat(r) for _, r in train_df.iterrows()]})
val_dataset   = Dataset.from_dict({"text": [format_chat(r) for _, r in val_df.iterrows()]})
print(f"Example:\n{train_dataset[0]['text'][:300]}")

Example:
<|im_start|>system
You are a Bengali dialect normalizer. Convert the input regional Bengali dialect into standard Bengali. Rules:
- Output exactly one sentence
- Preserve all meaning, names, numbers, and places
- Do not add or remove any information
- Do not explain or add commentary<|im_end|>
<|im_


In [ ]:
# TRAINING

from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model            = model,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    processing_class = tokenizer,
    args             = SFTConfig(
        output_dir                = f"{SAVE_PATH}_sft",
        num_train_epochs          = 5,           # more epochs, rely on early stopping
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 8,         # effective batch = 32
        learning_rate             = 3e-5,        # lower LR = less overfitting
        warmup_steps              = 50,
        lr_scheduler_type         = 'cosine',
        weight_decay              = 0.01,
        logging_steps             = 10,
        eval_strategy             = 'steps',
        eval_steps                = 50,
        save_strategy             = 'steps',
        save_steps                = 50,
        save_total_limit          = 3,           # keep only 3 best checkpoints
        load_best_model_at_end    = True,
        metric_for_best_model     = 'eval_loss',
        greater_is_better         = False,
        bf16                      = True,
        dataset_text_field        = 'text',
        max_length                = 256,
        report_to                 = "none",
        optim                     = "paged_adamw_32bit",
    ),
)

trainer.train()
trainer.save_model(f"{SAVE_PATH}_sft_best")
tokenizer.save_pretrained(f"{SAVE_PATH}_sft_best")
print("SFT done")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Adding EOS to train dataset:   0%|          | 0/9375 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9375 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1250 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1250 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss
50,0.975400,0.772643
100,0.632000,0.549143
150,0.569600,0.502953
200,0.541100,0.471365
250,0.526000,0.449114
300,0.498300,0.433657
350,0.481500,0.419364
400,0.463800,0.413402
450,0.459400,0.403973
500,0.451300,0.399975


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using

SFT done


In [ ]:
# SAVING + LOADING BEST CHECKPOINT

import torch, os, json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

DRIVE_PATH = '/content/drive/MyDrive/bengali_translation'
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
SAVE_PATH  = f'{DRIVE_PATH}/qwen_r2s_v3'
BEST_CKPT  = f'{SAVE_PATH}_sft/checkpoint-1250'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BEST_CKPT, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.eos_token_id
model = PeftModel.from_pretrained(model, BEST_CKPT, is_trainable=True)
model.eval()
print("Best SFT checkpoint loaded")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Best SFT checkpoint loaded


In [ ]:
# SCORING METRICS

import pandas as pd
from sacrebleu.metrics import BLEU, CHRF

SYSTEM_PROMPT = (
    "You are a Bengali dialect normalizer. "
    "Convert the input regional Bengali dialect into standard Bengali. "
    "Rules:\n"
    "- Output exactly one sentence\n"
    "- Preserve all meaning, names, numbers, and places\n"
    "- Do not add or remove any information\n"
    "- Do not explain or add commentary"
)

def build_prompt(text):
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": text},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

def translate_r2s(text):
    prompt  = build_prompt(text)
    inputs  = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

test_df    = pd.read_csv('vashantor_test.csv')
hypotheses = []
references = []

for _, row in test_df.iterrows():
    hyp = translate_r2s(row['rural'])
    hypotheses.append(hyp)
    references.append(row['standard'])
    print(f"Rural:    {row['rural']}")
    print(f"Expected: {row['standard']}")
    print(f"Got:      {hyp}")
    print()

print("="*50)
print(BLEU().corpus_score(hypotheses, [references]))
print(CHRF().corpus_score(hypotheses, [references]))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Streaming output truncated to the last 5000 lines.

Rural:    এক্কান বিবাহিত মহিলা রাস্তাত দিয়েরে হাডি যার
Expected: একটা বিবাহিত মহিলা রাস্তা দিয়ে হেতে যাচ্ছে
Got:      একজন বিবাহিত মহিলা রাস্তায় দৌড়াচ্ছে

Rural:    আর আম্মু আর ফন্নাফরার হরছ নিত ন চায়
Expected: আমার আম্মু আমার পড়াশোনার খরচ দিতে চায় না
Got:      আমার বড় বোন এখটা ছেলের পরিবারে ঘুরতে চাইছি না

Rural:    আর মা অনেক হষ্ট গরি আরার বড় গইজ্জে
Expected: আমার মা অনেক কষ্ট করে আমাদের বড় করেছেন
Got:      আমার মা অনেক কষ্ট পাইয়ে আমাদের বড় হয়েছে

Rural:    তোয়ার ছোড ভআই ইবা আগত নেশাত আসক্ত আসিল
Expected: তোমার ছোট ভাই  আগে নেশায় আসক্ত ছিল
Got:      তোমার ছোট বোন এখনও নেশায় পড়েছিল

Rural:    ফোয়া ইবা ফুর দিন বারত বাইরত তাহে
Expected: ছেলেটি সারাদিন বাহিরে বাহিরে থাকে
Got:      ছেলেটি প্রথম দিনই ঘরে বাহিরে চলে যায়

Rural:    তোয়ার ফুফুর চাচা তারাও আরে বহুত ভালোবাসে দে
Expected: আমার ফুফু চাচারাও আমাকে অনেক ভালবাসে
Got:      তোমার ছোটো বোনের চাচাগুলি এখন আমাকে অনেক ভালবাসে

Rural:    আর আম্মু আগর তুলনায় এহন বহুত ভালো আসে দে


## Section 2 — Rural Bengali → Standard Bengali: mBART-50 Baseline
Fine-tuned mBART-50 on Vashantor rural→standard pairs. 31.64 BLEU — starting point for all subsequent strategies.

In [ ]:
# CONFIG

DRIVE_PATH      = '/content/drive/MyDrive/bengali_translation'
MODEL_NAME      = 'facebook/mbart-large-50-many-to-many-mmt'
MRASP2_SAVE     = f'{DRIVE_PATH}/mbart_mrasp2_pretrained'
R2S_SAVE        = f'{DRIVE_PATH}/mbart_mrasp2_r2s'
CURRICULUM_SAVE = f'{DRIVE_PATH}/mbart_mrasp2_curriculum'
PASS2_SAVE      = f'{DRIVE_PATH}/mbart_pass2_refinement'
SRC_LANG        = 'bn_IN'
device          = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# LOADING SCORING METRICS AND DATASETS
import os
import torch
import pandas as pd
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from sacrebleu.metrics import BLEU, CHRF
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/bengali_translation'
MODEL_NAME  = 'facebook/mbart-large-50-many-to-many-mmt'
MRASP2_SAVE = f'{DRIVE_PATH}/mbart_mrasp2_pretrained'
R2S_SAVE    = f'{DRIVE_PATH}/mbart_mrasp2_r2s'
SRC_LANG    = 'bn_IN'
device      = 'cuda' if torch.cuda.is_available() else 'cpu'

train_df = pd.read_csv('vashantor_train.csv').dropna().reset_index(drop=True)
val_df   = pd.read_csv('vashantor_validation.csv').dropna().reset_index(drop=True)
test_df  = pd.read_csv('vashantor_test.csv').dropna().reset_index(drop=True)

bleu = BLEU()
chrf = CHRF()

print(f"Device: {device}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Train: 9375 | Val: 1250 | Test: 1871


In [ ]:
# LOADING mBART-50 MODEL

eval_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
eval_tokenizer.src_lang = SRC_LANG
forced_bos = eval_tokenizer.lang_code_to_id[SRC_LANG]

mrasp2_model = MBartForConditionalGeneration.from_pretrained(
    R2S_SAVE, local_files_only=True
).to(device)
mrasp2_model.eval()

baseline_model = MBartForConditionalGeneration.from_pretrained(
    f'{DRIVE_PATH}/mbart_r2s', local_files_only=True
).to(device)
baseline_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
# EVALUVATING mBART-50

import pandas as pd
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from sacrebleu.metrics import BLEU, CHRF
import torch
from google.colab import drive

drive.mount('/content/drive')

MBART_PATH = '/content/drive/MyDrive/bengali_translation/mbart_r2s' # Save model in 'bengali_translation' folder

# Load tokenizer from HuggingFace directly, model from Drive
print("Loading tokenizer from HuggingFace...")
tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
tokenizer.src_lang = "bn_IN"
forced_bos = tokenizer.lang_code_to_id["bn_IN"]

print("Loading model from Drive...")
model = MBartForConditionalGeneration.from_pretrained(MBART_PATH)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
print("Loaded!")

def translate_r2s(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            max_new_tokens=128,
            num_beams=4,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading tokenizer from HuggingFace...
Loading model from Drive...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Loaded!


## Section 3 — Rural Bengali → Standard Bengali: mRASP2 Contrastive Pretraining
mRASP2 pulls rural and standard Bengali closer in representation space before fine-tuning. 33.58 BLEU, +1.94 over baseline.

In [ ]:
# mRASP2 Pipeline

import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    get_cosine_schedule_with_warmup
)
from datasets import Dataset as HFDataset
from sacrebleu.metrics import BLEU, CHRF
from google.colab import drive

drive.mount('/content/drive')

# ── Config ─────────────────────────────────────────────────────────────
DRIVE_PATH  = '/content/drive/MyDrive/bengali_translation'
MODEL_NAME  = 'facebook/mbart-large-50-many-to-many-mmt'
MRASP2_SAVE = f'{DRIVE_PATH}/mbart_mrasp2_pretrained'
R2S_SAVE    = f'{DRIVE_PATH}/mbart_mrasp2_r2s'
SRC_LANG    = 'bn_IN'
device      = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Load Data ──────────────────────────────────────────────────────────
train_df = pd.read_csv('vashantor_train.csv').dropna().reset_index(drop=True)
val_df   = pd.read_csv('vashantor_validation.csv').dropna().reset_index(drop=True)
test_df  = pd.read_csv('vashantor_test.csv').dropna().reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ── mRASP2 Model ───────────────────────────────────────────────────────
class mRASP2Model(nn.Module):
    def __init__(self, model_name, temperature=0.07):
        super().__init__()
        self.mbart       = MBartForConditionalGeneration.from_pretrained(model_name)
        self.temperature = temperature
        hidden_size      = self.mbart.config.d_model
        self.projector   = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 256)
        )

    def encode(self, input_ids, attention_mask):
        encoder_out = self.mbart.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden = encoder_out.last_hidden_state
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1)
        return self.projector(pooled)

    def forward(
        self,
        input_ids, attention_mask, labels,
        anchor_ids, anchor_mask,
        pos_ids, pos_mask,
        hard_neg_ids=None, hard_neg_mask=None,
        lm_weight=1.0, cl_weight=0.5
    ):
        lm_out  = self.mbart(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        lm_loss = lm_out.loss

        anchor_emb = F.normalize(self.encode(anchor_ids, anchor_mask), dim=-1)
        pos_emb    = F.normalize(self.encode(pos_ids,    pos_mask),    dim=-1)
        sim_matrix = torch.matmul(anchor_emb, pos_emb.T) / self.temperature
        batch_size = anchor_emb.size(0)
        cl_labels  = torch.arange(batch_size, device=anchor_emb.device)
        cl_loss    = (
            F.cross_entropy(sim_matrix,   cl_labels) +
            F.cross_entropy(sim_matrix.T, cl_labels)
        ) / 2

        if hard_neg_ids is not None:
            hard_emb      = F.normalize(self.encode(hard_neg_ids, hard_neg_mask), dim=-1)
            hard_sim      = (anchor_emb * hard_emb).sum(-1) / self.temperature
            hard_neg_loss = F.relu(hard_sim + 0.5).mean()
            cl_loss       = cl_loss + 0.1 * hard_neg_loss

        total_loss = lm_weight * lm_loss + cl_weight * cl_loss
        return total_loss, lm_loss.item(), cl_loss.item()


# ── Dataset ────────────────────────────────────────────────────────────
class DialectDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128, use_hard_negatives=True):
        self.df                 = df.reset_index(drop=True)
        self.tokenizer          = tokenizer
        self.max_len            = max_len
        self.use_hard_negatives = use_hard_negatives
        if 'dialect' in df.columns:
            self.dialect_groups = df.groupby('dialect').indices
        else:
            self.dialect_groups = None

    def tokenize(self, text):
        return self.tokenizer(
            text, max_length=self.max_len,
            truncation=True, padding='max_length',
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        src    = self.tokenize(row['rural'])
        tgt    = self.tokenizer(
            text_target=row['standard'],
            max_length=self.max_len,
            truncation=True, padding='max_length',
            return_tensors='pt'
        )
        labels = tgt['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        anchor = self.tokenize(row['rural'])
        pos    = self.tokenize(row['standard'])

        hard_neg_ids  = None
        hard_neg_mask = None
        if self.use_hard_negatives and self.dialect_groups is not None:
            dialect = row.get('dialect', None)
            if dialect and dialect in self.dialect_groups:
                candidates = [i for i in self.dialect_groups[dialect] if i != idx]
                if candidates:
                    hard      = self.tokenize(self.df.iloc[random.choice(candidates)]['rural'])
                    hard_neg_ids  = hard['input_ids'].squeeze()
                    hard_neg_mask = hard['attention_mask'].squeeze()

        return {
            'input_ids':      src['input_ids'].squeeze(),
            'attention_mask': src['attention_mask'].squeeze(),
            'labels':         labels,
            'anchor_ids':     anchor['input_ids'].squeeze(),
            'anchor_mask':    anchor['attention_mask'].squeeze(),
            'pos_ids':        pos['input_ids'].squeeze(),
            'pos_mask':       pos['attention_mask'].squeeze(),
            'hard_neg_ids':   hard_neg_ids,
            'hard_neg_mask':  hard_neg_mask,
        }


def collate_fn(batch):
    keys = ['input_ids', 'attention_mask', 'labels',
            'anchor_ids', 'anchor_mask', 'pos_ids', 'pos_mask']
    out = {k: torch.stack([b[k] for b in batch]) for k in keys}
    if all(b['hard_neg_ids'] is not None for b in batch):
        out['hard_neg_ids']  = torch.stack([b['hard_neg_ids']  for b in batch])
        out['hard_neg_mask'] = torch.stack([b['hard_neg_mask'] for b in batch])
    else:
        out['hard_neg_ids']  = None
        out['hard_neg_mask'] = None
    return out


# ── Stage 1: mRASP2 Contrastive Pretraining ───────────────────────────
print("\n" + "="*60)
print("STAGE 1 — mRASP2 Contrastive Pretraining")
print("="*60)

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG

model = mRASP2Model(MODEL_NAME, temperature=0.07).to(device)

train_dataset = DialectDataset(train_df, tokenizer, use_hard_negatives=True)
val_dataset   = DialectDataset(val_df,   tokenizer, use_hard_negatives=False)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True,
                           collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset, batch_size=8, shuffle=False,
                           collate_fn=collate_fn, num_workers=2)

optimizer = AdamW([
    {'params': model.mbart.parameters(),     'lr': 1e-5},
    {'params': model.projector.parameters(), 'lr': 1e-4},
], weight_decay=0.01)

PRETRAIN_EPOCHS = 3
GRAD_ACCUM      = 4
total_steps     = (len(train_loader) // GRAD_ACCUM) * PRETRAIN_EPOCHS
warmup_steps    = total_steps // 10
scheduler       = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler          = torch.cuda.amp.GradScaler()
best_val_loss   = float('inf')

print(f"Steps: {total_steps} | Warmup: {warmup_steps}")

for epoch in range(PRETRAIN_EPOCHS):
    model.train()
    total_loss, total_lm, total_cl = 0, 0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        anchor_ids     = batch['anchor_ids'].to(device)
        anchor_mask    = batch['anchor_mask'].to(device)
        pos_ids        = batch['pos_ids'].to(device)
        pos_mask       = batch['pos_mask'].to(device)
        hard_neg_ids   = batch['hard_neg_ids'].to(device)  if batch['hard_neg_ids']  is not None else None
        hard_neg_mask  = batch['hard_neg_mask'].to(device) if batch['hard_neg_mask'] is not None else None

        with torch.cuda.amp.autocast():
            loss, lm_l, cl_l = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels,
                anchor_ids=anchor_ids, anchor_mask=anchor_mask,
                pos_ids=pos_ids, pos_mask=pos_mask,
                hard_neg_ids=hard_neg_ids, hard_neg_mask=hard_neg_mask,
                lm_weight=1.0, cl_weight=0.5
            )
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM
        total_lm   += lm_l
        total_cl   += cl_l

        if step % 100 == 0:
            print(f"  Epoch {epoch+1} step {step}/{len(train_loader)} "
                  f"| loss {total_loss/(step+1):.4f} "
                  f"| lm {total_lm/(step+1):.4f} "
                  f"| cl {total_cl/(step+1):.4f}")

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            with torch.cuda.amp.autocast():
                loss, _, _ = model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch['labels'].to(device),
                    anchor_ids=batch['anchor_ids'].to(device),
                    anchor_mask=batch['anchor_mask'].to(device),
                    pos_ids=batch['pos_ids'].to(device),
                    pos_mask=batch['pos_mask'].to(device),
                    lm_weight=1.0, cl_weight=0.5
                )
            val_loss += loss.item()

    avg_val = val_loss / len(val_loader)
    print(f"Epoch {epoch+1} | val_loss: {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        os.makedirs(MRASP2_SAVE, exist_ok=True)
        model.mbart.save_pretrained(MRASP2_SAVE)
        tokenizer.save_pretrained(MRASP2_SAVE)
        print(f"  Saved to {MRASP2_SAVE}")

del model
torch.cuda.empty_cache()
print("Stage 1 complete!")


# ── Stage 2: Fine-tune on r2s ──────────────────────────────────────────
print("\n" + "="*60)
print("STAGE 2 — Fine-tune on r2s task")
print("="*60)

ft_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
ft_tokenizer.src_lang = SRC_LANG
forced_bos   = ft_tokenizer.lang_code_to_id[SRC_LANG]

ft_model = MBartForConditionalGeneration.from_pretrained(
    MRASP2_SAVE, local_files_only=True
)

MAX_LEN = 128

def tokenize_r2s(batch):
    model_inputs = ft_tokenizer(
        batch['rural'], max_length=MAX_LEN,
        truncation=True, padding='max_length'
    )
    labels = ft_tokenizer(
        text_target=batch['standard'],
        max_length=MAX_LEN, truncation=True, padding='max_length'
    )
    model_inputs['labels'] = [
        [-100 if t == ft_tokenizer.pad_token_id else t for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

train_hf  = HFDataset.from_dict({'rural': train_df['rural'].tolist(),
                                  'standard': train_df['standard'].tolist()})
val_hf    = HFDataset.from_dict({'rural': val_df['rural'].tolist(),
                                  'standard': val_df['standard'].tolist()})
train_tok = train_hf.map(tokenize_r2s, batched=True, batch_size=64,
                          remove_columns=['rural', 'standard'])
val_tok   = val_hf.map(tokenize_r2s, batched=True, batch_size=64,
                        remove_columns=['rural', 'standard'])

ft_args = Seq2SeqTrainingArguments(
    output_dir=R2S_SAVE,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    warmup_steps=100,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=True,
    logging_steps=20,
    report_to='none',
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    model=ft_model,
    args=ft_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=ft_tokenizer,
    data_collator=DataCollatorForSeq2Seq(ft_tokenizer, model=ft_model, padding=True),
)

print("Starting fine-tuning...")
trainer.train()
trainer.save_model(R2S_SAVE)
ft_tokenizer.save_pretrained(R2S_SAVE)
print(f"Saved to {R2S_SAVE}")

del ft_model, trainer
torch.cuda.empty_cache()


# ── Evaluation ─────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATION")
print("="*60)

eval_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
eval_tokenizer.src_lang = SRC_LANG
forced_bos = eval_tokenizer.lang_code_to_id[SRC_LANG]

mrasp2_model = MBartForConditionalGeneration.from_pretrained(
    R2S_SAVE, local_files_only=True
).to(device)
mrasp2_model.eval()

baseline_model = MBartForConditionalGeneration.from_pretrained(
    f'{DRIVE_PATH}/mbart_r2s', local_files_only=True
).to(device)
baseline_model.eval()

def translate(model, text):
    inputs = eval_tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            max_new_tokens=128,
            num_beams=4
        )
    return eval_tokenizer.decode(outputs[0], skip_special_tokens=True)

hyp_mrasp2   = []
hyp_baseline = []
references   = []

for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_mrasp2.append(translate(mrasp2_model,   row['rural']))
    hyp_baseline.append(translate(baseline_model, row['rural']))
    references.append(row['standard'])
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}")

bleu = BLEU()
chrf = CHRF()

print("\n" + "="*60)
for label, hyps in [
    ("mBART baseline r2s", hyp_baseline),
    ("mBART mRASP2 r2s",   hyp_mrasp2),
]:
    b = bleu.corpus_score(hyps, [references])
    c = chrf.corpus_score(hyps, [references])
    print(f"\n{label}:")
    print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")
    print(f"  {b}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: 9375 | Val: 1250 | Test: 1871

STAGE 1 — mRASP2 Contrastive Pretraining


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/bengali_translation/mbart_mrasp2_r2s

EVALUATION


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871


mBART baseline r2s:
  BLEU: 31.64 | chrF: 61.71
  BLEU = 31.64 62.9/40.2/26.0/17.0 (BP = 0.973 ratio = 0.974 hyp_len = 13263 ref_len = 13622)

mBART mRASP2 r2s:
  BLEU: 33.58 | chrF: 64.26
  BLEU = 33.58 63.9/41.3/27.5/18.4 (BP = 0.988 ratio = 0.988 hyp_len = 13458 ref_len = 13622)


In [ ]:
# BEAM VOTE

def translate_beam_vote(model1, model2, text):
    inputs = eval_tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        out1 = model1.generate(
            **inputs, forced_bos_token_id=forced_bos,
            max_new_tokens=128, num_beams=4,
            output_scores=True, return_dict_in_generate=True,
        )
        out2 = model2.generate(
            **inputs, forced_bos_token_id=forced_bos,
            max_new_tokens=128, num_beams=4,
            output_scores=True, return_dict_in_generate=True,
        )
    if out1.sequences_scores.item() >= out2.sequences_scores.item():
        return eval_tokenizer.decode(out1.sequences[0], skip_special_tokens=True)
    else:
        return eval_tokenizer.decode(out2.sequences[0], skip_special_tokens=True)

hyp_beam_vote = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_beam_vote.append(translate_beam_vote(baseline_model, mrasp2_model, row['rural']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}")

b = bleu.corpus_score(hyp_beam_vote, [references])
c = chrf.corpus_score(hyp_beam_vote, [references])
print(f"Ensemble (beam vote): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
Ensemble (beam vote): BLEU: 33.91 | chrF: 64.30


In [ ]:
# LOGIT AVERAGING

from transformers import LogitsProcessor, LogitsProcessorList
from transformers.modeling_outputs import BaseModelOutput
from torch.amp import autocast
import torch.nn.functional as F

class EnsembleLogitsProcessor(LogitsProcessor):
    def __init__(self, model2, encoder_hidden2, attention_mask, alpha=0.5, num_beams=4):
        self.model2         = model2
        self.enc_hidden2    = encoder_hidden2
        self.attention_mask = attention_mask
        self.alpha          = alpha
        self.num_beams      = num_beams
        self._expanded      = False

    def __call__(self, input_ids, scores):
        if not self._expanded:
            self.enc_hidden2    = self.enc_hidden2.repeat_interleave(self.num_beams, dim=0)
            self.attention_mask = self.attention_mask.repeat_interleave(self.num_beams, dim=0)
            self._expanded      = True

        with torch.inference_mode(), autocast('cuda'):
            out2 = self.model2(
                attention_mask    = self.attention_mask,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(
                    last_hidden_state=self.enc_hidden2
                ),
                use_cache = False,
            )

        lp1 = F.log_softmax(scores,                         dim=-1)
        lp2 = F.log_softmax(out2.logits[:, -1, :].float(),  dim=-1)

        blended = self.alpha * lp1 + (1 - self.alpha) * lp2
        return blended


@torch.inference_mode()
def translate_ensemble_generate(model1, model2, texts, alpha=0.5, batch_size=64, num_beams=4):
    all_outputs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = eval_tokenizer(
            batch_texts, return_tensors='pt',
            truncation=True, max_length=128,
            padding=True
        ).to(device)

        with autocast('cuda'):
            enc1 = model1.model.encoder(**inputs)
            enc2 = model2.model.encoder(**inputs)

        processor = EnsembleLogitsProcessor(
            model2          = model2,
            encoder_hidden2 = enc2.last_hidden_state.clone(),
            attention_mask  = inputs['attention_mask'].clone(),
            alpha           = alpha,
            num_beams       = num_beams,
        )

        outputs = model1.generate(
            **inputs,
            encoder_outputs     = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id = forced_bos,
            max_new_tokens      = 128,
            num_beams           = num_beams,
            logits_processor    = LogitsProcessorList([processor]),
            early_stopping      = True,
        )

        for seq in outputs:
            all_outputs.append(
                eval_tokenizer.decode(seq, skip_special_tokens=True)
            )

        if i % 200 == 0:
            print(f"  {i}/{len(texts)}")

        torch.cuda.empty_cache()

    return all_outputs


# Run
hyp_ensemble_logit = translate_ensemble_generate(
    baseline_model, mrasp2_model,
    texts, alpha=0.5, batch_size=64, num_beams=4
)

b = bleu.corpus_score(hyp_ensemble_logit, [references])
c = chrf.corpus_score(hyp_ensemble_logit, [references])
print(f"Ensemble (logit avg): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

  0/1871
  1600/1871
Ensemble (logit avg): BLEU: 34.78 | chrF: 64.67


In [ ]:
# ORACLE CEILING

hyp_ensemble_vote = []

for i in range(len(test_df)):
    s1 = chrf.sentence_score(hyp_baseline[i], [references[i]]).score
    s2 = chrf.sentence_score(hyp_mrasp2[i],   [references[i]]).score
    hyp_ensemble_vote.append(hyp_baseline[i] if s1 >= s2 else hyp_mrasp2[i])

b = bleu.corpus_score(hyp_ensemble_vote, [references])
c = chrf.corpus_score(hyp_ensemble_vote, [references])
print(f"Ensemble (vote): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Ensemble (vote): BLEU: 37.33 | chrF: 67.28


In [ ]:
# ORACLE EXAMINATION

mrasp2_wins = sum(
    chrf.sentence_score(hyp_mrasp2[i], [references[i]]).score >
    chrf.sentence_score(hyp_baseline[i], [references[i]]).score
    for i in range(len(test_df))
)
print(f"mRASP2 wins: {mrasp2_wins}/{len(test_df)} ({100*mrasp2_wins/len(test_df):.1f}%)")

mRASP2 wins: 780/1871 (41.7%)


## Section 4 — Rural Bengali → Standard Bengali: Curriculum Learning
Training examples ordered easy→hard by chrF similarity. Starts from mRASP2 checkpoint. 35.42 BLEU, +3.78 over baseline.

In [ ]:
# CURRICULUM LEARNING

print("\n" + "="*60)
print("CURRICULUM LEARNING FROM mRASP2")
print("="*60)

from sacrebleu.metrics import CHRF
from torch.utils.data import Subset
from datasets import Dataset as HFDataset
import numpy as np
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
CURRICULUM_SAVE = f'{DRIVE_PATH}/mbart_mrasp2_curriculum'

# ── Step 1: Score difficulty of each training sample ──────────────────
print("Scoring training sample difficulty...")
chrf_scorer = CHRF()

difficulty_scores = []
for _, row in train_df.iterrows():
    score = chrf_scorer.sentence_score(
        row['rural'], [row['standard']]
    ).score
    difficulty_scores.append(score)

train_df['difficulty'] = difficulty_scores
# High chrF = similar = easy (low difficulty rank)
# Low chrF  = different = hard (high difficulty rank)
train_df['difficulty_rank'] = train_df['difficulty'].rank(ascending=False)

print(f"Easy sample (high chrF):  {train_df.nlargest(1, 'difficulty').iloc[0]['rural']}")
print(f"Hard sample (low chrF):   {train_df.nsmallest(1, 'difficulty').iloc[0]['rural']}")
print(f"Mean chrF: {train_df['difficulty'].mean():.2f}")
print(f"Std chrF:  {train_df['difficulty'].std():.2f}")

# ── Step 2: Split into curriculum buckets ─────────────────────────────
train_sorted = train_df.sort_values('difficulty', ascending=False).reset_index(drop=True)

n = len(train_sorted)
easy_df   = train_sorted.iloc[:int(n * 0.33)]   # top 33% most similar
medium_df = train_sorted.iloc[:int(n * 0.66)]   # top 66%
full_df   = train_sorted                          # all data

print(f"\nCurriculum buckets:")
print(f"  Easy:   {len(easy_df)} samples (mean chrF: {easy_df['difficulty'].mean():.2f})")
print(f"  Medium: {len(medium_df)} samples (mean chrF: {medium_df['difficulty'].mean():.2f})")
print(f"  Full:   {len(full_df)} samples (mean chrF: {full_df['difficulty'].mean():.2f})")

# ── Step 3: Tokenization ───────────────────────────────────────────────
curr_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
curr_tokenizer.src_lang = SRC_LANG
MAX_LEN = 128

def tokenize_df(df):
    hf = HFDataset.from_dict({
        'rural':    df['rural'].tolist(),
        'standard': df['standard'].tolist()
    })
    def tokenize(batch):
        model_inputs = curr_tokenizer(
            batch['rural'], max_length=MAX_LEN,
            truncation=True, padding='max_length'
        )
        labels = curr_tokenizer(
            text_target=batch['standard'],
            max_length=MAX_LEN, truncation=True, padding='max_length'
        )
        model_inputs['labels'] = [
            [-100 if t == curr_tokenizer.pad_token_id else t for t in label]
            for label in labels['input_ids']
        ]
        return model_inputs
    return hf.map(tokenize, batched=True, batch_size=64,
                  remove_columns=['rural', 'standard'])

print("\nTokenizing curriculum buckets...")
easy_tok   = tokenize_df(easy_df)
medium_tok = tokenize_df(medium_df)
full_tok   = tokenize_df(full_df)
val_tok    = tokenize_df(val_df)

# ── Step 4: Curriculum Training ────────────────────────────────────────
curr_model = MBartForConditionalGeneration.from_pretrained(
    MRASP2_SAVE, local_files_only=True
)

collator = DataCollatorForSeq2Seq(curr_tokenizer, model=curr_model, padding=True)

curriculum_stages = [
    ('easy',   easy_tok,   2),
    ('medium', medium_tok, 2),
    ('full',   full_tok,   3),
]

for stage_name, stage_data, n_epochs in curriculum_stages:
    print(f"\n{'='*40}")
    print(f"Stage: {stage_name} | Samples: {len(stage_data)} | Epochs: {n_epochs}")
    print(f"{'='*40}")

    stage_args = Seq2SeqTrainingArguments(
        output_dir           = f'{CURRICULUM_SAVE}_{stage_name}',
        num_train_epochs     = n_epochs,
        per_device_train_batch_size = 8,
        per_device_eval_batch_size  = 8,
        gradient_accumulation_steps = 4,
        learning_rate        = 3e-5,
        warmup_steps         = 50,
        weight_decay         = 0.01,
        lr_scheduler_type    = 'cosine',
        predict_with_generate= True,
        generation_max_length= 128,
        generation_num_beams = 4,
        eval_strategy        = 'epoch',
        save_strategy        = 'epoch',
        load_best_model_at_end    = True,
        metric_for_best_model     = 'eval_loss',
        fp16                 = True,
        logging_steps        = 20,
        report_to            = 'none',
        save_total_limit     = 1,
        label_smoothing_factor = 0.1,
    )

    trainer = Seq2SeqTrainer(
        model            = curr_model,
        args             = stage_args,
        train_dataset    = stage_data,
        eval_dataset     = val_tok,
        processing_class = curr_tokenizer,
        data_collator    = collator,
    )

    trainer.train()

    # Update model reference to best checkpoint for next stage
    curr_model = trainer.model
    print(f"Stage {stage_name} complete")

# ── Save Final Model ───────────────────────────────────────────────────
os.makedirs(CURRICULUM_SAVE, exist_ok=True)
curr_model.save_pretrained(CURRICULUM_SAVE)
curr_tokenizer.save_pretrained(CURRICULUM_SAVE)
print(f"\nSaved to {CURRICULUM_SAVE}")

del curr_model, trainer
torch.cuda.empty_cache()

# ── Evaluate ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING CURRICULUM MODEL")
print("="*60)

curr_eval_model = MBartForConditionalGeneration.from_pretrained(
    CURRICULUM_SAVE, local_files_only=True
).to(device)
curr_eval_model.eval()

hyp_curriculum = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_curriculum.append(translate(curr_eval_model, row['rural']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}")

b = bleu.corpus_score(hyp_curriculum, [references])
c = chrf.corpus_score(hyp_curriculum, [references])
print(f"\nCurriculum mRASP2:")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")


CURRICULUM LEARNING FROM mRASP2
Scoring training sample difficulty...
Easy sample (high chrF):  তোমার কি মন খারাপ?
Hard sample (low chrF):   ইগু কিলা ওয়!
Mean chrF: 41.57
Std chrF:  20.44

Curriculum buckets:
  Easy:   3093 samples (mean chrF: 65.55)
  Medium: 6187 samples (mean chrF: 52.25)
  Full:   9375 samples (mean chrF: 41.57)

Tokenizing curriculum buckets...


Map:   0%|          | 0/3093 [00:00<?, ? examples/s]

Map:   0%|          | 0/6187 [00:00<?, ? examples/s]

Map:   0%|          | 0/9375 [00:00<?, ? examples/s]

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

...

Stage full complete


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved to /content/drive/MyDrive/bengali_translation/mbart_mrasp2_curriculum

EVALUATING CURRICULUM MODEL


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

NameError: name 'translate' is not defined

In [ ]:
# EVALUVATIONS

from transformers import LogitsProcessor, LogitsProcessorList
from transformers.modeling_outputs import BaseModelOutput
from torch.amp import autocast
import torch.nn.functional as F

class EnsembleLogitsProcessor(LogitsProcessor):
    def __init__(self, model2, encoder_hidden2, attention_mask, alpha=0.5, num_beams=4):
        self.model2         = model2
        self.enc_hidden2    = encoder_hidden2
        self.attention_mask = attention_mask
        self.alpha          = alpha
        self.num_beams      = num_beams
        self._expanded      = False

    def __call__(self, input_ids, scores):
        if not self._expanded:
            self.enc_hidden2    = self.enc_hidden2.repeat_interleave(self.num_beams, dim=0)
            self.attention_mask = self.attention_mask.repeat_interleave(self.num_beams, dim=0)
            self._expanded      = True

        with torch.inference_mode(), autocast('cuda'):
            out2 = self.model2(
                attention_mask    = self.attention_mask,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(
                    last_hidden_state=self.enc_hidden2
                ),
                use_cache = False,
            )

        lp1 = F.log_softmax(scores,                         dim=-1)
        lp2 = F.log_softmax(out2.logits[:, -1, :].float(),  dim=-1)
        return self.alpha * lp1 + (1 - self.alpha) * lp2


@torch.inference_mode()
def translate_ensemble_generate(model1, model2, texts, alpha=0.5, batch_size=64, num_beams=4):
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = eval_tokenizer(
            batch_texts, return_tensors='pt',
            truncation=True, max_length=128,
            padding=True
        ).to(device)

        with autocast('cuda'):
            enc1 = model1.model.encoder(**inputs)
            enc2 = model2.model.encoder(**inputs)

        processor = EnsembleLogitsProcessor(
            model2          = model2,
            encoder_hidden2 = enc2.last_hidden_state.clone(),
            attention_mask  = inputs['attention_mask'].clone(),
            alpha           = alpha,
            num_beams       = num_beams,
        )

        outputs = model1.generate(
            **inputs,
            encoder_outputs     = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id = forced_bos,
            max_new_tokens      = 128,
            num_beams           = num_beams,
            logits_processor    = LogitsProcessorList([processor]),
            early_stopping      = True,
        )

        for seq in outputs:
            all_outputs.append(
                eval_tokenizer.decode(seq, skip_special_tokens=True)
            )

        if i % 200 == 0:
            print(f"  {i}/{len(texts)}")

        torch.cuda.empty_cache()

    return all_outputs


texts = test_df['rural'].tolist()

# Curriculum + mRASP2
print("Evaluating Curriculum + mRASP2 logit ensemble...")
hyp_curr_mrasp2 = translate_ensemble_generate(
    curr_eval_model, mrasp2_model,
    texts, alpha=0.5, batch_size=64, num_beams=4
)
b = bleu.corpus_score(hyp_curr_mrasp2, [references])
c = chrf.corpus_score(hyp_curr_mrasp2, [references])
print(f"Curriculum + mRASP2: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Curriculum + baseline
print("\nEvaluating Curriculum + Baseline logit ensemble...")
hyp_curr_baseline = translate_ensemble_generate(
    curr_eval_model, baseline_model,
    texts, alpha=0.5, batch_size=64, num_beams=4
)
b = bleu.corpus_score(hyp_curr_baseline, [references])
c = chrf.corpus_score(hyp_curr_baseline, [references])
print(f"Curriculum + Baseline: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Evaluating Curriculum + mRASP2 logit ensemble...
  0/1871
  1600/1871
Curriculum + mRASP2: BLEU: 35.91 | chrF: 66.18

Evaluating Curriculum + Baseline logit ensemble...
  0/1871
  1600/1871
Curriculum + Baseline: BLEU: 35.01 | chrF: 65.65


In [ ]:
from torch.amp import autocast
from transformers.modeling_outputs import BaseModelOutput
from transformers import LogitsProcessor, LogitsProcessorList
import torch.nn.functional as F

class EnsembleLogitsProcessor(LogitsProcessor):
    def __init__(self, model2, encoder_hidden2, attention_mask, alpha=0.5, num_beams=4):
        self.model2         = model2
        self.enc_hidden2    = encoder_hidden2
        self.attention_mask = attention_mask
        self.alpha          = alpha
        self.num_beams      = num_beams
        self._expanded      = False

    def __call__(self, input_ids, scores):
        if not self._expanded:
            self.enc_hidden2    = self.enc_hidden2.repeat_interleave(self.num_beams, dim=0)
            self.attention_mask = self.attention_mask.repeat_interleave(self.num_beams, dim=0)
            self._expanded      = True

        with torch.inference_mode(), autocast('cuda'):
            out2 = self.model2(
                attention_mask    = self.attention_mask,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(
                    last_hidden_state=self.enc_hidden2
                ),
                use_cache = False,
            )

        lp1 = F.log_softmax(scores,                         dim=-1)
        lp2 = F.log_softmax(out2.logits[:, -1, :].float(),  dim=-1)
        return self.alpha * lp1 + (1 - self.alpha) * lp2

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

torch.cuda.empty_cache()

In [ ]:
del baseline_model
torch.cuda.empty_cache()

In [ ]:

import numpy as np

## Section 5 — Rural Bengali → Standard Bengali: Pass 2 Refinement (Negative Result)
Confidence-gated second-pass refinement. 35.86 BLEU — did not improve over the ensemble. Included for completeness.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset as HFDataset

# Skip recalculation — already computed
threshold = -0.2147

In [ ]:
'''# ── Pass 2 Refinement Model ────────────────────────────────────────────
from datasets import Dataset as HFDataset  # FIX 1: import HFDataset

print("\n" + "="*60)
print("PASS 2 REFINEMENT MODEL")
print("="*60)

# ── Step 1: Filter training data to chrF > 50 ─────────────────────────
print("Filtering training data...")
chrf_scorer = CHRF()

if 'difficulty' not in train_df.columns:
    difficulty_scores = []
    for _, row in train_df.iterrows():
        score = chrf_scorer.sentence_score(
            row['rural'], [row['standard']]
        ).score
        difficulty_scores.append(score)
    train_df['difficulty'] = difficulty_scores

refinement_df = train_df[train_df['difficulty'] > 50].reset_index(drop=True)
print(f"Refinement training samples (chrF>50): {len(refinement_df)}/{len(train_df)}")
print(f"Mean chrF of refinement set: {refinement_df['difficulty'].mean():.2f}")

# ── Step 2: Generate Pass 1 outputs for refinement training set ────────
print("\nGenerating Pass 1 outputs for refinement training data...")

# Load curriculum model if not already loaded
if 'curr_eval_model' not in dir():
    curr_eval_model = MBartForConditionalGeneration.from_pretrained(
        CURRICULUM_SAVE, local_files_only=True
    ).to(device)
    curr_eval_model.eval()

if 'mrasp2_model' not in dir():
    mrasp2_model = MBartForConditionalGeneration.from_pretrained(
        R2S_SAVE, local_files_only=True
    ).to(device)
    mrasp2_model.eval()

# FIX 2: Removed generate_pass1_fast (dead code). Using generate_pass1_batch
# throughout, which already exists elsewhere and takes two models.

# Generate Pass 1 for refinement training set
refinement_texts   = refinement_df['rural'].tolist()
refinement_standards = refinement_df['standard'].tolist()

pass1_refinement, scores_refinement = generate_pass1_batch(
    curr_eval_model, mrasp2_model, refinement_texts, batch_size=64)

# ── Step 3: Calibrate confidence threshold on validation set ──────────
print("\nCalibrating confidence threshold on validation set...")
val_texts = val_df['rural'].tolist()
_, val_scores = generate_pass1_batch(
    curr_eval_model, mrasp2_model, val_texts
)

threshold = np.percentile(val_scores, 70)
print(f"70th percentile beam score threshold: {threshold:.4f}")

# ── Step 4: Build Pass 2 training data ────────────────────────────────
# Input: rural + pass1 output | Target: standard
print("\nBuilding Pass 2 training data...")

pass2_inputs  = []
pass2_targets = []

for i in range(len(refinement_texts)):
    combined = f"{refinement_texts[i]} | {pass1_refinement[i]}"
    pass2_inputs.append(combined)
    pass2_targets.append(refinement_standards[i])

print(f"Pass 2 training samples: {len(pass2_inputs)}")
print(f"Sample input:  {pass2_inputs[0]}")
print(f"Sample target: {pass2_targets[0]}")

# ── Step 5: Tokenize Pass 2 data ──────────────────────────────────────
p2_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
p2_tokenizer.src_lang = SRC_LANG
MAX_LEN = 256  # longer to fit both rural and pass1

def tokenize_pass2(batch):
    model_inputs = p2_tokenizer(
        batch['input'], max_length=MAX_LEN,
        truncation=True, padding='max_length'
    )
    labels = p2_tokenizer(
        text_target=batch['target'],
        max_length=128, truncation=True, padding='max_length'
    )
    model_inputs['labels'] = [
        [-100 if t == p2_tokenizer.pad_token_id else t for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

# Build val pass1 outputs for pass2 val set
pass1_val, _ = generate_pass1_batch(
    curr_eval_model, mrasp2_model, val_texts
)
val_pass2_inputs = [
    f"{val_texts[i]} | {pass1_val[i]}"
    for i in range(len(val_texts))
]

train_p2_hf = HFDataset.from_dict({
    'input':  pass2_inputs,
    'target': pass2_targets
})
val_p2_hf = HFDataset.from_dict({
    'input':  val_pass2_inputs,
    'target': val_df['standard'].tolist()
})

train_p2_tok = train_p2_hf.map(
    tokenize_pass2, batched=True, batch_size=64,
    remove_columns=['input', 'target']
)
val_p2_tok = val_p2_hf.map(
    tokenize_pass2, batched=True, batch_size=64,
    remove_columns=['input', 'target']
)

# ── Step 6: Fine-tune Pass 2 model ────────────────────────────────────
print("\nFine-tuning Pass 2 refinement model...")
PASS2_SAVE = f'{DRIVE_PATH}/mbart_pass2_refinement'

p2_model = MBartForConditionalGeneration.from_pretrained(
    CURRICULUM_SAVE, local_files_only=True
)

p2_args = Seq2SeqTrainingArguments(
    output_dir                  = PASS2_SAVE,
    num_train_epochs            = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 4,
    learning_rate               = 5e-6,   # very low — conservative corrections
    warmup_steps                = 50,
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    predict_with_generate       = True,
    generation_max_length       = 128,
    generation_num_beams        = 4,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    fp16                        = True,
    logging_steps               = 20,
    report_to                   = 'none',
    save_total_limit            = 1,
    label_smoothing_factor      = 0.15,
)

p2_trainer = Seq2SeqTrainer(
    model            = p2_model,
    args             = p2_args,
    train_dataset    = train_p2_tok,
    eval_dataset     = val_p2_tok,
    processing_class = p2_tokenizer,
    data_collator    = DataCollatorForSeq2Seq(
        p2_tokenizer, model=p2_model, padding=True
    ),
)

p2_trainer.train()
p2_trainer.save_model(PASS2_SAVE)
p2_tokenizer.save_pretrained(PASS2_SAVE)
print(f"Saved Pass 2 model to {PASS2_SAVE}")

del p2_model, p2_trainer
torch.cuda.empty_cache()

# ── Step 7: Evaluate Full Pipeline ────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING FULL PIPELINE WITH PASS 2 REFINEMENT")
print("="*60)

p2_eval_model = MBartForConditionalGeneration.from_pretrained(
    PASS2_SAVE, local_files_only=True
).to(device)
p2_eval_model.eval()

def translate_full_pipeline(
    model1, model2, p2_model, p2_tok,
    text, threshold, alpha_p2=0.25
):
    # Pass 1 — ensemble
    inputs = eval_tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)

    with autocast('cuda'):
        enc1 = model1.model.encoder(**inputs)
        enc2 = model2.model.encoder(**inputs)

    processor = EnsembleLogitsProcessor(
        model2          = model2,
        encoder_hidden2 = enc2.last_hidden_state.clone(),
        attention_mask  = inputs['attention_mask'].clone(),
        alpha           = 0.5,
        num_beams       = 4,
    )

    with torch.no_grad():
        pass1_out = model1.generate(
            **inputs,
            encoder_outputs         = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id     = forced_bos,
            max_new_tokens          = 128,
            num_beams               = 4,
            logits_processor        = LogitsProcessorList([processor]),
            early_stopping          = True,
            output_scores           = True,
            return_dict_in_generate = True,
        )

    pass1_text  = eval_tokenizer.decode(
        pass1_out.sequences[0], skip_special_tokens=True
    )
    pass1_score = pass1_out.sequences_scores[0].item()

    # Confidence gate — only refine if above threshold
    if pass1_score < threshold:
        return pass1_text

    # Pass 2 — refinement with logit blend
    combined = f"{text} | {pass1_text}"
    p2_inputs = p2_tok(
        combined, return_tensors='pt',
        truncation=True, max_length=256
    ).to(device)

    with torch.no_grad():
        p2_enc = p2_model.model.encoder(**p2_inputs)

    # Blend Pass 1 encoder + Pass 2 encoder via logit averaging
    class Pass2LogitsProcessor(LogitsProcessor):
        def __init__(self, p2_model, p2_enc_hidden, p2_mask, alpha=0.25, num_beams=4):
            self.p2_model    = p2_model
            self.p2_enc      = p2_enc_hidden
            self.p2_mask     = p2_mask
            self.alpha       = alpha
            self.num_beams   = num_beams
            self._expanded   = False

        def __call__(self, input_ids, scores):
            if not self._expanded:
                self.p2_enc  = self.p2_enc.repeat_interleave(self.num_beams, dim=0)
                self.p2_mask = self.p2_mask.repeat_interleave(self.num_beams, dim=0)
                self._expanded = True

            with torch.inference_mode(), autocast('cuda'):
                out = self.p2_model(
                    attention_mask    = self.p2_mask,
                    decoder_input_ids = input_ids,
                    encoder_outputs   = BaseModelOutput(
                        last_hidden_state=self.p2_enc
                    ),
                    use_cache = False,
                )

            lp1 = F.log_softmax(scores,                       dim=-1)
            lp2 = F.log_softmax(out.logits[:, -1, :].float(), dim=-1)
            return (1 - self.alpha) * lp1 + self.alpha * lp2

    p2_processor = Pass2LogitsProcessor(
        p2_model      = p2_eval_model,
        p2_enc_hidden = p2_enc.last_hidden_state.clone(),
        p2_mask       = p2_inputs['attention_mask'].clone(),
        alpha         = alpha_p2,
        num_beams     = 4,
    )

    with torch.no_grad():
        final_out = model1.generate(
            **inputs,
            encoder_outputs     = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id = forced_bos,
            max_new_tokens      = 128,
            num_beams           = 4,
            logits_processor    = LogitsProcessorList([
                processor, p2_processor
            ]),
            early_stopping      = True,
        )

    return eval_tokenizer.decode(final_out[0], skip_special_tokens=True)


# Run evaluation
hyp_pipeline = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_pipeline.append(
        translate_full_pipeline(
            curr_eval_model, mrasp2_model,
            p2_eval_model, p2_tokenizer,
            row['rural'], threshold, alpha_p2=0.25
        )
    )
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}")

b = bleu.corpus_score(hyp_pipeline, [references])
c = chrf.corpus_score(hyp_pipeline, [references])
print(f"\nFull Pipeline (Curriculum+mRASP2 ensemble + Pass2 refinement):")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")


Building Pass 2 training data...
Pass 2 training samples: 2885
Sample input:  আমনের কি বই পড়ার অভ্যাস আছে? | আপনার কি বই পড়ার অভ্যাস আছে ?
Sample target: আপনার কি বই পড়ার অভ্যাস আছে ?
  0/1250


Map:   0%|          | 0/2885 [00:00<?, ? examples/s]

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]


Fine-tuning Pass 2 refinement model...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,10.042251,3.384337


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from transformers import MBartForConditionalGeneration
from sacrebleu.metrics import BLEU, CHRF

# Load saved Pass 2 model
PASS2_SAVE = f'{DRIVE_PATH}/mbart_pass2_refinement'

p2_tokenizer = MBart50TokenizerFast.from_pretrained(PASS2_SAVE)
p2_eval_model = MBartForConditionalGeneration.from_pretrained(
    PASS2_SAVE, local_files_only=True
).to(device)
p2_eval_model.eval()

# Make sure curr_eval_model and mrasp2_model are loaded
if 'curr_eval_model' not in dir():
    curr_eval_model = MBartForConditionalGeneration.from_pretrained(
        CURRICULUM_SAVE, local_files_only=True
    ).to(device)
    curr_eval_model.eval()

if 'mrasp2_model' not in dir():
    mrasp2_model = MBartForConditionalGeneration.from_pretrained(
        R2S_SAVE, local_files_only=True
    ).to(device)
    mrasp2_model.eval()

# Hardcode already-computed threshold
threshold = -0.2147

# Run evaluation
bleu = BLEU()
chrf = CHRF()
references = test_df['standard'].tolist()

hyp_pipeline = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_pipeline.append(
        translate_full_pipeline(
            curr_eval_model, mrasp2_model,
            p2_eval_model, p2_tokenizer,
            row['rural'], threshold, alpha_p2=0.25
        )
    )
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}")

b = bleu.corpus_score(hyp_pipeline, [references])
c = chrf.corpus_score(hyp_pipeline, [references])
print(f"\nFull Pipeline (Curriculum+mRASP2 ensemble + Pass2 refinement):")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871

Full Pipeline (Curriculum+mRASP2 ensemble + Pass2 refinement):
  BLEU: 35.86 | chrF: 66.17


## Section 6 — Rural Bengali → Standard Bengali: Dialect-Aware Contrastive Fine-Tuning (DACF)
Novel contribution. Explicitly separates the five Bengali dialect varieties in encoder space using triplet + InfoNCE loss. Best single model at 36.09 BLEU. DACF + Curriculum logit ensemble achieves 36.74 BLEU — best r2s result.

In [ ]:
# ── Dialect-Aware Contrastive Fine-tuning (DACF) ──────────────────────
print("\n" + "="*60)
print("DIALECT-AWARE CONTRASTIVE FINE-TUNING (DACF)")
print("="*60)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    get_cosine_schedule_with_warmup
)
from torch.amp import autocast
import random
import numpy as np

DACF_SAVE = f'{DRIVE_PATH}/mbart_dacf'

# ── DACF Model ─────────────────────────────────────────────────────────
class DACFModel(nn.Module):
    def __init__(self, model_name_or_path, temperature=0.1):
        super().__init__()
        self.mbart       = MBartForConditionalGeneration.from_pretrained(
            model_name_or_path, local_files_only=True
        )
        self.temperature = temperature
        hidden_size      = self.mbart.config.d_model
        self.projector   = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 256)
        )

    def encode(self, input_ids, attention_mask):
        enc = self.mbart.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden = enc.last_hidden_state
        mask   = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1)
        return F.normalize(self.projector(pooled), dim=-1)

    def forward(
        self,
        input_ids, attention_mask, labels,
        pos_ids, pos_mask,
        neg_ids, neg_mask,
        lm_weight=1.0, cl_weight=0.3
    ):
        # LM loss
        lm_out  = self.mbart(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        lm_loss = lm_out.loss

        # Dialect contrastive loss
        # anchor = rural sentence from dialect A
        # positive = different rural sentence from SAME dialect A
        # negative = rural sentence from DIFFERENT dialect B
        anchor_emb = self.encode(input_ids, attention_mask)
        pos_emb    = self.encode(pos_ids,   pos_mask)
        neg_emb    = self.encode(neg_ids,   neg_mask)

        # Triplet loss — push same-dialect together, different-dialect apart
        pos_sim = (anchor_emb * pos_emb).sum(-1)
        neg_sim = (anchor_emb * neg_emb).sum(-1)
        triplet_loss = F.relu(
            neg_sim - pos_sim + 0.3  # margin
        ).mean()

        # Also add InfoNCE across the batch for tighter clustering
        sim_matrix = torch.matmul(anchor_emb, pos_emb.T) / self.temperature
        batch_size = anchor_emb.size(0)
        cl_labels  = torch.arange(batch_size, device=anchor_emb.device)
        infonce_loss = (
            F.cross_entropy(sim_matrix,   cl_labels) +
            F.cross_entropy(sim_matrix.T, cl_labels)
        ) / 2

        cl_loss    = triplet_loss + 0.5 * infonce_loss
        total_loss = lm_weight * lm_loss + cl_weight * cl_loss

        return total_loss, lm_loss.item(), cl_loss.item()


# ── Dataset ────────────────────────────────────────────────────────────
class DACFDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

        # Group by dialect for sampling
        self.dialect_groups = df.groupby('dialect').indices

    def tokenize(self, text):
        return self.tokenizer(
            text, max_length=self.max_len,
            truncation=True, padding='max_length',
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        dialect = row['dialect']

        # Tokenize anchor (rural input)
        src    = self.tokenize(row['rural'])
        labels = self.tokenizer(
            text_target=row['standard'],
            max_length=self.max_len,
            truncation=True, padding='max_length',
            return_tensors='pt'
        )['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        # Positive — same dialect, different sentence
        same_dialect = [
            i for i in self.dialect_groups[dialect] if i != idx
        ]
        pos_idx  = random.choice(same_dialect)
        pos      = self.tokenize(self.df.iloc[pos_idx]['rural'])

        # Negative — different dialect
        other_dialects = [
            d for d in self.dialect_groups if d != dialect
        ]
        neg_dialect = random.choice(other_dialects)
        neg_idx     = random.choice(list(self.dialect_groups[neg_dialect]))
        neg         = self.tokenize(self.df.iloc[neg_idx]['rural'])

        return {
            'input_ids':      src['input_ids'].squeeze(),
            'attention_mask': src['attention_mask'].squeeze(),
            'labels':         labels,
            'pos_ids':        pos['input_ids'].squeeze(),
            'pos_mask':       pos['attention_mask'].squeeze(),
            'neg_ids':        neg['input_ids'].squeeze(),
            'neg_mask':       neg['attention_mask'].squeeze(),
        }


def dacf_collate(batch):
    keys = [
        'input_ids', 'attention_mask', 'labels',
        'pos_ids', 'pos_mask', 'neg_ids', 'neg_mask'
    ]
    return {k: torch.stack([b[k] for b in batch]) for k in keys}


# ── Training ───────────────────────────────────────────────────────────
dacf_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
dacf_tokenizer.src_lang = SRC_LANG

# Start from curriculum checkpoint
dacf_model = DACFModel(CURRICULUM_SAVE, temperature=0.1).to(device)

train_dataset = DACFDataset(train_df, dacf_tokenizer)
val_dataset   = DACFDataset(val_df,   dacf_tokenizer)

train_loader = DataLoader(
    train_dataset, batch_size=8, shuffle=True,
    collate_fn=dacf_collate, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=8, shuffle=False,
    collate_fn=dacf_collate, num_workers=2
)

optimizer = AdamW([
    {'params': dacf_model.mbart.parameters(),     'lr': 5e-6},
    {'params': dacf_model.projector.parameters(), 'lr': 1e-4},
], weight_decay=0.01)

DACF_EPOCHS  = 3
GRAD_ACCUM   = 4
total_steps  = (len(train_loader) // GRAD_ACCUM) * DACF_EPOCHS
warmup_steps = total_steps // 10

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler        = torch.cuda.amp.GradScaler()
best_val_loss = float('inf')

print(f"Steps: {total_steps} | Warmup: {warmup_steps}")

for epoch in range(DACF_EPOCHS):
    dacf_model.train()
    total_loss, total_lm, total_cl = 0, 0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        pos_ids        = batch['pos_ids'].to(device)
        pos_mask       = batch['pos_mask'].to(device)
        neg_ids        = batch['neg_ids'].to(device)
        neg_mask       = batch['neg_mask'].to(device)

        with autocast('cuda'):
            loss, lm_l, cl_l = dacf_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                pos_ids=pos_ids, pos_mask=pos_mask,
                neg_ids=neg_ids, neg_mask=neg_mask,
                lm_weight=1.0, cl_weight=0.3
            )
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(dacf_model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM
        total_lm   += lm_l
        total_cl   += cl_l

        if step % 100 == 0:
            print(
                f"  Epoch {epoch+1} step {step}/{len(train_loader)} "
                f"| loss {total_loss/(step+1):.4f} "
                f"| lm {total_lm/(step+1):.4f} "
                f"| cl {total_cl/(step+1):.4f}",
                flush=True
            )

    # Validation
    dacf_model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            with autocast('cuda'):
                loss, _, _ = dacf_model(
                    input_ids      = batch['input_ids'].to(device),
                    attention_mask = batch['attention_mask'].to(device),
                    labels         = batch['labels'].to(device),
                    pos_ids        = batch['pos_ids'].to(device),
                    pos_mask       = batch['pos_mask'].to(device),
                    neg_ids        = batch['neg_ids'].to(device),
                    neg_mask       = batch['neg_mask'].to(device),
                    lm_weight=1.0, cl_weight=0.3
                )
            val_loss += loss.item()

    avg_val = val_loss / len(val_loader)
    print(f"Epoch {epoch+1} | val_loss: {avg_val:.4f}", flush=True)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        os.makedirs(DACF_SAVE, exist_ok=True)
        dacf_model.mbart.save_pretrained(DACF_SAVE)
        dacf_tokenizer.save_pretrained(DACF_SAVE)
        print(f"  Saved to {DACF_SAVE}", flush=True)

del dacf_model
torch.cuda.empty_cache()

# ── Evaluate ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING DACF MODEL")
print("="*60)

dacf_eval_model = MBartForConditionalGeneration.from_pretrained(
    DACF_SAVE, local_files_only=True
).to(device)
dacf_eval_model.eval()

hyp_dacf = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_dacf.append(translate(dacf_eval_model, row['rural']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_dacf, [references])
c = chrf.corpus_score(hyp_dacf, [references])
print(f"\nDAC Model:")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# ── Also ensemble DACF with curriculum+mRASP2 ─────────────────────────
print("\nEvaluating DACF + mRASP2 logit ensemble...")
hyp_dacf_mrasp2 = translate_ensemble_generate(
    dacf_eval_model, mrasp2_model,
    texts, alpha=0.5, batch_size=64, num_beams=4
)
b = bleu.corpus_score(hyp_dacf_mrasp2, [references])
c = chrf.corpus_score(hyp_dacf_mrasp2, [references])
print(f"DACF + mRASP2: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")


DIALECT-AWARE CONTRASTIVE FINE-TUNING (DACF)


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Steps: 879 | Warmup: 87


/tmp/ipykernel_4934/1256556680.py:194: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler        = torch.cuda.amp.GradScaler()


  Epoch 1 step 0/1172 | loss 0.7842 | lm 0.4137 | cl 1.2348
  Epoch 1 step 100/1172 | loss 0.9302 | lm 0.5095 | cl 1.4025
  Epoch 1 step 200/1172 | loss 0.8847 | lm 0.4721 | cl 1.3755
  Epoch 1 step 300/1172 | loss 0.8620 | lm 0.4533 | cl 1.3623
  Epoch 1 step 400/1172 | loss 0.8447 | lm 0.4407 | cl 1.3465
  Epoch 1 step 500/1172 | loss 0.8300 | lm 0.4302 | cl 1.3327
  Epoch 1 step 600/1172 | loss 0.8159 | lm 0.4203 | cl 1.3188
  Epoch 1 step 700/1172 | loss 0.8027 | lm 0.4107 | cl 1.3066
  Epoch 1 step 800/1172 | loss 0.7884 | lm 0.4007 | cl 1.2926
  Epoch 1 step 900/1172 | loss 0.7793 | lm 0.3948 | cl 1.2815
  Epoch 1 step 1000/1172 | loss 0.7687 | lm 0.3886 | cl 1.2669
  Epoch 1 step 1100/1172 | loss 0.7614 | lm 0.3857 | cl 1.2522
Epoch 1 | val_loss: 1.5713


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved to /content/drive/MyDrive/bengali_translation/mbart_dacf
  Epoch 2 step 0/1172 | loss 0.4701 | lm 0.1575 | cl 1.0421
  Epoch 2 step 100/1172 | loss 0.6089 | lm 0.2803 | cl 1.0952
  Epoch 2 step 200/1172 | loss 0.6068 | lm 0.2799 | cl 1.0897
  Epoch 2 step 300/1172 | loss 0.6124 | lm 0.2874 | cl 1.0836
  Epoch 2 step 400/1172 | loss 0.6128 | lm 0.2899 | cl 1.0764
  Epoch 2 step 500/1172 | loss 0.6126 | lm 0.2928 | cl 1.0662
  Epoch 2 step 600/1172 | loss 0.6115 | lm 0.2949 | cl 1.0552
  Epoch 2 step 700/1172 | loss 0.6077 | lm 0.2932 | cl 1.0485
  Epoch 2 step 800/1172 | loss 0.6049 | lm 0.2927 | cl 1.0409
  Epoch 2 step 900/1172 | loss 0.6040 | lm 0.2936 | cl 1.0346
  Epoch 2 step 1000/1172 | loss 0.5992 | lm 0.2907 | cl 1.0283
  Epoch 2 step 1100/1172 | loss 0.5983 | lm 0.2913 | cl 1.0231
Epoch 2 | val_loss: 1.5972
  Epoch 3 step 0/1172 | loss 0.5343 | lm 0.2408 | cl 0.9782
  Epoch 3 step 100/1172 | loss 0.5367 | lm 0.2566 | cl 0.9339
  Epoch 3 step 200/1172 | loss 0.5378 | lm

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

NameError: name 'translate' is not defined

In [ ]:
dacf_eval_model = MBartForConditionalGeneration.from_pretrained(
    DACF_SAVE, local_files_only=True
).to(device)
dacf_eval_model.eval()

def translate(model, text):
    inputs = eval_tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            max_new_tokens=128,
            num_beams=4
        )
    return eval_tokenizer.decode(outputs[0], skip_special_tokens=True)

hyp_dacf = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_dacf.append(translate(dacf_eval_model, row['rural']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_dacf, [references])
c = chrf.corpus_score(hyp_dacf, [references])
print(f"\nDAC Model:")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871

DAC Model:
  BLEU: 36.09 | chrF: 66.26


In [ ]:
from transformers import LogitsProcessor, LogitsProcessorList
from transformers.modeling_outputs import BaseModelOutput
from torch.amp import autocast
import torch.nn.functional as F

class EnsembleLogitsProcessor(LogitsProcessor):
    def __init__(self, model2, encoder_hidden2, attention_mask, alpha=0.5, num_beams=4):
        self.model2         = model2
        self.enc_hidden2    = encoder_hidden2
        self.attention_mask = attention_mask
        self.alpha          = alpha
        self.num_beams      = num_beams
        self._expanded      = False

    def __call__(self, input_ids, scores):
        if not self._expanded:
            self.enc_hidden2    = self.enc_hidden2.repeat_interleave(self.num_beams, dim=0)
            self.attention_mask = self.attention_mask.repeat_interleave(self.num_beams, dim=0)
            self._expanded      = True

        with torch.inference_mode(), autocast('cuda'):
            out2 = self.model2(
                attention_mask    = self.attention_mask,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(
                    last_hidden_state=self.enc_hidden2
                ),
                use_cache = False,
            )

        lp1 = F.log_softmax(scores,                         dim=-1)
        lp2 = F.log_softmax(out2.logits[:, -1, :].float(),  dim=-1)
        return self.alpha * lp1 + (1 - self.alpha) * lp2


@torch.inference_mode()
def translate_ensemble_generate(model1, model2, texts, alpha=0.5, batch_size=64, num_beams=4):
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = eval_tokenizer(
            batch_texts, return_tensors='pt',
            truncation=True, max_length=128,
            padding=True
        ).to(device)

        with autocast('cuda'):
            enc1 = model1.model.encoder(**inputs)
            enc2 = model2.model.encoder(**inputs)

        processor = EnsembleLogitsProcessor(
            model2          = model2,
            encoder_hidden2 = enc2.last_hidden_state.clone(),
            attention_mask  = inputs['attention_mask'].clone(),
            alpha           = alpha,
            num_beams       = num_beams,
        )

        outputs = model1.generate(
            **inputs,
            encoder_outputs     = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id = forced_bos,
            max_new_tokens      = 128,
            num_beams           = num_beams,
            logits_processor    = LogitsProcessorList([processor]),
            early_stopping      = True,
        )

        for seq in outputs:
            all_outputs.append(
                eval_tokenizer.decode(seq, skip_special_tokens=True)
            )

        if i % 200 == 0:
            print(f"  {i}/{len(texts)}", flush=True)

        torch.cuda.empty_cache()

    return all_outputs

In [ ]:
texts = test_df['rural'].tolist()

In [ ]:
hyp_dacf_mrasp2 = translate_ensemble_generate(
    dacf_eval_model, mrasp2_model,
    texts, alpha=0.5, batch_size=64, num_beams=4
)
b = bleu.corpus_score(hyp_dacf_mrasp2, [references])
c = chrf.corpus_score(hyp_dacf_mrasp2, [references])
print(f"DACF + mRASP2: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

  0/1871
  1600/1871
DACF + mRASP2: BLEU: 36.53 | chrF: 66.67


In [ ]:
hyp_dacf_curr = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    texts, alpha=0.5, batch_size=64, num_beams=4
)
b = bleu.corpus_score(hyp_dacf_curr, [references])
c = chrf.corpus_score(hyp_dacf_curr, [references])
print(f"DACF + Curriculum: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

  0/1871
  1600/1871
DACF + Curriculum: BLEU: 36.74 | chrF: 66.68


In [ ]:
# Three-way: use DACF as base, blend curriculum and mRASP2 equally
class ThreeWayLogitsProcessor(LogitsProcessor):
    def __init__(self, model2, model3, enc2, enc3, mask2, mask3, alpha=0.33, num_beams=4):
        self.model2    = model2
        self.model3    = model3
        self.enc2      = enc2
        self.enc3      = enc3
        self.mask2     = mask2
        self.mask3     = mask3
        self.alpha     = alpha
        self.num_beams = num_beams
        self._expanded = False

    def __call__(self, input_ids, scores):
        if not self._expanded:
            self.enc2  = self.enc2.repeat_interleave(self.num_beams, dim=0)
            self.enc3  = self.enc3.repeat_interleave(self.num_beams, dim=0)
            self.mask2 = self.mask2.repeat_interleave(self.num_beams, dim=0)
            self.mask3 = self.mask3.repeat_interleave(self.num_beams, dim=0)
            self._expanded = True

        with torch.inference_mode(), autocast('cuda'):
            out2 = self.model2(
                attention_mask    = self.mask2,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(last_hidden_state=self.enc2),
                use_cache         = False,
            )
            out3 = self.model3(
                attention_mask    = self.mask3,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(last_hidden_state=self.enc3),
                use_cache         = False,
            )

        lp1 = F.log_softmax(scores,                         dim=-1)
        lp2 = F.log_softmax(out2.logits[:, -1, :].float(),  dim=-1)
        lp3 = F.log_softmax(out3.logits[:, -1, :].float(),  dim=-1)
        return self.alpha * lp1 + self.alpha * lp2 + self.alpha * lp3


@torch.inference_mode()
def translate_three_way(model1, model2, model3, texts, batch_size=32, num_beams=4):
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = eval_tokenizer(
            batch_texts, return_tensors='pt',
            truncation=True, max_length=128,
            padding=True
        ).to(device)

        with autocast('cuda'):
            enc1 = model1.model.encoder(**inputs)
            enc2 = model2.model.encoder(**inputs)
            enc3 = model3.model.encoder(**inputs)

        processor = ThreeWayLogitsProcessor(
            model2=model2, model3=model3,
            enc2=enc2.last_hidden_state.clone(),
            enc3=enc3.last_hidden_state.clone(),
            mask2=inputs['attention_mask'].clone(),
            mask3=inputs['attention_mask'].clone(),
            alpha=1/3, num_beams=num_beams,
        )

        outputs = model1.generate(
            **inputs,
            encoder_outputs     = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id = forced_bos,
            max_new_tokens      = 128,
            num_beams           = num_beams,
            logits_processor    = LogitsProcessorList([processor]),
            early_stopping      = True,
        )

        for seq in outputs:
            all_outputs.append(
                eval_tokenizer.decode(seq, skip_special_tokens=True)
            )

        if i % 200 == 0:
            print(f"  {i}/{len(texts)}", flush=True)

        torch.cuda.empty_cache()

    return all_outputs


hyp_three_way = translate_three_way(
    dacf_eval_model, curr_eval_model, mrasp2_model,
    texts, batch_size=64, num_beams=4
)
b = bleu.corpus_score(hyp_three_way, [references])
c = chrf.corpus_score(hyp_three_way, [references])
print(f"Three-way (DACF + Curriculum + mRASP2): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

  0/1871
  1600/1871
Three-way (DACF + Curriculum + mRASP2): BLEU: 36.44 | chrF: 66.65


## Section 7 — Standard Bengali → English: Helsinki MarianMT Fine-Tuning
Helsinki-NLP/opus-mt-bn-en fine-tuned on Vashantor. 49.47 BLEU on clean input, 34.67 BLEU on normalized input — 14.80 BLEU gap motivates pipeline-aware training.

## Standard Bengali To English + Entire Pipeline


In [ ]:
# ── Helsinki Bengali-English Fine-tuning ──────────────────────────────
print("\n" + "="*60)
print("HELSINKI BN-EN FINE-TUNING")
print("="*60)

from transformers import (
    MarianMTModel, MarianTokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset

HELSINKI_MODEL = 'Helsinki-NLP/opus-mt-bn-en'
HELSINKI_SAVE  = f'{DRIVE_PATH}/helsinki_s2e'

# ── Load tokenizer and model ───────────────────────────────────────────
print("Loading Helsinki model...")
hel_tokenizer = MarianTokenizer.from_pretrained(HELSINKI_MODEL)
hel_model     = MarianMTModel.from_pretrained(HELSINKI_MODEL)
print("Loaded!")

MAX_LEN = 128

# ── Tokenize ───────────────────────────────────────────────────────────
def tokenize_s2e(batch):
    model_inputs = hel_tokenizer(
        batch['standard'], max_length=MAX_LEN,
        truncation=True, padding='max_length'
    )
    labels = hel_tokenizer(
        text_target=batch['english'],
        max_length=MAX_LEN, truncation=True, padding='max_length'
    )
    model_inputs['labels'] = [
        [-100 if t == hel_tokenizer.pad_token_id else t for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

train_hf = HFDataset.from_dict({
    'standard': train_df['standard'].tolist(),
    'english':  train_df['english'].tolist()
})
val_hf = HFDataset.from_dict({
    'standard': val_df['standard'].tolist(),
    'english':  val_df['english'].tolist()
})

print("Tokenizing...")
train_tok = train_hf.map(
    tokenize_s2e, batched=True, batch_size=64,
    remove_columns=['standard', 'english']
)
val_tok = val_hf.map(
    tokenize_s2e, batched=True, batch_size=64,
    remove_columns=['standard', 'english']
)

# ── Training args ──────────────────────────────────────────────────────
hel_args = Seq2SeqTrainingArguments(
    output_dir                  = HELSINKI_SAVE,
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    gradient_accumulation_steps = 2,
    learning_rate               = 2e-5,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    predict_with_generate       = True,
    generation_max_length       = 128,
    generation_num_beams        = 4,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    fp16                        = True,
    logging_steps               = 20,
    report_to                   = 'none',
    save_total_limit            = 2,
    label_smoothing_factor      = 0.1,
)

trainer = Seq2SeqTrainer(
    model            = hel_model,
    args             = hel_args,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    processing_class = hel_tokenizer,
    data_collator    = DataCollatorForSeq2Seq(
        hel_tokenizer, model=hel_model, padding=True
    ),
)

print("Fine-tuning...")
trainer.train()
trainer.save_model(HELSINKI_SAVE)
hel_tokenizer.save_pretrained(HELSINKI_SAVE)
print(f"Saved to {HELSINKI_SAVE}")

del hel_model, trainer
torch.cuda.empty_cache()

# ── Evaluate ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING HELSINKI S2E")
print("="*60)

hel_eval_model = MarianMTModel.from_pretrained(
    HELSINKI_SAVE, local_files_only=True
).to(device)
hel_eval_model.eval()
hel_eval_tok = MarianTokenizer.from_pretrained(HELSINKI_SAVE)

def translate_helsinki(text):
    inputs = hel_eval_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = hel_eval_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
        )
    return hel_eval_tok.decode(outputs[0], skip_special_tokens=True)

# Evaluate on gold standard Bengali
references_en = test_df['english'].tolist()
hyp_helsinki  = []

for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_helsinki.append(translate_helsinki(row['standard']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_helsinki, [references_en])
c = chrf.corpus_score(hyp_helsinki, [references_en])
print(f"\nHelsinki s2e (gold standard input):")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# ── Also evaluate on DACF+curriculum normalized output ────────────────
print("\nEvaluating on normalized rural Bengali input...")
hyp_helsinki_pipeline = []

for i, (_, row) in enumerate(test_df.iterrows()):
    # Use DACF+curriculum ensemble output as input
    normalized = hyp_dacf_curr[i]
    hyp_helsinki_pipeline.append(translate_helsinki(normalized))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_helsinki_pipeline, [references_en])
c = chrf.corpus_score(hyp_helsinki_pipeline, [references_en])
print(f"\nHelsinki s2e (normalized rural input via DACF+curriculum):")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")


HELSINKI BN-EN FINE-TUNING
Loading Helsinki model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/806k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  0/1871


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871

Helsinki s2e (gold standard input):
  BLEU: 49.47 | chrF: 68.78

Evaluating on normalized rural Bengali input...
  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871

Helsinki s2e (normalized rur

## Section 8 — Standard Bengali → English: Joint Encoder Multi-Decoder (Negative Result)
Shared DACF encoder with two decoders trained simultaneously. 29.31 BLEU — underperforms due to cold-start English decoder lacking Bengali-English pretraining.

In [ ]:
# ── Joint Encoder Multi-Decoder Training ──────────────────────────────
print("\n" + "="*60)
print("JOINT ENCODER MULTI-DECODER TRAINING")
print("="*60)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    get_cosine_schedule_with_warmup
)
from torch.amp import autocast

JOINT_SAVE = f'{DRIVE_PATH}/mbart_joint'

# ── Joint Model ────────────────────────────────────────────────────────
class JointModel(nn.Module):
    def __init__(self, dacf_path, base_model_name):
        super().__init__()

        # Load DACF model as r2s backbone
        self.r2s = MBartForConditionalGeneration.from_pretrained(
            dacf_path, local_files_only=True
        )

        # Untie lm_head from embeddings to avoid parameter group conflict
        self.r2s.lm_head = nn.Linear(
            self.r2s.config.d_model,
            self.r2s.config.vocab_size,
            bias=False
        )
        self.r2s.lm_head.weight = nn.Parameter(
            self.r2s.model.shared.weight.clone().detach()
        )

        # Load fresh mBART for s2e decoder
        s2e_base = MBartForConditionalGeneration.from_pretrained(
            base_model_name
        )

        # Untie s2e lm_head too
        self.s2e_decoder    = s2e_base.model.decoder
        self.s2e_lm_head    = nn.Linear(
            s2e_base.config.d_model,
            s2e_base.config.vocab_size,
            bias=False
        )
        self.s2e_lm_head.weight = nn.Parameter(
            s2e_base.model.shared.weight.clone().detach()
        )
        self.s2e_final_bias = nn.Parameter(
            s2e_base.final_logits_bias.clone().detach()
        )

        del s2e_base
        torch.cuda.empty_cache()

    def forward_r2s(self, input_ids, attention_mask, labels):
        return self.r2s(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        ).loss

    def forward_s2e(self, input_ids, attention_mask, labels, forced_bos_id):
        # Encode with shared DACF encoder
        encoder_outputs = self.r2s.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Prepare decoder input ids (shift right)
        decoder_input_ids = labels.clone()
        decoder_input_ids[decoder_input_ids == -100] = 1
        decoder_input_ids = torch.roll(decoder_input_ids, 1, dims=1)
        decoder_input_ids[:, 0] = forced_bos_id

        # Decode to English
        decoder_outputs = self.s2e_decoder(
            input_ids=decoder_input_ids,
            encoder_hidden_states=encoder_outputs.last_hidden_state,
            encoder_attention_mask=attention_mask,
        )

        # Compute lm loss
        logits   = self.s2e_lm_head(decoder_outputs.last_hidden_state) + self.s2e_final_bias
        loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
        s2e_loss = loss_fct(
            logits.view(-1, logits.size(-1)),
            labels.view(-1)
        )
        return s2e_loss

    def forward(
        self,
        r2s_input_ids, r2s_attention_mask, r2s_labels,
        s2e_input_ids, s2e_attention_mask, s2e_labels,
        forced_bos_id,
        r2s_weight=0.5, s2e_weight=0.5
    ):
        r2s_loss = self.forward_r2s(
            r2s_input_ids, r2s_attention_mask, r2s_labels
        )
        s2e_loss = self.forward_s2e(
            s2e_input_ids, s2e_attention_mask, s2e_labels, forced_bos_id
        )
        total_loss = r2s_weight * r2s_loss + s2e_weight * s2e_loss
        return total_loss, r2s_loss.item(), s2e_loss.item()


# ── Dataset ────────────────────────────────────────────────────────────
class JointDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def tokenize(self, text, is_target=False):
        if is_target:
            return self.tokenizer(
                text_target=text,
                max_length=self.max_len,
                truncation=True, padding='max_length',
                return_tensors='pt'
            )
        return self.tokenizer(
            text, max_length=self.max_len,
            truncation=True, padding='max_length',
            return_tensors='pt'
        )

    def make_labels(self, input_ids):
        labels = input_ids.squeeze().clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # r2s: rural → standard
        r2s_src    = self.tokenize(row['rural'])
        r2s_tgt    = self.tokenize(row['standard'], is_target=True)
        r2s_labels = self.make_labels(r2s_tgt['input_ids'])

        # s2e: standard → english
        s2e_src    = self.tokenize(row['standard'])
        s2e_tgt    = self.tokenize(row['english'],  is_target=True)
        s2e_labels = self.make_labels(s2e_tgt['input_ids'])

        return {
            'r2s_input_ids':      r2s_src['input_ids'].squeeze(),
            'r2s_attention_mask': r2s_src['attention_mask'].squeeze(),
            'r2s_labels':         r2s_labels,
            's2e_input_ids':      s2e_src['input_ids'].squeeze(),
            's2e_attention_mask': s2e_src['attention_mask'].squeeze(),
            's2e_labels':         s2e_labels,
        }


def joint_collate(batch):
    keys = [
        'r2s_input_ids', 'r2s_attention_mask', 'r2s_labels',
        's2e_input_ids', 's2e_attention_mask', 's2e_labels'
    ]
    return {k: torch.stack([b[k] for b in batch]) for k in keys}


# ── Training Setup ─────────────────────────────────────────────────────
joint_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
joint_tokenizer.src_lang = SRC_LANG
forced_bos_bn = joint_tokenizer.lang_code_to_id[SRC_LANG]
forced_bos_en = joint_tokenizer.lang_code_to_id['en_XX']

joint_model = JointModel(
    dacf_path       = DACF_SAVE,
    base_model_name = MODEL_NAME
).to(device)

train_dataset = JointDataset(train_df, joint_tokenizer)
val_dataset   = JointDataset(val_df,   joint_tokenizer)

train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True,
    collate_fn=joint_collate, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False,
    collate_fn=joint_collate, num_workers=2
)

# Collect all named parameters and assign learning rates by component
def get_optimizer(model):
    encoder_params  = []
    r2s_dec_params  = []
    r2s_lm_params   = []
    s2e_params      = []

    seen = set()

    for name, param in model.named_parameters():
        if id(param) in seen:
            continue
        seen.add(id(param))

        if 'r2s.model.encoder' in name:
            encoder_params.append(param)
        elif 'r2s.model.decoder' in name or 'r2s.lm_head' in name:
            r2s_dec_params.append(param)
        elif 's2e' in name:
            s2e_params.append(param)
        else:
            # r2s.model.shared and anything else — treat as encoder
            encoder_params.append(param)

    return AdamW([
        {'params': encoder_params,  'lr': 1e-6},
        {'params': r2s_dec_params,  'lr': 3e-6},
        {'params': s2e_params,      'lr': 5e-6},
    ], weight_decay=0.01)

optimizer = get_optimizer(joint_model)

JOINT_EPOCHS  = 3
GRAD_ACCUM    = 8
total_steps   = (len(train_loader) // GRAD_ACCUM) * JOINT_EPOCHS
warmup_steps  = total_steps // 10

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler        = torch.cuda.amp.GradScaler()
best_val_loss = float('inf')

print(f"Steps: {total_steps} | Warmup: {warmup_steps}")

# ── Training Loop ──────────────────────────────────────────────────────
for epoch in range(JOINT_EPOCHS):
    joint_model.train()
    total_loss, total_r2s, total_s2e = 0, 0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        r2s_input_ids      = batch['r2s_input_ids'].to(device)
        r2s_attention_mask = batch['r2s_attention_mask'].to(device)
        r2s_labels         = batch['r2s_labels'].to(device)
        s2e_input_ids      = batch['s2e_input_ids'].to(device)
        s2e_attention_mask = batch['s2e_attention_mask'].to(device)
        s2e_labels         = batch['s2e_labels'].to(device)

        with autocast('cuda'):
            loss, r2s_l, s2e_l = joint_model(
                r2s_input_ids      = r2s_input_ids,
                r2s_attention_mask = r2s_attention_mask,
                r2s_labels         = r2s_labels,
                s2e_input_ids      = s2e_input_ids,
                s2e_attention_mask = s2e_attention_mask,
                s2e_labels         = s2e_labels,
                forced_bos_id      = forced_bos_en,
                r2s_weight         = 0.5,
                s2e_weight         = 0.5,
            )
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(joint_model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM
        total_r2s  += r2s_l
        total_s2e  += s2e_l

        if step % 100 == 0:
            print(
                f"  Epoch {epoch+1} step {step}/{len(train_loader)} "
                f"| loss {total_loss/(step+1):.4f} "
                f"| r2s {total_r2s/(step+1):.4f} "
                f"| s2e {total_s2e/(step+1):.4f}",
                flush=True
            )

    # Validation
    joint_model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            with autocast('cuda'):
                loss, _, _ = joint_model(
                    r2s_input_ids      = batch['r2s_input_ids'].to(device),
                    r2s_attention_mask = batch['r2s_attention_mask'].to(device),
                    r2s_labels         = batch['r2s_labels'].to(device),
                    s2e_input_ids      = batch['s2e_input_ids'].to(device),
                    s2e_attention_mask = batch['s2e_attention_mask'].to(device),
                    s2e_labels         = batch['s2e_labels'].to(device),
                    forced_bos_id      = forced_bos_en,
                )
            val_loss += loss.item()

    avg_val = val_loss / len(val_loader)
    print(f"Epoch {epoch+1} | val_loss: {avg_val:.4f}", flush=True)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        os.makedirs(f'{JOINT_SAVE}/r2s', exist_ok=True)
        joint_model.r2s.save_pretrained(f'{JOINT_SAVE}/r2s')
        torch.save(
            joint_model.s2e_decoder.state_dict(),
            f'{JOINT_SAVE}/s2e_decoder.pt'
        )
        torch.save(
            joint_model.s2e_lm_head.state_dict(),
            f'{JOINT_SAVE}/s2e_lm_head.pt'
        )
        torch.save(
            joint_model.s2e_final_bias,
            f'{JOINT_SAVE}/s2e_bias.pt'
        )
        joint_tokenizer.save_pretrained(JOINT_SAVE)
        print(f"  Saved to {JOINT_SAVE}", flush=True)

del joint_model
torch.cuda.empty_cache()

# ── Evaluation ─────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING JOINT MODEL")
print("="*60)

joint_r2s = MBartForConditionalGeneration.from_pretrained(
    f'{JOINT_SAVE}/r2s', local_files_only=True
).to(device)
joint_r2s.eval()

s2e_base = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
joint_s2e_decoder = s2e_base.model.decoder
joint_s2e_lm_head = nn.Linear(
    s2e_base.config.d_model,
    s2e_base.config.vocab_size,
    bias=False
)
joint_s2e_decoder.load_state_dict(
    torch.load(f'{JOINT_SAVE}/s2e_decoder.pt')
)
joint_s2e_lm_head.load_state_dict(
    torch.load(f'{JOINT_SAVE}/s2e_lm_head.pt')
)
joint_s2e_bias = torch.load(f'{JOINT_SAVE}/s2e_bias.pt')
joint_s2e_decoder = joint_s2e_decoder.to(device)
joint_s2e_lm_head = joint_s2e_lm_head.to(device)
joint_s2e_bias    = joint_s2e_bias.to(device)
del s2e_base
torch.cuda.empty_cache()

joint_eval_tok = MBart50TokenizerFast.from_pretrained(JOINT_SAVE)
joint_eval_tok.src_lang = SRC_LANG


def translate_joint_pipeline(rural_text):
    # Step 1: r2s
    inputs = joint_eval_tok(
        rural_text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)

    with torch.no_grad():
        r2s_out  = joint_r2s.generate(
            **inputs,
            forced_bos_token_id=forced_bos_bn,
            max_new_tokens=128,
            num_beams=4,
        )
    standard = joint_eval_tok.decode(r2s_out[0], skip_special_tokens=True)

    # Step 2: s2e using shared encoder
    s2e_inputs = joint_eval_tok(
        standard, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)

    with torch.no_grad():
        enc_out       = joint_r2s.model.encoder(**s2e_inputs)
        decoder_input = torch.tensor([[forced_bos_en]], device=device)
        generated     = [forced_bos_en]

        for _ in range(128):
            dec_out  = joint_s2e_decoder(
                input_ids=decoder_input,
                encoder_hidden_states=enc_out.last_hidden_state,
                encoder_attention_mask=s2e_inputs['attention_mask'],
            )
            logits   = joint_s2e_lm_head(
                dec_out.last_hidden_state[:, -1, :]
            ) + joint_s2e_bias
            next_tok = logits.argmax(-1).item()
            if next_tok == joint_eval_tok.eos_token_id:
                break
            generated.append(next_tok)
            decoder_input = torch.tensor([generated], device=device)

    return joint_eval_tok.decode(generated[1:], skip_special_tokens=True)


references_en = test_df['english'].tolist()
hyp_joint     = []

for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_joint.append(translate_joint_pipeline(row['rural']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_joint, [references_en])
c = chrf.corpus_score(hyp_joint, [references_en])
print(f"\nJoint model (rural → English):")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")


JOINT ENCODER MULTI-DECODER TRAINING


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Steps: 879 | Warmup: 87


/tmp/ipykernel_4934/3454075481.py:240: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler        = torch.cuda.amp.GradScaler()


...

  Saved to /content/drive/MyDrive/bengali_translation/mbart_joint
  Epoch 3 step 0/2344 | loss 0.2272 | r2s 0.2523 | s2e 0.2021
  Epoch 3 step 100/2344 | loss 0.3665 | r2s 0.2402 | s2e 0.4928
  Epoch 3 step 200/2344 | loss 0.3888 | r2s 0.2512 | s2e 0.5265
  Epoch 3 step 300/2344 | loss 0.3873 | r2s 0.2513 | s2e 0.5232
  Epoch 3 step 400/2344 | loss 0.3774 | r2s 0.2495 | s2e 0.5054
  Epoch 3 step 500/2344 | loss 0.3779 | r2s 0.2432 | s2e 0.5126
  Epoch 3 step 600/2344 | loss 0.3784 | r2s 0.2426 | s2e 0.5143
  Epoch 3 step 700/2344 | loss 0.3783 | r2s 0.2452 | s2e 0.5114
  Epoch 3 step 800/2344 | loss 0.3788 | r2s 0.2444 | s2e 0.5132
  Epoch 3 step 900/2344 | loss 0.3783 | r2s 0.2460 | s2e 0.5106
  Epoch 3 step 1000/2344 | loss 0.3757 | r2s 0.2431 | s2e 0.5084
  Epoch 3 step 1100/2344 | loss 0.3755 | r2s 0.2434 | s2e 0.5076
  Epoch 3 step 1200/2344 | loss 0.3745 | r2s 0.2428 | s2e 0.5063
  Epoch 3 step 1300/2344 | loss 0.3752 | r2s 0.2431 | s2e 0.5072
  Epoch 3 step 1400/2344 | loss 0.37

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871

Joint model (rural → English):
  BLEU: 29.31 | chrF: 50.41


## Section 9 — Standard Bengali → English: Pipeline-Aware mBART
Trained on normalized r2s outputs + gold standard + raw rural Bengali simultaneously. 36.92 BLEU on normalized input, narrowing clean-to-noisy gap from 14.80 to 11.22 BLEU. Use checkpoint-879.

In [ ]:
# ── Pipeline-Aware mBART S2E Fine-tuning ──────────────────────────────
print("\n" + "="*60)
print("PIPELINE-AWARE MBART S2E FINE-TUNING")
print("="*60)

PIPELINE_S2E_SAVE = f'{DRIVE_PATH}/mbart_pipeline_s2e'

# ── Step 1: Generate normalized training inputs ────────────────────────
print("Generating normalized training set using DACF+Curriculum ensemble...")

# Make sure both models are loaded
if 'dacf_eval_model' not in dir():
    dacf_eval_model = MBartForConditionalGeneration.from_pretrained(
        DACF_SAVE, local_files_only=True
    ).to(device)
    dacf_eval_model.eval()

if 'curr_eval_model' not in dir():
    curr_eval_model = MBartForConditionalGeneration.from_pretrained(
        CURRICULUM_SAVE, local_files_only=True
    ).to(device)
    curr_eval_model.eval()

train_rural   = train_df['rural'].tolist()
train_english = train_df['english'].tolist()
val_rural     = val_df['rural'].tolist()
val_english   = val_df['english'].tolist()

# Generate normalized outputs for train and val
normalized_train = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    train_rural, alpha=0.5, batch_size=64, num_beams=4
)
print(f"Generated {len(normalized_train)} normalized training sentences")

normalized_val = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    val_rural, alpha=0.5, batch_size=64, num_beams=4
)
print(f"Generated {len(normalized_val)} normalized val sentences")

# ── Step 2: Build mixed dataset ────────────────────────────────────────
# Mix normalized outputs with gold standard for robustness
# This trains the model to handle both clean and noisy Bengali
print("\nBuilding mixed training dataset...")

from datasets import Dataset as HFDataset
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

s2e_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
s2e_tokenizer.src_lang = SRC_LANG
forced_bos_en = s2e_tokenizer.lang_code_to_id['en_XX']

MAX_LEN = 128

def tokenize_s2e_mixed(batch):
    model_inputs = s2e_tokenizer(
        batch['input'], max_length=MAX_LEN,
        truncation=True, padding='max_length'
    )
    labels = s2e_tokenizer(
        text_target=batch['english'],
        max_length=MAX_LEN, truncation=True, padding='max_length'
    )
    model_inputs['labels'] = [
        [-100 if t == s2e_tokenizer.pad_token_id else t for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

# Three sources of training data:
# 1. Normalized rural → English (pipeline-aware)
# 2. Gold standard → English (clean)
# 3. Raw rural → English (robustness)
train_hf_mixed = HFDataset.from_dict({
    'input':   normalized_train          +  # normalized rural
               train_df['standard'].tolist() +  # gold standard
               train_rural,                      # raw rural
    'english': train_english +
               train_english +
               train_english,
})

# Val uses only normalized outputs for honest pipeline evaluation
val_hf_mixed = HFDataset.from_dict({
    'input':   normalized_val,
    'english': val_english,
})

print(f"Mixed training samples: {len(train_hf_mixed)}")
print(f"Val samples: {len(val_hf_mixed)}")

train_tok_mixed = train_hf_mixed.map(
    tokenize_s2e_mixed, batched=True, batch_size=64,
    remove_columns=['input', 'english']
)
val_tok_mixed = val_hf_mixed.map(
    tokenize_s2e_mixed, batched=True, batch_size=64,
    remove_columns=['input', 'english']
)

# ── Step 3: Load best mBART r2e as starting point ─────────────────────
print("\nLoading mBART r2e as starting point...")
s2e_model = MBartForConditionalGeneration.from_pretrained(
    f'{DRIVE_PATH}/mbart_r2e', local_files_only=True
)
print("Loaded!")

# ── Step 4: Fine-tune ──────────────────────────────────────────────────
s2e_args = Seq2SeqTrainingArguments(
    output_dir                  = PIPELINE_S2E_SAVE,
    num_train_epochs            = 5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 4,
    learning_rate               = 1e-5,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    predict_with_generate       = True,
    generation_max_length       = 128,
    generation_num_beams        = 4,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    fp16                        = True,
    logging_steps               = 20,
    report_to                   = 'none',
    save_total_limit            = 2,
    label_smoothing_factor      = 0.1,
)

s2e_trainer = Seq2SeqTrainer(
    model            = s2e_model,
    args             = s2e_args,
    train_dataset    = train_tok_mixed,
    eval_dataset     = val_tok_mixed,
    processing_class = s2e_tokenizer,
    data_collator    = DataCollatorForSeq2Seq(
        s2e_tokenizer, model=s2e_model, padding=True
    ),
)

print("Fine-tuning pipeline-aware s2e model...")
s2e_trainer.train()
s2e_trainer.save_model(PIPELINE_S2E_SAVE)
s2e_tokenizer.save_pretrained(PIPELINE_S2E_SAVE)
print(f"Saved to {PIPELINE_S2E_SAVE}")

del s2e_model, s2e_trainer
torch.cuda.empty_cache()

# ── Step 5: Evaluate ───────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING PIPELINE-AWARE S2E")
print("="*60)

pipeline_s2e_model = MBartForConditionalGeneration.from_pretrained(
    PIPELINE_S2E_SAVE, local_files_only=True
).to(device)
pipeline_s2e_model.eval()

pipeline_s2e_tok = MBart50TokenizerFast.from_pretrained(PIPELINE_S2E_SAVE)
pipeline_s2e_tok.src_lang = SRC_LANG

def translate_pipeline_s2e(normalized_text):
    inputs = pipeline_s2e_tok(
        normalized_text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = pipeline_s2e_model.generate(
            **inputs,
            forced_bos_token_id = forced_bos_en,
            max_new_tokens      = 128,
            num_beams           = 4,
        )
    return pipeline_s2e_tok.decode(outputs[0], skip_special_tokens=True)

references_en = test_df['english'].tolist()

# Evaluate on gold standard input
print("Evaluating on gold standard Bengali...")
hyp_pipeline_gold = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_pipeline_gold.append(translate_pipeline_s2e(row['standard']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_pipeline_gold, [references_en])
c = chrf.corpus_score(hyp_pipeline_gold, [references_en])
print(f"\nPipeline-aware s2e (gold standard input):")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Evaluate on DACF+curriculum normalized input
print("\nEvaluating on DACF+Curriculum normalized input...")
hyp_pipeline_normalized = []
for i, (_, row) in enumerate(test_df.iterrows()):
    normalized = hyp_dacf_curr[i]  # reuse already generated outputs
    hyp_pipeline_normalized.append(translate_pipeline_s2e(normalized))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_pipeline_normalized, [references_en])
c = chrf.corpus_score(hyp_pipeline_normalized, [references_en])
print(f"\nPipeline-aware s2e (normalized rural input):")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Evaluate on raw rural input directly
print("\nEvaluating on raw rural Bengali input...")
hyp_pipeline_rural = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_pipeline_rural.append(translate_pipeline_s2e(row['rural']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_pipeline_rural, [references_en])
c = chrf.corpus_score(hyp_pipeline_rural, [references_en])
print(f"\nPipeline-aware s2e (raw rural input):")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")


PIPELINE-AWARE MBART S2E FINE-TUNING
Generating normalized training set using DACF+Curriculum ensemble...
  0/9375
  1600/9375
  3200/9375
  4800/9375
  6400/9375
  8000/9375
Generated 9375 normalized training sentences
  0/1250
Generated 1250 normalized val sentences

Building mixed training dataset...
Mixed training samples: 28125
Val samples: 1250


Map:   0%|          | 0/28125 [00:00<?, ? examples/s]

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]


Loading mBART r2e as starting point...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Loaded!
Fine-tuning pipeline-aware s2e model...


Epoch,Training Loss,Validation Loss
1,7.045004,2.667398
2,6.642926,2.738445


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Kill training, load epoch 1 checkpoint
import os
# Find best checkpoint
checkpoints = [
    f for f in os.listdir(PIPELINE_S2E_SAVE)
    if f.startswith('checkpoint')
]
print(checkpoints)

['checkpoint-879', 'checkpoint-1758']


In [ ]:
import os

# Find the epoch 1 checkpoint path
checkpoint_path = f'{PIPELINE_S2E_SAVE}/checkpoint-879'  # replace with your actual checkpoint name

pipeline_s2e_model = MBartForConditionalGeneration.from_pretrained(
    checkpoint_path
).to(device)
pipeline_s2e_model.eval()

pipeline_s2e_tok = MBart50TokenizerFast.from_pretrained(checkpoint_path)
pipeline_s2e_tok.src_lang = SRC_LANG
forced_bos_en = pipeline_s2e_tok.lang_code_to_id['en_XX']

def translate_pipeline_s2e(text):
    inputs = pipeline_s2e_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = pipeline_s2e_model.generate(
            **inputs,
            forced_bos_token_id = forced_bos_en,
            max_new_tokens      = 128,
            num_beams           = 4,
        )
    return pipeline_s2e_tok.decode(outputs[0], skip_special_tokens=True)

references_en = test_df['english'].tolist()

# Gold standard input
print("Evaluating on gold standard Bengali...")
hyp_pipeline_gold = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_pipeline_gold.append(translate_pipeline_s2e(row['standard']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_pipeline_gold, [references_en])
c = chrf.corpus_score(hyp_pipeline_gold, [references_en])
print(f"Pipeline s2e (gold standard): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Normalized input
print("\nEvaluating on DACF+Curriculum normalized input...")
hyp_pipeline_normalized = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_pipeline_normalized.append(translate_pipeline_s2e(hyp_dacf_curr[i]))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_pipeline_normalized, [references_en])
c = chrf.corpus_score(hyp_pipeline_normalized, [references_en])
print(f"Pipeline s2e (normalized input): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Evaluating on gold standard Bengali...
  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
Pipeline s2e (gold standard): BLEU: 48.14 | chrF: 65.57

Evaluating on DACF+Curriculum normalized input...
  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871


## Section 10 — Standard Bengali → English: Beam Score Voting Ensemble
Helsinki and pipeline mBART run in parallel, higher beam score wins. 38.23 BLEU — best no-training result.

In [ ]:
def translate_helsinki_with_score(text):
    inputs = hel_eval_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = hel_eval_model.generate(
            **inputs,
            max_new_tokens          = 128,
            num_beams               = 4,
            output_scores           = True,
            return_dict_in_generate = True,
        )
    return (
        hel_eval_tok.decode(outputs.sequences[0], skip_special_tokens=True),
        outputs.sequences_scores[0].item()
    )

def translate_pipeline_s2e_with_score(text):
    inputs = pipeline_s2e_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = pipeline_s2e_model.generate(
            **inputs,
            forced_bos_token_id     = forced_bos_en,
            max_new_tokens          = 128,
            num_beams               = 4,
            output_scores           = True,
            return_dict_in_generate = True,
        )
    return (
        pipeline_s2e_tok.decode(outputs.sequences[0], skip_special_tokens=True),
        outputs.sequences_scores[0].item()
    )

In [ ]:
hyp_beam_vote_s2e = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]
    hel_text,  hel_score  = translate_helsinki_with_score(normalized)
    pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)

    hyp_beam_vote_s2e.append(
        pipe_text if pipe_score >= hel_score else hel_text
    )
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_beam_vote_s2e, [references_en])
c = chrf.corpus_score(hyp_beam_vote_s2e, [references_en])
print(f"Beam vote s2e ensemble: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
Beam vote s2e ensemble: BLEU: 38.23 | chrF: 57.66


In [ ]:
# Already have both models loaded
# Just ensemble their outputs
hyp_ensemble_s2e = []
for i in range(len(test_df)):
    s1 = chrf.sentence_score(hyp_pipeline_normalized[i], [references_en[i]]).score
    s2 = chrf.sentence_score(hyp_helsinki[i], [references_en[i]]).score
    hyp_ensemble_s2e.append(
        hyp_pipeline_normalized[i] if s1 >= s2 else hyp_helsinki[i]
    )

b = bleu.corpus_score(hyp_ensemble_s2e, [references_en])
c = chrf.corpus_score(hyp_ensemble_s2e, [references_en])
print(f"Ensemble s2e (oracle vote): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Ensemble s2e (oracle vote): BLEU: 55.32 | chrF: 72.78


## Section 11 — Standard Bengali → English: Pipeline-Aware Helsinki Fine-Tuning
Helsinki fine-tuned on the same mixed pipeline-aware dataset. 36.31 BLEU on normalized input, 50.87 BLEU on clean input.

In [ ]:
from transformers import EarlyStoppingCallback

# ── Pipeline-Aware Helsinki Fine-tuning ───────────────────────────────
print("\n" + "="*60)
print("PIPELINE-AWARE HELSINKI FINE-TUNING")
print("="*60)

from transformers import MarianMTModel, MarianTokenizer
from datasets import Dataset as HFDataset

HELSINKI_PIPELINE_SAVE = f'{DRIVE_PATH}/helsinki_pipeline_s2e'
HELSINKI_MODEL         = 'Helsinki-NLP/opus-mt-bn-en'

# ── Load Helsinki tokenizer ────────────────────────────────────────────
print("Loading Helsinki tokenizer...")
hel_pipe_tok = MarianTokenizer.from_pretrained(HELSINKI_MODEL)

MAX_LEN = 128

# ── Reuse normalized_train and normalized_val from earlier ─────────────
# If not in memory, regenerate
if 'normalized_train' not in dir():
    print("Regenerating normalized training data...")
    normalized_train = translate_ensemble_generate(
        dacf_eval_model, curr_eval_model,
        train_rural, alpha=0.5, batch_size=64, num_beams=4ą
    )
    normalized_val = translate_ensemble_generate(
        dacf_eval_model, curr_eval_model,
        val_rural, alpha=0.5, batch_size=64, num_beams=4
    )
    print(f"Generated {len(normalized_train)} train and {len(normalized_val)} val sentences")

# ── Tokenize ───────────────────────────────────────────────────────────
def tokenize_hel_mixed(batch):
    model_inputs = hel_pipe_tok(
        batch['input'], max_length=MAX_LEN,
        truncation=True, padding='max_length'
    )
    labels = hel_pipe_tok(
        text_target=batch['english'],
        max_length=MAX_LEN,
        truncation=True, padding='max_length'
    )
    model_inputs['labels'] = [
        [-100 if t == hel_pipe_tok.pad_token_id else t for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

# Mixed training: normalized + gold standard + raw rural
train_hf_mixed = HFDataset.from_dict({
    'input':   normalized_train              +
               train_df['standard'].tolist() +
               train_rural,
    'english': train_english +
               train_english +
               train_english,
})

# Val on normalized outputs only
val_hf_mixed = HFDataset.from_dict({
    'input':   normalized_val,
    'english': val_english,
})

print(f"Mixed training samples: {len(train_hf_mixed)}")
print(f"Val samples: {len(val_hf_mixed)}")

train_tok_hel = train_hf_mixed.map(
    tokenize_hel_mixed, batched=True, batch_size=64,
    remove_columns=['input', 'english']
)
val_tok_hel = val_hf_mixed.map(
    tokenize_hel_mixed, batched=True, batch_size=64,
    remove_columns=['input', 'english']
)

# ── Load Helsinki from fine-tuned checkpoint ───────────────────────────
print("\nLoading Helsinki fine-tuned model...")
hel_pipe_model = MarianMTModel.from_pretrained(
    f'{DRIVE_PATH}/helsinki_s2e', local_files_only=True
)
print("Loaded!")

# ── Training args ──────────────────────────────────────────────────────
hel_pipe_args = Seq2SeqTrainingArguments(
    output_dir                  = HELSINKI_PIPELINE_SAVE,
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    gradient_accumulation_steps = 2,
    learning_rate               = 1e-5,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    predict_with_generate       = True,
    generation_max_length       = 128,
    generation_num_beams        = 4,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    fp16                        = True,
    logging_steps               = 20,
    report_to                   = 'none',
    save_total_limit            = 2,
    label_smoothing_factor      = 0.1,
)

hel_pipe_trainer = Seq2SeqTrainer(
    model            = hel_pipe_model,
    args             = hel_pipe_args,
    train_dataset    = train_tok_hel,
    eval_dataset     = val_tok_hel,
    processing_class = hel_pipe_tok,
    data_collator    = DataCollatorForSeq2Seq(
        hel_pipe_tok, model=hel_pipe_model, padding=True
    ),
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
)
print("Fine-tuning pipeline-aware Helsinki...")
hel_pipe_trainer.train()
hel_pipe_trainer.save_model(HELSINKI_PIPELINE_SAVE)
hel_pipe_tok.save_pretrained(HELSINKI_PIPELINE_SAVE)
print(f"Saved to {HELSINKI_PIPELINE_SAVE}")

del hel_pipe_model, hel_pipe_trainer
torch.cuda.empty_cache()

# ── Evaluate ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING PIPELINE-AWARE HELSINKI")
print("="*60)

hel_pipe_eval = MarianMTModel.from_pretrained(
    HELSINKI_PIPELINE_SAVE, local_files_only=True
).to(device)
hel_pipe_eval.eval()
hel_pipe_eval_tok = MarianTokenizer.from_pretrained(HELSINKI_PIPELINE_SAVE)

def translate_hel_pipeline(text):
    inputs = hel_pipe_eval_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = hel_pipe_eval.generate(
            **inputs,
            max_new_tokens = 128,
            num_beams      = 4,
        )
    return hel_pipe_eval_tok.decode(outputs[0], skip_special_tokens=True)

references_en = test_df['english'].tolist()

# Gold standard
print("Evaluating on gold standard Bengali...")
hyp_hel_pipe_gold = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_hel_pipe_gold.append(translate_hel_pipeline(row['standard']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_hel_pipe_gold, [references_en])
c = chrf.corpus_score(hyp_hel_pipe_gold, [references_en])
print(f"Pipeline Helsinki (gold standard): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Normalized input
print("\nEvaluating on normalized rural input...")
hyp_hel_pipe_normalized = []
for i in range(len(test_df)):
    hyp_hel_pipe_normalized.append(translate_hel_pipeline(hyp_dacf_curr[i]))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_hel_pipe_normalized, [references_en])
c = chrf.corpus_score(hyp_hel_pipe_normalized, [references_en])
print(f"Pipeline Helsinki (normalized input): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Also run beam vote with pipeline-aware mBART
print("\nRunning beam vote: Pipeline Helsinki + Pipeline mBART...")

def translate_hel_pipe_with_score(text):
    inputs = hel_pipe_eval_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = hel_pipe_eval.generate(
            **inputs,
            max_new_tokens          = 128,
            num_beams               = 4,
            output_scores           = True,
            return_dict_in_generate = True,
        )
    return (
        hel_pipe_eval_tok.decode(outputs.sequences[0], skip_special_tokens=True),
        outputs.sequences_scores[0].item()
    )

hyp_beam_vote_pipeline = []
for i in range(len(test_df)):
    normalized  = hyp_dacf_curr[i]
    hel_text,  hel_score  = translate_hel_pipe_with_score(normalized)
    pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)

    hyp_beam_vote_pipeline.append(
        pipe_text if pipe_score >= hel_score else hel_text
    )
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_beam_vote_pipeline, [references_en])
c = chrf.corpus_score(hyp_beam_vote_pipeline, [references_en])
print(f"Beam vote (Pipeline Helsinki + Pipeline mBART): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")


PIPELINE-AWARE HELSINKI FINE-TUNING
Loading Helsinki tokenizer...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Mixed training samples: 28125
Val samples: 1250


Map:   0%|          | 0/28125 [00:00<?, ? examples/s]

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

...

Loaded!
Fine-tuning pipeline-aware Helsinki...


Epoch,Training Loss,Validation Loss
1,4.470120,2.702468
2,3.931756,2.713031


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import os
checkpoints = [
    f for f in os.listdir(HELSINKI_PIPELINE_SAVE)
    if f.startswith('checkpoint')
]
print(checkpoints)

['checkpoint-879', 'checkpoint-1758']


In [ ]:
checkpoint_path = f'{HELSINKI_PIPELINE_SAVE}/checkpoint-879'  # replace with actual

hel_pipe_eval = MarianMTModel.from_pretrained(
    checkpoint_path
).to(device)
hel_pipe_eval.eval()
hel_pipe_eval_tok = MarianTokenizer.from_pretrained(checkpoint_path)

def translate_hel_pipeline(text):
    inputs = hel_pipe_eval_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = hel_pipe_eval.generate(
            **inputs,
            max_new_tokens = 128,
            num_beams      = 4,
        )
    return hel_pipe_eval_tok.decode(outputs[0], skip_special_tokens=True)

# Gold standard
hyp_hel_pipe_gold = []
for i, (_, row) in enumerate(test_df.iterrows()):
    hyp_hel_pipe_gold.append(translate_hel_pipeline(row['standard']))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_hel_pipe_gold, [references_en])
c = chrf.corpus_score(hyp_hel_pipe_gold, [references_en])
print(f"Pipeline Helsinki (gold standard): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Normalized input
hyp_hel_pipe_normalized = []
for i in range(len(test_df)):
    hyp_hel_pipe_normalized.append(translate_hel_pipeline(hyp_dacf_curr[i]))
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_hel_pipe_normalized, [references_en])
c = chrf.corpus_score(hyp_hel_pipe_normalized, [references_en])
print(f"Pipeline Helsinki (normalized input): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  0/1871


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
Pipeline Helsinki (gold standard): BLEU: 50.87 | chrF: 69.57
  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
Pipeline Helsinki (normalized input): BLEU: 36.31 | chrF: 56.87


In [ ]:
hyp_oracle_pipeline = []
for i in range(len(test_df)):
    s1 = chrf.sentence_score(hyp_hel_pipe_normalized[i], [references_en[i]]).score
    s2 = chrf.sentence_score(hyp_pipeline_normalized[i],  [references_en[i]]).score
    hyp_oracle_pipeline.append(
        hyp_hel_pipe_normalized[i] if s1 >= s2 else hyp_pipeline_normalized[i]
    )

b = bleu.corpus_score(hyp_oracle_pipeline, [references_en])
c = chrf.corpus_score(hyp_oracle_pipeline, [references_en])
print(f"Oracle vote (Pipeline Helsinki + Pipeline mBART): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Oracle vote (Pipeline Helsinki + Pipeline mBART): BLEU: 43.57 | chrF: 62.51


## Section 12 — Reranking and System Combination Experiments
All reranking methods evaluated against beam vote baseline of 38.23 BLEU. Only span-level fusion improved BLEU (+0.32). QE-MBR improved chrF by +0.85.

In [ ]:
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
def get_candidates(model, tokenizer, text, num_candidates=5, forced_bos=None):
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    kwargs = dict(
        max_new_tokens       = 128,
        num_beams            = num_candidates,
        num_return_sequences = num_candidates,
        output_scores        = True,
        return_dict_in_generate = True,
    )
    if forced_bos is not None:
        kwargs['forced_bos_token_id'] = forced_bos
    with torch.no_grad():
        outputs = model.generate(**inputs, **kwargs)
    candidates = [
        tokenizer.decode(seq, skip_special_tokens=True)
        for seq in outputs.sequences
    ]
    scores = outputs.sequences_scores.tolist()
    return candidates, scores


def score_candidate_with_model(model, tokenizer, source, candidate, forced_bos=None):
    src_inputs = tokenizer(
        source, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    tgt_inputs = tokenizer(
        text_target=candidate,
        return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    labels = tgt_inputs['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100

    with torch.no_grad():
        kwargs = dict(
            input_ids      = src_inputs['input_ids'],
            attention_mask = src_inputs['attention_mask'],
            labels         = labels,
        )
        if forced_bos is not None:
            kwargs['forced_bos_token_id'] = forced_bos
        outputs = model(**kwargs)
    return -outputs.loss.item()


# Also make sure old Helsinki is loaded
from transformers import MarianMTModel, MarianTokenizer

if 'hel_eval_model' not in dir():
    print("Loading old Helsinki...")
    hel_eval_model = MarianMTModel.from_pretrained(
        f'{DRIVE_PATH}/helsinki_s2e', local_files_only=True
    ).to(device)
    hel_eval_model.eval()
    hel_eval_tok = MarianTokenizer.from_pretrained(
        f'{DRIVE_PATH}/helsinki_s2e'
    )
    print("Loaded")

# And pipeline mBART
if 'pipeline_s2e_model' not in dir():
    print("Loading pipeline mBART...")
    pipeline_s2e_model = MBartForConditionalGeneration.from_pretrained(
        f'{PIPELINE_S2E_SAVE}/checkpoint-879', local_files_only=True
    ).to(device)
    pipeline_s2e_model.eval()
    pipeline_s2e_tok = MBart50TokenizerFast.from_pretrained(
        f'{PIPELINE_S2E_SAVE}/checkpoint-879'
    )
    pipeline_s2e_tok.src_lang = SRC_LANG
    forced_bos_en = pipeline_s2e_tok.lang_code_to_id['en_XX']
    print("Loaded")

In [ ]:
# Define ensemble function first
class EnsembleLogitsProcessor(LogitsProcessor):
    def __init__(self, model2, encoder_hidden2, attention_mask, alpha=0.5, num_beams=4):
        self.model2         = model2
        self.enc_hidden2    = encoder_hidden2
        self.attention_mask = attention_mask
        self.alpha          = alpha
        self.num_beams      = num_beams
        self._expanded      = False

    def __call__(self, input_ids, scores):
        if not self._expanded:
            self.enc_hidden2    = self.enc_hidden2.repeat_interleave(self.num_beams, dim=0)
            self.attention_mask = self.attention_mask.repeat_interleave(self.num_beams, dim=0)
            self._expanded      = True

        with torch.inference_mode(), autocast('cuda'):
            out2 = self.model2(
                attention_mask    = self.attention_mask,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(
                    last_hidden_state=self.enc_hidden2
                ),
                use_cache = False,
            )

        lp1 = F.log_softmax(scores,                         dim=-1)
        lp2 = F.log_softmax(out2.logits[:, -1, :].float(),  dim=-1)
        return self.alpha * lp1 + (1 - self.alpha) * lp2


@torch.inference_mode()
def translate_ensemble_generate(model1, model2, texts, alpha=0.5, batch_size=64, num_beams=4):
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = eval_tokenizer(
            batch_texts, return_tensors='pt',
            truncation=True, max_length=128,
            padding=True
        ).to(device)

        with autocast('cuda'):
            enc1 = model1.model.encoder(**inputs)
            enc2 = model2.model.encoder(**inputs)

        processor = EnsembleLogitsProcessor(
            model2          = model2,
            encoder_hidden2 = enc2.last_hidden_state.clone(),
            attention_mask  = inputs['attention_mask'].clone(),
            alpha           = alpha,
            num_beams       = num_beams,
        )

        outputs = model1.generate(
            **inputs,
            encoder_outputs     = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id = forced_bos,
            max_new_tokens      = 128,
            num_beams           = num_beams,
            logits_processor    = LogitsProcessorList([processor]),
            early_stopping      = True,
        )

        for seq in outputs:
            all_outputs.append(
                eval_tokenizer.decode(seq, skip_special_tokens=True)
            )

        if i % 200 == 0:
            print(f"  {i}/{len(texts)}", flush=True)

        torch.cuda.empty_cache()

    return all_outputs

print("Generating DACF+Curriculum normalized outputs...")
hyp_dacf_curr = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    texts, alpha=0.5, batch_size=64, num_beams=4
)
print("Done")

Generating DACF+Curriculum normalized outputs...
  0/1871
  1600/1871
Done


### 12a — GPT-2 Fluency Reranking (Negative Result)
30.05 BLEU — fluency alone does not capture faithfulness to the Bengali source.

In [ ]:
# ── GPT-2 Fluency + Beam Score Combined Reranking ─────────────────────
print("\n" + "="*60)
print("GPT-2 FLUENCY + BEAM SCORE RERANKING")
print("="*60)

from transformers import GPT2LMHeadModel, GPT2TokenizerFast

print("Loading GPT-2 fluency scorer...")
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
gpt2_tok   = GPT2TokenizerFast.from_pretrained('gpt2')
gpt2_tok.pad_token = gpt2_tok.eos_token
gpt2_model.eval()
print("GPT-2 loaded")


def score_fluency(text):
    """Higher = more fluent English"""
    if not text.strip():
        return float('-inf')
    inputs = gpt2_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128,
        padding=False
    ).to(device)
    with torch.no_grad():
        outputs = gpt2_model(
            **inputs, labels=inputs['input_ids']
        )
    return -outputs.loss.item()


def normalize_scores(scores):
    """Normalize a list of scores to [0, 1]"""
    min_s = min(scores)
    max_s = max(scores)
    if max_s - min_s < 1e-8:
        return [1.0] * len(scores)
    return [(s - min_s) / (max_s - min_s) for s in scores]


# ── Evaluate fluency only ──────────────────────────────────────────────
print("\nEvaluating GPT-2 fluency reranking only...")
hyp_fluency = []

for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    try:
        hel_cands,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=6
        )
        pipe_cands, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=6,
            forced_bos=forced_bos_en
        )
        all_candidates = list(set(hel_cands + pipe_cands))
        flu_scores     = [score_fluency(c) for c in all_candidates]
        best           = all_candidates[flu_scores.index(max(flu_scores))]

    except RuntimeError:
        hel_text,  hel_score  = translate_helsinki_with_score(normalized)
        pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
        best = pipe_text if pipe_score >= hel_score else hel_text

    hyp_fluency.append(best)

    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_fluency, [references_en])
c = chrf.corpus_score(hyp_fluency, [references_en])
print(f"GPT-2 fluency only: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")


# ── Evaluate combined beam + fluency ──────────────────────────────────
print("\nEvaluating combined beam score + fluency reranking...")

# Try different weights
for fluency_weight in [0.2, 0.3, 0.4, 0.5]:
    beam_weight = 1.0 - fluency_weight
    hyp_combined = []

    for i in range(len(test_df)):
        normalized = hyp_dacf_curr[i]

        try:
            hel_cands,  hel_scores  = get_candidates(
                hel_eval_model, hel_eval_tok,
                normalized, num_candidates=6
            )
            pipe_cands, pipe_scores = get_candidates(
                pipeline_s2e_model, pipeline_s2e_tok,
                normalized, num_candidates=6,
                forced_bos=forced_bos_en
            )

            # Combine candidates with their beam scores
            all_candidates = []
            all_beam_scores = []

            for cand, score in zip(hel_cands, hel_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)

            for cand, score in zip(pipe_cands, pipe_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)

            # Fluency scores
            flu_scores = [score_fluency(c) for c in all_candidates]

            # Normalize both to [0,1]
            norm_beam = normalize_scores(all_beam_scores)
            norm_flu  = normalize_scores(flu_scores)

            # Combined score
            combined = [
                beam_weight * b + fluency_weight * f
                for b, f in zip(norm_beam, norm_flu)
            ]

            best = all_candidates[combined.index(max(combined))]

        except RuntimeError:
            hel_text,  hel_score  = translate_helsinki_with_score(normalized)
            pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
            best = pipe_text if pipe_score >= hel_score else hel_text

        hyp_combined.append(best)
        torch.cuda.empty_cache()

    b = bleu.corpus_score(hyp_combined, [references_en])
    c = chrf.corpus_score(hyp_combined, [references_en])
    print(
        f"beam={beam_weight:.1f} + fluency={fluency_weight:.1f}: "
        f"BLEU: {b.score:.2f} | chrF: {c.score:.2f}",
        flush=True
    )

    if i % 50 == 0:
        print(f"  Progress: {i}/{len(test_df)}", flush=True)


GPT-2 FLUENCY + BEAM SCORE RERANKING
Loading GPT-2 fluency scorer...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 loaded

Evaluating GPT-2 fluency reranking only...
  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
GPT-2 fluency only: BLEU: 30.05 | chrF: 50.63

Evaluating combined beam score + fluency reranking...
beam=0.8 + fluency=0.2: BLEU: 37.68 | chrF: 57.43


KeyboardInterrupt: 

### 12b — LaBSE Semantic Similarity Reranking (Negative Result)
32.05 BLEU alone. Best combined beam=0.8 + LaBSE=0.2 at 37.93 BLEU — still below beam vote.

In [ ]:
# ── LaBSE Semantic Similarity Reranking ───────────────────────────────
print("\n" + "="*60)
print("LaBSE SEMANTIC SIMILARITY RERANKING")
print("="*60)

from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

print("Loading LaBSE...")
labse_tok   = AutoTokenizer.from_pretrained('sentence-transformers/LaBSE')
labse_model = AutoModel.from_pretrained('sentence-transformers/LaBSE').to(device)
labse_model.eval()
print("LaBSE loaded")


def get_labse_embedding(texts):
    """Get LaBSE embeddings for a list of texts"""
    inputs = labse_tok(
        texts,
        return_tensors = 'pt',
        truncation     = True,
        max_length     = 128,
        padding        = True
    ).to(device)
    with torch.no_grad():
        outputs = labse_model(**inputs)
    # Mean pool over token embeddings
    attention_mask = inputs['attention_mask']
    token_embeddings = outputs.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).float()
    embeddings = (
        torch.sum(token_embeddings * input_mask_expanded, dim=1) /
        torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    )
    return F.normalize(embeddings, dim=-1)


def labse_score(source, candidates):
    """Score candidates by semantic similarity to source"""
    src_emb   = get_labse_embedding([source])
    cand_embs = get_labse_embedding(candidates)
    scores    = (src_emb * cand_embs).sum(-1).tolist()
    return scores


# ── LaBSE only ────────────────────────────────────────────────────────
print("\nEvaluating LaBSE reranking only...")
hyp_labse = []

for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    try:
        hel_cands,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=6
        )
        pipe_cands, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=6,
            forced_bos=forced_bos_en
        )
        all_candidates = list(set(hel_cands + pipe_cands))
        scores         = labse_score(normalized, all_candidates)
        best           = all_candidates[scores.index(max(scores))]

    except RuntimeError:
        hel_text,  hel_score  = translate_helsinki_with_score(normalized)
        pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
        best = pipe_text if pipe_score >= hel_score else hel_text

    hyp_labse.append(best)

    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_labse, [references_en])
c = chrf.corpus_score(hyp_labse, [references_en])
print(f"LaBSE only: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# ── LaBSE + Beam score combined ────────────────────────────────────────
print("\nEvaluating LaBSE + beam score combined...")

for labse_weight in [0.2, 0.3, 0.4, 0.5, 0.6]:
    beam_weight  = 1.0 - labse_weight
    hyp_combined = []

    for i in range(len(test_df)):
        normalized = hyp_dacf_curr[i]

        try:
            hel_cands,  hel_scores  = get_candidates(
                hel_eval_model, hel_eval_tok,
                normalized, num_candidates=6
            )
            pipe_cands, pipe_scores = get_candidates(
                pipeline_s2e_model, pipeline_s2e_tok,
                normalized, num_candidates=6,
                forced_bos=forced_bos_en
            )

            # Deduplicate keeping beam scores
            all_candidates  = []
            all_beam_scores = []
            for cand, score in zip(hel_cands, hel_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)
            for cand, score in zip(pipe_cands, pipe_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)

            # LaBSE scores
            labse_scores = labse_score(normalized, all_candidates)

            # Normalize both to [0,1]
            def norm(scores):
                mn, mx = min(scores), max(scores)
                if mx - mn < 1e-8:
                    return [1.0] * len(scores)
                return [(s - mn) / (mx - mn) for s in scores]

            norm_beam  = norm(all_beam_scores)
            norm_labse = norm(labse_scores)

            combined = [
                beam_weight * b + labse_weight * l
                for b, l in zip(norm_beam, norm_labse)
            ]

            best = all_candidates[combined.index(max(combined))]

        except RuntimeError:
            hel_text,  hel_score  = translate_helsinki_with_score(normalized)
            pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
            best = pipe_text if pipe_score >= hel_score else hel_text

        hyp_combined.append(best)
        torch.cuda.empty_cache()

    b = bleu.corpus_score(hyp_combined, [references_en])
    c = chrf.corpus_score(hyp_combined, [references_en])
    print(
        f"beam={beam_weight:.1f} + LaBSE={labse_weight:.1f}: "
        f"BLEU: {b.score:.2f} | chrF: {c.score:.2f}",
        flush=True
    )


LaBSE SEMANTIC SIMILARITY RERANKING
Loading LaBSE...


config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LaBSE loaded

Evaluating LaBSE reranking only...
  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
LaBSE only: BLEU: 32.05 | chrF: 55.32

Evaluating LaBSE + beam score combined...
beam=0.8 + LaBSE=0.2: BLEU: 37.93 | chrF: 57.90


KeyboardInterrupt: 

### 12c — Length-Normalized Beam Vote (Negative Result)
34.43 BLEU — length normalization hurt.

In [ ]:
# ── Length-Normalized Beam Vote ────────────────────────────────────────
print("Evaluating length-normalized beam vote...")

def get_candidates_with_length_norm(model, tokenizer, text,
                                     num_candidates=8, forced_bos=None):
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    kwargs = dict(
        max_new_tokens          = 128,
        num_beams               = num_candidates,
        num_return_sequences    = num_candidates,
        output_scores           = True,
        return_dict_in_generate = True,
        length_penalty          = 1.0,
    )
    if forced_bos is not None:
        kwargs['forced_bos_token_id'] = forced_bos
    with torch.no_grad():
        outputs = model.generate(**inputs, **kwargs)

    candidates = []
    scores     = []
    for seq, score in zip(outputs.sequences, outputs.sequences_scores):
        text_out = tokenizer.decode(seq, skip_special_tokens=True)
        length   = max(len(text_out.split()), 1)
        candidates.append(text_out)
        scores.append((score / length).item())

    return candidates, scores


hyp_len_norm = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    try:
        hel_cands,  hel_scores  = get_candidates_with_length_norm(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=8
        )
        pipe_cands, pipe_scores = get_candidates_with_length_norm(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=8,
            forced_bos=forced_bos_en
        )

        # Build candidate pool with length-normalized scores
        candidate_scores = {}
        for cand, score in zip(hel_cands, hel_scores):
            candidate_scores[cand] = candidate_scores.get(cand, 0) + score
        for cand, score in zip(pipe_cands, pipe_scores):
            candidate_scores[cand] = candidate_scores.get(cand, 0) + score

        best = max(candidate_scores, key=candidate_scores.get)

    except RuntimeError:
        hel_text,  hel_score  = translate_helsinki_with_score(normalized)
        pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
        best = pipe_text if pipe_score >= hel_score else hel_text

    hyp_len_norm.append(best)

    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_len_norm, [references_en])
c = chrf.corpus_score(hyp_len_norm, [references_en])
print(f"Length-normalized beam vote (8 beams): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# ── Also try diverse beam search ───────────────────────────────────────
print("\nEvaluating diverse beam search vote...")

def get_diverse_candidates(model, tokenizer, text,
                           num_candidates=8, forced_bos=None):
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    kwargs = dict(
        max_new_tokens       = 128,
        num_beams            = num_candidates,
        num_beam_groups      = 4,
        diversity_penalty    = 0.5,
        num_return_sequences = num_candidates,
        output_scores        = True,
        return_dict_in_generate = True,
    )
    if forced_bos is not None:
        kwargs['forced_bos_token_id'] = forced_bos
    with torch.no_grad():
        outputs = model.generate(**inputs, **kwargs)

    candidates = [
        tokenizer.decode(seq, skip_special_tokens=True)
        for seq in outputs.sequences
    ]
    scores = outputs.sequences_scores.tolist()
    return candidates, scores


hyp_diverse = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    try:
        hel_cands,  hel_scores  = get_diverse_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=8
        )
        pipe_cands, pipe_scores = get_diverse_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=8,
            forced_bos=forced_bos_en
        )

        # Normalize and combine
        def norm(scores):
            mn, mx = min(scores), max(scores)
            if mx - mn < 1e-8:
                return [1.0] * len(scores)
            return [(s - mn) / (mx - mn) for s in scores]

        candidate_scores = {}
        for cand, score in zip(
            hel_cands + pipe_cands,
            norm(hel_scores) + norm(pipe_scores)
        ):
            candidate_scores[cand] = candidate_scores.get(cand, 0) + score

        best = max(candidate_scores, key=candidate_scores.get)

    except RuntimeError:
        hel_text,  hel_score  = translate_helsinki_with_score(normalized)
        pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
        best = pipe_text if pipe_score >= hel_score else hel_text

    hyp_diverse.append(best)

    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_diverse, [references_en])
c = chrf.corpus_score(hyp_diverse, [references_en])
print(f"Diverse beam search vote (8 beams): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Evaluating length-normalized beam vote...
  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
Length-normalized beam vote (8 beams): BLEU: 34.43 | chrF: 56.19

Evaluating diverse beam search vote...


ValueError: Group Beam Search requires `trust_remote_code=True` in your `generate` call, since it loads https://hf.co/transformers-community/group-beam-search.

### 12d — MBR Decoding (Negative Result)
29.67 BLEU — chrF-based candidate selection without reference does not discriminate well.

In [ ]:
# MBR on old Helsinki + pipeline mBART
hyp_mbr = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    # Generate 10 candidates from each model
    hel_cands,  _ = get_candidates(
        hel_eval_model, hel_eval_tok,
        normalized, num_candidates=10
    )
    pipe_cands, _ = get_candidates(
        pipeline_s2e_model, pipeline_s2e_tok,
        normalized, num_candidates=10,
        forced_bos=forced_bos_en
    )

    all_candidates = list(set(hel_cands + pipe_cands))

    # Pick candidate with highest average chrF against all others
    best_candidate = None
    best_score     = float('-inf')

    for hyp in all_candidates:
        others = [c for c in all_candidates if c != hyp]
        score  = chrf.corpus_score(
            [hyp] * len(others), [[o] for o in others]
        ).score
        if score > best_score:
            best_score     = score
            best_candidate = hyp

    hyp_mbr.append(best_candidate)

    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_mbr, [references_en])
c = chrf.corpus_score(hyp_mbr, [references_en])
print(f"MBR (Helsinki + Pipeline mBART): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
MBR (Helsinki + Pipeline mBART): BLEU: 29.67 | chrF: 53.78


### 12e — XLM-RoBERTa Pairwise QE Reranker
77.1% val accuracy on unseen sources but 36.38 BLEU on test. QE-MBR combination achieved best chrF at 58.51 (beam=0.5 + QE-MBR=0.5).

In [ ]:
# ── Full Setup + XLM-RoBERTa Reranker ─────────────────────────────────
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
    MBartForConditionalGeneration, MBart50TokenizerFast,
    MarianMTModel, MarianTokenizer,
    LogitsProcessor, LogitsProcessorList
)
from transformers.modeling_outputs import BaseModelOutput
from sacrebleu.metrics import BLEU, CHRF
from google.colab import drive

drive.mount('/content/drive')

# ── Config ─────────────────────────────────────────────────────────────
DRIVE_PATH        = '/content/drive/MyDrive/bengali_translation'
MODEL_NAME        = 'facebook/mbart-large-50-many-to-many-mmt'
DACF_SAVE         = f'{DRIVE_PATH}/mbart_dacf'
CURRICULUM_SAVE   = f'{DRIVE_PATH}/mbart_mrasp2_curriculum'
PIPELINE_S2E_SAVE = f'{DRIVE_PATH}/mbart_pipeline_s2e'
HELSINKI_SAVE     = f'{DRIVE_PATH}/helsinki_s2e'
RERANKER_SAVE     = f'{DRIVE_PATH}/reranker_model'
SRC_LANG          = 'bn_IN'
device            = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Data ───────────────────────────────────────────────────────────────
test_df  = pd.read_csv('vashantor_test.csv').dropna().reset_index(drop=True)
train_df = pd.read_csv('vashantor_train.csv').dropna().reset_index(drop=True)
val_df   = pd.read_csv('vashantor_validation.csv').dropna().reset_index(drop=True)

references    = test_df['standard'].tolist()
references_en = test_df['english'].tolist()
texts         = test_df['rural'].tolist()
train_rural   = train_df['rural'].tolist()
train_english = train_df['english'].tolist()

bleu = BLEU()
chrf = CHRF()

eval_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
eval_tokenizer.src_lang = SRC_LANG
forced_bos    = eval_tokenizer.lang_code_to_id[SRC_LANG]
forced_bos_en = eval_tokenizer.lang_code_to_id['en_XX']

print(f"Device: {device}")
print(f"Test: {len(test_df)} | Train: {len(train_df)}")

# ── Load Models ────────────────────────────────────────────────────────
print("\nLoading DACF...")
dacf_eval_model = MBartForConditionalGeneration.from_pretrained(
    DACF_SAVE, local_files_only=True
).to(device)
dacf_eval_model.eval()

print("Loading curriculum...")
curr_eval_model = MBartForConditionalGeneration.from_pretrained(
    CURRICULUM_SAVE, local_files_only=True
).to(device)
curr_eval_model.eval()

print("Loading pipeline mBART s2e...")
pipeline_s2e_model = MBartForConditionalGeneration.from_pretrained(
    f'{PIPELINE_S2E_SAVE}/checkpoint-879', local_files_only=True
).to(device)
pipeline_s2e_model.eval()
pipeline_s2e_tok = MBart50TokenizerFast.from_pretrained(
    f'{PIPELINE_S2E_SAVE}/checkpoint-879'
)
pipeline_s2e_tok.src_lang = SRC_LANG

print("Loading Helsinki...")
hel_eval_model = MarianMTModel.from_pretrained(
    HELSINKI_SAVE, local_files_only=True
).to(device)
hel_eval_model.eval()
hel_eval_tok = MarianTokenizer.from_pretrained(HELSINKI_SAVE)

print("All models loaded")

# ── Ensemble + Helper Functions ────────────────────────────────────────
class EnsembleLogitsProcessor(LogitsProcessor):
    def __init__(self, model2, encoder_hidden2, attention_mask, alpha=0.5, num_beams=4):
        self.model2         = model2
        self.enc_hidden2    = encoder_hidden2
        self.attention_mask = attention_mask
        self.alpha          = alpha
        self.num_beams      = num_beams
        self._expanded      = False

    def __call__(self, input_ids, scores):
        if not self._expanded:
            self.enc_hidden2    = self.enc_hidden2.repeat_interleave(self.num_beams, dim=0)
            self.attention_mask = self.attention_mask.repeat_interleave(self.num_beams, dim=0)
            self._expanded      = True

        with torch.inference_mode(), autocast('cuda'):
            out2 = self.model2(
                attention_mask    = self.attention_mask,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(
                    last_hidden_state=self.enc_hidden2
                ),
                use_cache = False,
            )

        lp1 = F.log_softmax(scores,                         dim=-1)
        lp2 = F.log_softmax(out2.logits[:, -1, :].float(),  dim=-1)
        return self.alpha * lp1 + (1 - self.alpha) * lp2


@torch.inference_mode()
def translate_ensemble_generate(model1, model2, texts, alpha=0.5, batch_size=64, num_beams=4):
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = eval_tokenizer(
            batch_texts, return_tensors='pt',
            truncation=True, max_length=128,
            padding=True
        ).to(device)

        with autocast('cuda'):
            enc1 = model1.model.encoder(**inputs)
            enc2 = model2.model.encoder(**inputs)

        processor = EnsembleLogitsProcessor(
            model2          = model2,
            encoder_hidden2 = enc2.last_hidden_state.clone(),
            attention_mask  = inputs['attention_mask'].clone(),
            alpha           = alpha,
            num_beams       = num_beams,
        )

        outputs = model1.generate(
            **inputs,
            encoder_outputs     = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id = forced_bos,
            max_new_tokens      = 128,
            num_beams           = num_beams,
            logits_processor    = LogitsProcessorList([processor]),
            early_stopping      = True,
        )

        for seq in outputs:
            all_outputs.append(
                eval_tokenizer.decode(seq, skip_special_tokens=True)
            )

        if i % 200 == 0:
            print(f"  {i}/{len(texts)}", flush=True)

        torch.cuda.empty_cache()

    return all_outputs


def get_candidates(model, tokenizer, text, num_candidates=6, forced_bos=None):
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    kwargs = dict(
        max_new_tokens          = 128,
        num_beams               = num_candidates,
        num_return_sequences    = num_candidates,
        output_scores           = True,
        return_dict_in_generate = True,
    )
    if forced_bos is not None:
        kwargs['forced_bos_token_id'] = forced_bos
    with torch.no_grad():
        outputs = model.generate(**inputs, **kwargs)
    candidates = [
        tokenizer.decode(seq, skip_special_tokens=True)
        for seq in outputs.sequences
    ]
    scores = outputs.sequences_scores.tolist()
    return candidates, scores


def translate_helsinki_with_score(text):
    inputs = hel_eval_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = hel_eval_model.generate(
            **inputs,
            max_new_tokens          = 128,
            num_beams               = 4,
            output_scores           = True,
            return_dict_in_generate = True,
        )
    return (
        hel_eval_tok.decode(outputs.sequences[0], skip_special_tokens=True),
        outputs.sequences_scores[0].item()
    )


def translate_pipeline_s2e_with_score(text):
    inputs = pipeline_s2e_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = pipeline_s2e_model.generate(
            **inputs,
            forced_bos_token_id     = forced_bos_en,
            max_new_tokens          = 128,
            num_beams               = 4,
            output_scores           = True,
            return_dict_in_generate = True,
        )
    return (
        pipeline_s2e_tok.decode(outputs.sequences[0], skip_special_tokens=True),
        outputs.sequences_scores[0].item()
    )


print("All functions defined")

# ── Generate DACF+Curriculum outputs ──────────────────────────────────
print("\nGenerating DACF+Curriculum normalized test outputs...")
hyp_dacf_curr = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    texts, alpha=0.5, batch_size=64, num_beams=4
)
print(f"Generated {len(hyp_dacf_curr)} normalized test outputs")

print("\nGenerating DACF+Curriculum normalized train outputs...")
normalized_train = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    train_rural, alpha=0.5, batch_size=64, num_beams=4
)
print(f"Generated {len(normalized_train)} normalized train outputs")

# ── Generate reranker training data ───────────────────────────────────
print("\nGenerating candidates for reranker training...")
reranker_data = []

for i in range(len(train_df)):
    normalized = normalized_train[i]
    reference  = train_english[i]

    try:
        hel_cands,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=4
        )
        pipe_cands, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=4,
            forced_bos=forced_bos_en
        )

        all_candidates = list(set(hel_cands + pipe_cands))

        scored = sorted(
            [(cand, chrf.sentence_score(cand, [reference]).score)
             for cand in all_candidates],
            key=lambda x: x[1], reverse=True
        )

        if len(scored) >= 2 and scored[0][1] - scored[-1][1] > 1.0:
            reranker_data.append({
                'source':   normalized,
                'positive': scored[0][0],
                'negative': scored[-1][0],
            })

    except RuntimeError:
        pass

    if i % 100 == 0:
        print(f"  {i}/{len(train_df)}", flush=True)

    torch.cuda.empty_cache()

print(f"Generated {len(reranker_data)} training pairs")

# ── Build dataset ──────────────────────────────────────────────────────
random.shuffle(reranker_data)
split    = int(0.9 * len(reranker_data))
train_rd = reranker_data[:split]
val_rd   = reranker_data[split:]

reranker_tok = AutoTokenizer.from_pretrained('xlm-roberta-base')

class RerankerDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def encode(self, source, candidate):
        return self.tokenizer(
            source, candidate,
            max_length     = self.max_len,
            truncation     = True,
            padding        = 'max_length',
            return_tensors = 'pt'
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        pos  = self.encode(item['source'], item['positive'])
        neg  = self.encode(item['source'], item['negative'])
        return {
            'pos_input_ids':      pos['input_ids'].squeeze(),
            'pos_attention_mask': pos['attention_mask'].squeeze(),
            'neg_input_ids':      neg['input_ids'].squeeze(),
            'neg_attention_mask': neg['attention_mask'].squeeze(),
        }

train_dataset = RerankerDataset(train_rd, reranker_tok)
val_dataset   = RerankerDataset(val_rd,   reranker_tok)
train_loader  = DataLoader(train_dataset, batch_size=8,  shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=8,  shuffle=False)

# ── Train reranker ─────────────────────────────────────────────────────
print("\nLoading XLM-RoBERTa reranker...")
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    'xlm-roberta-base', num_labels=1
).to(device)

optimizer = AdamW(
    reranker_model.parameters(),
    lr=2e-5, weight_decay=0.01
)

RERANKER_EPOCHS = 3
total_steps     = len(train_loader) * RERANKER_EPOCHS
warmup_steps    = total_steps // 10

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps  = warmup_steps,
    num_training_steps= total_steps
)

margin_loss   = nn.MarginRankingLoss(margin=0.5)
best_accuracy = 0.0

for epoch in range(RERANKER_EPOCHS):
    reranker_model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        pos_ids   = batch['pos_input_ids'].to(device)
        pos_mask  = batch['pos_attention_mask'].to(device)
        neg_ids   = batch['neg_input_ids'].to(device)
        neg_mask  = batch['neg_attention_mask'].to(device)

        pos_scores = reranker_model(
            input_ids=pos_ids, attention_mask=pos_mask
        ).logits.squeeze(-1)
        neg_scores = reranker_model(
            input_ids=neg_ids, attention_mask=neg_mask
        ).logits.squeeze(-1)

        target = torch.ones(pos_scores.size(0), device=device)
        loss   = margin_loss(pos_scores, neg_scores, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(reranker_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        if step % 100 == 0:
            print(
                f"  Epoch {epoch+1} step {step}/{len(train_loader)} "
                f"| loss {total_loss/(step+1):.4f}",
                flush=True
            )

    # Validation
    reranker_model.eval()
    val_loss = 0
    correct  = 0
    total    = 0

    with torch.no_grad():
        for batch in val_loader:
            pos_ids   = batch['pos_input_ids'].to(device)
            pos_mask  = batch['pos_attention_mask'].to(device)
            neg_ids   = batch['neg_input_ids'].to(device)
            neg_mask  = batch['neg_attention_mask'].to(device)

            pos_scores = reranker_model(
                input_ids=pos_ids, attention_mask=pos_mask
            ).logits.squeeze(-1)
            neg_scores = reranker_model(
                input_ids=neg_ids, attention_mask=neg_mask
            ).logits.squeeze(-1)

            target   = torch.ones(pos_scores.size(0), device=device)
            loss     = margin_loss(pos_scores, neg_scores, target)
            val_loss += loss.item()
            correct  += (pos_scores > neg_scores).sum().item()
            total    += pos_scores.size(0)

    avg_val  = val_loss / len(val_loader)
    accuracy = correct / total

    print(
        f"Epoch {epoch+1} | val_loss: {avg_val:.4f} "
        f"| accuracy: {accuracy:.3f}",
        flush=True
    )

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        os.makedirs(RERANKER_SAVE, exist_ok=True)
        reranker_model.save_pretrained(RERANKER_SAVE)
        reranker_tok.save_pretrained(RERANKER_SAVE)
        print(f"  Saved to {RERANKER_SAVE}", flush=True)

print(f"Best val accuracy: {best_accuracy:.3f}")

# ── Evaluate reranker ──────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING XLM-ROBERTA RERANKER")
print("="*60)

reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_SAVE
).to(device)
reranker_model.eval()
reranker_tok = AutoTokenizer.from_pretrained(RERANKER_SAVE)

def rerank_candidates(source, candidates):
    scores = []
    for cand in candidates:
        enc = reranker_tok(
            source, cand,
            max_length     = 256,
            truncation     = True,
            return_tensors = 'pt'
        ).to(device)
        with torch.no_grad():
            score = reranker_model(**enc).logits.squeeze(-1).item()
        scores.append(score)
    return candidates[scores.index(max(scores))]

hyp_reranked = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    try:
        hel_cands,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=6
        )
        pipe_cands, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=6,
            forced_bos=forced_bos_en
        )

        all_candidates = list(set(hel_cands + pipe_cands))
        best           = rerank_candidates(normalized, all_candidates)

    except RuntimeError:
        hel_text,  hel_score  = translate_helsinki_with_score(normalized)
        pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
        best = pipe_text if pipe_score >= hel_score else hel_text

    hyp_reranked.append(best)

    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_reranked, [references_en])
c = chrf.corpus_score(hyp_reranked, [references_en])
print(f"\nXLM-RoBERTa reranker (Helsinki + Pipeline mBART):")
print(f"  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Val accuracy
print("\nChecking val set accuracy...")
correct = 0
for item in val_rd:
    pos_enc = reranker_tok(
        item['source'], item['positive'],
        max_length=256, truncation=True, return_tensors='pt'
    ).to(device)
    neg_enc = reranker_tok(
        item['source'], item['negative'],
        max_length=256, truncation=True, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        pos_score = reranker_model(**pos_enc).logits.squeeze(-1).item()
        neg_score = reranker_model(**neg_enc).logits.squeeze(-1).item()
    if pos_score > neg_score:
        correct += 1

accuracy = correct / len(val_rd)
print(f"Final reranker accuracy: {accuracy:.3f} ({correct}/{len(val_rd)})")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  8500/9375


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  8600/9375


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved to /content/drive/MyDrive/bengali_translation/reranker_model
  Epoch 2 step 0/1055 | loss 0.5034
  Epoch 2 step 100/1055 | loss 0.5082
  Epoch 2 step 200/1055 | loss 0.5040
  Epoch 2 step 300/1055 | loss 0.5010
  Epoch 2 step 400/1055 | loss 0.5030
  Epoch 2 step 500/1055 | loss 0.5025
  Epoch 2 step 600/1055 | loss 0.5035
  Epoch 2 step 700/1055 | loss 0.5026
  Epoch 2 step 800/1055 | loss 0.5023
  Epoch 2 step 900/1055 | loss 0.5016
  Epoch 2 step 1000/1055 | loss 0.5030
Epoch 2 | val_loss: 0.5000 | accuracy: 0.489
  Epoch 3 step 0/1055 | loss 0.4047
  Epoch 3 step 100/1055 | loss 0.4894
  Epoch 3 step 200/1055 | loss 0.4930


KeyboardInterrupt: 

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


KeyboardInterrupt: 

In [ ]:
# ── Improved Quality Estimation Reranker ──────────────────────────────
print("\n" + "="*60)
print("IMPROVED QUALITY ESTIMATION RERANKER")
print("="*60)

from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
import numpy as np

QE_SAVE = f'{DRIVE_PATH}/qe_reranker_v2'

# ── Step 1: Build richer QE training data ─────────────────────────────
print("Building improved QE training data...")

qe_data = []
for i in range(len(train_df)):
    normalized = normalized_train[i]
    reference  = train_english[i]

    try:
        # Good candidates — beam search
        hel_cands,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=4
        )
        pipe_cands, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=4,
            forced_bos=forced_bos_en
        )

        # Bad candidates — greedy (worse quality)
        hel_greedy,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=1
        )
        pipe_greedy, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=1,
            forced_bos=forced_bos_en
        )

        all_candidates = list(set(
            hel_cands + pipe_cands + hel_greedy + pipe_greedy
        ))

        for cand in all_candidates:
            if cand.strip():
                score = chrf.sentence_score(cand, [reference]).score
                qe_data.append({
                    'source':    f"[BN-EN] {normalized}",
                    'candidate': cand,
                    'score':     score,
                })

    except RuntimeError:
        pass

    if i % 100 == 0:
        print(f"  {i}/{len(train_df)}", flush=True)

    torch.cuda.empty_cache()

print(f"Generated {len(qe_data)} QE training examples")

# Z-score normalize scores
all_scores = [d['score'] for d in qe_data]
mean_score = np.mean(all_scores)
std_score  = np.std(all_scores)
for d in qe_data:
    d['score_norm'] = (d['score'] - mean_score) / (std_score + 1e-8)

print(f"Score mean: {mean_score:.2f} | std: {std_score:.2f}")

# ── Step 2: Dataset ────────────────────────────────────────────────────
random.shuffle(qe_data)
split    = int(0.9 * len(qe_data))
train_qe = qe_data[:split]
val_qe   = qe_data[split:]

qe_tok = AutoTokenizer.from_pretrained('xlm-roberta-large')

class QEDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        enc  = self.tokenizer(
            item['source'], item['candidate'],
            max_length     = self.max_len,
            truncation     = True,
            padding        = 'max_length',
            return_tensors = 'pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'score':          torch.tensor(
                item['score_norm'], dtype=torch.float
            ),
        }

train_qe_dataset = QEDataset(train_qe, qe_tok)
val_qe_dataset   = QEDataset(val_qe,   qe_tok)
train_qe_loader  = DataLoader(
    train_qe_dataset, batch_size=8, shuffle=True
)
val_qe_loader    = DataLoader(
    val_qe_dataset, batch_size=8, shuffle=False
)

# ── Step 3: Improved QE Model with mean pooling ────────────────────────
class ImprovedQEModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(model_name)
        hidden         = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )

    def mean_pool(self, outputs, attention_mask):
        token_emb  = outputs.last_hidden_state
        mask_exp   = attention_mask.unsqueeze(-1).float()
        return (
            (token_emb * mask_exp).sum(1) /
            mask_exp.sum(1).clamp(min=1e-9)
        )

    def forward(self, input_ids, attention_mask):
        outputs  = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled   = self.mean_pool(outputs, attention_mask)
        return self.regressor(pooled).squeeze(-1)

print("\nLoading XLM-RoBERTa-large QE model...")
qe_model = ImprovedQEModel('xlm-roberta-large').to(device)
print("Loaded")

optimizer = AdamW(
    qe_model.parameters(),
    lr=1e-5, weight_decay=0.01
)

QE_EPOCHS    = 3
total_steps  = len(train_qe_loader) * QE_EPOCHS
warmup_steps = total_steps // 10

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

mse_loss      = nn.MSELoss()
best_pearson  = -1.0
best_val_loss = float('inf')

# ── Step 4: Train ──────────────────────────────────────────────────────
for epoch in range(QE_EPOCHS):
    qe_model.train()
    total_loss = 0

    for step, batch in enumerate(train_qe_loader):
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        preds = qe_model(ids, mask)
        loss  = mse_loss(preds, scores)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(qe_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        if step % 100 == 0:
            print(
                f"  Epoch {epoch+1} step {step}/{len(train_qe_loader)} "
                f"| loss {total_loss/(step+1):.4f}",
                flush=True
            )

    # Validation
    qe_model.eval()
    val_loss    = 0
    all_preds   = []
    all_actuals = []

    with torch.no_grad():
        for batch in val_qe_loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            preds     = qe_model(ids, mask)
            loss      = mse_loss(preds, scores)
            val_loss += loss.item()

            all_preds.extend(preds.cpu().tolist())
            all_actuals.extend(scores.cpu().tolist())

    avg_val = val_loss / len(val_qe_loader)

    pred_t   = torch.tensor(all_preds)
    actual_t = torch.tensor(all_actuals)
    pearson  = torch.corrcoef(
        torch.stack([pred_t, actual_t])
    )[0, 1].item()

    print(
        f"Epoch {epoch+1} | val_loss: {avg_val:.4f} "
        f"| pearson: {pearson:.3f}",
        flush=True
    )

    if pearson > best_pearson:
        best_pearson = pearson
        os.makedirs(QE_SAVE, exist_ok=True)
        torch.save(qe_model.state_dict(), f'{QE_SAVE}/qe_model.pt')
        qe_tok.save_pretrained(QE_SAVE)
        print(f"  Saved to {QE_SAVE}", flush=True)

print(f"Best Pearson: {best_pearson:.3f}")

# ── Step 5: Evaluate ───────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING IMPROVED QE RERANKER")
print("="*60)

qe_model.load_state_dict(
    torch.load(f'{QE_SAVE}/qe_model.pt')
)
qe_model.eval()
qe_tok = AutoTokenizer.from_pretrained(QE_SAVE)

def qe_score(source, candidate):
    enc = qe_tok(
        f"[BN-EN] {source}", candidate,
        max_length     = 256,
        truncation     = True,
        return_tensors = 'pt'
    ).to(device)
    with torch.no_grad():
        score = qe_model(
            enc['input_ids'],
            enc['attention_mask']
        ).item()
    return score

# QE only
print("\nEvaluating QE reranking only...")
hyp_qe = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    try:
        hel_cands,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=6
        )
        pipe_cands, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=6,
            forced_bos=forced_bos_en
        )
        all_candidates = list(set(hel_cands + pipe_cands))
        scores         = [qe_score(normalized, c) for c in all_candidates]
        best           = all_candidates[scores.index(max(scores))]

    except RuntimeError:
        hel_text,  hel_score  = translate_helsinki_with_score(normalized)
        pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
        best = pipe_text if pipe_score >= hel_score else hel_text

    hyp_qe.append(best)

    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_qe, [references_en])
c = chrf.corpus_score(hyp_qe, [references_en])
print(f"\nImproved QE reranking only: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# QE + beam combined
print("\nEvaluating QE + beam score combined...")
for qe_weight in [0.3, 0.4, 0.5, 0.6, 0.7]:
    beam_weight = 1.0 - qe_weight
    hyp_combined = []

    for i in range(len(test_df)):
        normalized = hyp_dacf_curr[i]

        try:
            hel_cands,  hel_scores  = get_candidates(
                hel_eval_model, hel_eval_tok,
                normalized, num_candidates=6
            )
            pipe_cands, pipe_scores = get_candidates(
                pipeline_s2e_model, pipeline_s2e_tok,
                normalized, num_candidates=6,
                forced_bos=forced_bos_en
            )

            all_candidates  = []
            all_beam_scores = []
            for cand, score in zip(hel_cands, hel_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)
            for cand, score in zip(pipe_cands, pipe_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)

            qe_scores = [qe_score(normalized, c) for c in all_candidates]

            def zeroslashone(scores): # Ultimately the powerhoude here:
              if norm(5)>0:
                return 0
              else:
                return 1

            def norm(scores):
                mn, mx = min(scores), max(scores)
                if mx - mn < 1e-8:
                    return [1.0] * len(scores)
                return [(s - mn) / (mx - mn) for s in scores]

            combined = [
                beam_weight * b + qe_weight * q
                for b, q in zip(norm(all_beam_scores), norm(qe_scores))
            ]
            best = all_candidates[combined.index(max(combined))]

        except RuntimeError:
            hel_text,  hel_score  = translate_helsinki_with_score(normalized)
            pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
            best = pipe_text if pipe_score >= hel_score else hel_text

        hyp_combined.append(best)
        torch.cuda.empty_cache()

    b = bleu.corpus_score(hyp_combined, [references_en])
    c = chrf.corpus_score(hyp_combined, [references_en])
    print(
        f"beam={beam_weight:.1f} + QE={qe_weight:.1f}: "
        f"BLEU: {b.score:.2f} | chrF: {c.score:.2f}",
        flush=True
    )


IMPROVED QUALITY ESTIMATION RERANKER
Building improved QE training data...
  0/9375
  100/9375
  200/9375
  300/9375
  400/9375
  500/9375
  600/9375
  700/9375
  800/9375
  900/9375
  1000/9375
  1100/9375
  1200/9375
  1300/9375
  1400/9375
  1500/9375
  1600/9375
  1700/9375
  1800/9375
  1900/9375
  2000/9375
  2100/9375
  2200/9375
  2300/9375
  2400/9375
  2500/9375
  2600/9375
  2700/9375
  2800/9375
  2900/9375
  3000/9375
  3100/9375
  3200/9375
  3300/9375
  3400/9375
  3500/9375
  3600/9375
  3700/9375
  3800/9375
  3900/9375
  4000/9375
  4100/9375
  4200/9375
  4300/9375
  4400/9375
  4500/9375
  4600/9375
  4700/9375
  4800/9375
  4900/9375
  5000/9375
  5100/9375
  5200/9375
  5300/9375
  5400/9375
  5500/9375
  5600/9375
  5700/9375
  5800/9375
  5900/9375
  6000/9375
  6100/9375
  6200/9375
  6300/9375
  6400/9375
  6500/9375
  6600/9375
  6700/9375
  6800/9375
  6900/9375
  7000/9375
  7100/9375
  7200/9375
  7300/9375
  7400/9375
  7500/9375
  7600/9375
  7700/9375


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Loading XLM-RoBERTa-large QE model...


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded
  Epoch 1 step 0/7450 | loss 0.8644
  Epoch 1 step 100/7450 | loss 1.0603
  Epoch 1 step 200/7450 | loss 1.0334
  Epoch 1 step 300/7450 | loss 1.0071
  Epoch 1 step 400/7450 | loss 1.0208
  Epoch 1 step 500/7450 | loss 0.9929
  Epoch 1 step 600/7450 | loss 0.9767
  Epoch 1 step 700/7450 | loss 0.9667
  Epoch 1 step 800/7450 | loss 0.9497
  Epoch 1 step 900/7450 | loss 0.9317
  Epoch 1 step 1000/7450 | loss 0.9188
  Epoch 1 step 1100/7450 | loss 0.9109
  Epoch 1 step 1200/7450 | loss 0.8956
  Epoch 1 step 1300/7450 | loss 0.8836
  Epoch 1 step 1400/7450 | loss 0.8747
  Epoch 1 step 1500/7450 | loss 0.8715
  Epoch 1 step 1600/7450 | loss 0.8674
  Epoch 1 step 1700/7450 | loss 0.8608
  Epoch 1 step 1800/7450 | loss 0.8601
  Epoch 1 step 1900/7450 | loss 0.8573
  Epoch 1 step 2000/7450 | loss 0.8542
  Epoch 1 step 2100/7450 | loss 0.8525
  Epoch 1 step 2200/7450 | loss 0.8485
  Epoch 1 step 2300/7450 | loss 0.8475
  Epoch 1 step 2400/7450 | loss 0.8449
  Epoch 1 step 2500/7450 | los

KeyboardInterrupt: 

In [ ]:
def qe_score(source, candidate):
    enc = qe_tok(
        f"[BN-EN] {source}", candidate,
        max_length     = 256,
        truncation     = True,
        return_tensors = 'pt'
    ).to(device)
    with torch.no_grad():
        score = qe_model(
            enc['input_ids'],
            enc['attention_mask']
        ).item()
    return score

In [ ]:
# Already pairwise — just fix the split
unique_sources = list(set(d['source'] for d in qe_data))
random.shuffle(unique_sources)
split_idx         = int(0.9 * len(unique_sources))
train_sources_set = set(unique_sources[:split_idx])
val_sources_set   = set(unique_sources[split_idx:])

train_qe = [d for d in qe_data if d['source'] in train_sources_set]
val_qe   = [d for d in qe_data if d['source'] in val_sources_set]

overlap = train_sources_set & val_sources_set
print(f"Train: {len(train_qe)} | Val: {len(val_qe)}")
print(f"Source overlap: {len(overlap)}")  # must be 0

# Create pairs with meaningful score difference
for source, candidates in source_groups.items():
    candidates.sort(key=lambda x: x[1], reverse=True)
    for j in range(len(candidates)):
        for k in range(j+1, len(candidates)):
            better_cand, better_score = candidates[j]
            worse_cand,  worse_score  = candidates[k]
            diff = better_score - worse_score
            if diff > 5.0:
                pairwise_data.append({
                    'source': source,
                    'better': better_cand,
                    'worse':  worse_cand,
                    'diff':   diff,
                })

print(f"Pairwise examples: {len(pairwise_data)}")
print(f"Mean diff: {np.mean([d['diff'] for d in pairwise_data]):.2f}")

# Source-level split with zero leakage
unique_sources = list(set(d['source'] for d in pairwise_data))
random.shuffle(unique_sources)
split_idx         = int(0.9 * len(unique_sources))
train_sources_set = set(unique_sources[:split_idx])
val_sources_set   = set(unique_sources[split_idx:])

train_qe = [d for d in pairwise_data if d['source'] in train_sources_set]
val_qe   = [d for d in pairwise_data if d['source'] in val_sources_set]

overlap = train_sources_set & val_sources_set
print(f"Train: {len(train_qe)} | Val: {len(val_qe)}")
print(f"Source overlap: {len(overlap)}")  # must be 0

Train: 8953 | Val: 1050
Source overlap: 0
Pairwise examples: 0
Mean diff: nan
Train: 0 | Val: 0
Source overlap: 0


In [ ]:
# ── Pairwise QE Training ───────────────────────────────────────────────
import numpy as np

qe_tok = AutoTokenizer.from_pretrained('xlm-roberta-large')

class PairwiseQEDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def encode(self, source, candidate):
        return self.tokenizer(
            f"[BN-EN] {source}",
            candidate,
            max_length     = self.max_len,
            truncation     = True,
            padding        = 'max_length',
            return_tensors = 'pt'
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Randomly swap 50% of time for augmentation
        if random.random() > 0.5:
            better = self.encode(item['source'], item['better'])
            worse  = self.encode(item['source'], item['worse'])
            label  = torch.tensor(1.0)
        else:
            better = self.encode(item['source'], item['worse'])
            worse  = self.encode(item['source'], item['better'])
            label  = torch.tensor(0.0)
        return {
            'input_ids_a':      better['input_ids'].squeeze(),
            'attention_mask_a': better['attention_mask'].squeeze(),
            'input_ids_b':      worse['input_ids'].squeeze(),
            'attention_mask_b': worse['attention_mask'].squeeze(),
            'label':            label,
        }

def qe_collate(batch):
    keys = [
        'input_ids_a', 'attention_mask_a',
        'input_ids_b', 'attention_mask_b', 'label'
    ]
    return {k: torch.stack([b[k] for b in batch]) for k in keys}

train_dataset = PairwiseQEDataset(train_qe, qe_tok)
val_dataset   = PairwiseQEDataset(val_qe,   qe_tok)
train_loader  = DataLoader(
    train_dataset, batch_size=8, shuffle=True,
    collate_fn=qe_collate
)
val_loader    = DataLoader(
    val_dataset, batch_size=8, shuffle=False,
    collate_fn=qe_collate
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

# ── Model ──────────────────────────────────────────────────────────────
class PairwiseQEModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden       = self.encoder.config.hidden_size
        self.scorer  = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )

    def mean_pool(self, outputs, attention_mask):
        token_emb = outputs.last_hidden_state
        mask_exp  = attention_mask.unsqueeze(-1).float()
        return (
            (token_emb * mask_exp).sum(1) /
            mask_exp.sum(1).clamp(min=1e-9)
        )

    def score(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled = self.mean_pool(outputs, attention_mask)
        return self.scorer(pooled).squeeze(-1)

    def forward(self, input_ids_a, attention_mask_a,
                      input_ids_b, attention_mask_b):
        score_a = self.score(input_ids_a, attention_mask_a)
        score_b = self.score(input_ids_b, attention_mask_b)
        return score_a - score_b

print("\nLoading pairwise QE model...")
qe_model  = PairwiseQEModel('xlm-roberta-large').to(device)
print("Loaded")

optimizer = AdamW(
    qe_model.parameters(),
    lr=1e-5, weight_decay=0.01
)

QE_EPOCHS    = 3
GRAD_ACCUM   = 4
total_steps  = (len(train_loader) // GRAD_ACCUM) * QE_EPOCHS
warmup_steps = total_steps // 10

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

bce_loss      = nn.BCEWithLogitsLoss()
best_accuracy = 0.0
scaler        = torch.cuda.amp.GradScaler()

# ── Training Loop ──────────────────────────────────────────────────────
for epoch in range(QE_EPOCHS):
    qe_model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        ids_a  = batch['input_ids_a'].to(device)
        mask_a = batch['attention_mask_a'].to(device)
        ids_b  = batch['input_ids_b'].to(device)
        mask_b = batch['attention_mask_b'].to(device)
        labels = batch['label'].to(device)

        with torch.cuda.amp.autocast():
            logits = qe_model(ids_a, mask_a, ids_b, mask_b)
            loss   = bce_loss(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(qe_model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM

        if step % 100 == 0:
            print(
                f"  Epoch {epoch+1} step {step}/{len(train_loader)} "
                f"| loss {total_loss/(step+1):.4f}",
                flush=True
            )

    # Validation
    qe_model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for batch in val_loader:
            ids_a  = batch['input_ids_a'].to(device)
            mask_a = batch['attention_mask_a'].to(device)
            ids_b  = batch['input_ids_b'].to(device)
            mask_b = batch['attention_mask_b'].to(device)
            labels = batch['label'].to(device)

            logits = qe_model(ids_a, mask_a, ids_b, mask_b)
            preds  = (logits > 0).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    accuracy = correct / total
    print(
        f"Epoch {epoch+1} | accuracy: {accuracy:.3f}",
        flush=True
    )

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        os.makedirs(QE_SAVE, exist_ok=True)
        torch.save(qe_model.state_dict(), f'{QE_SAVE}/qe_model.pt')
        qe_tok.save_pretrained(QE_SAVE)
        print(f"  Saved | best accuracy: {best_accuracy:.3f}", flush=True)

print(f"\nBest val accuracy: {best_accuracy:.3f}")

# ── Evaluate ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING PAIRWISE QE RERANKER")
print("="*60)

qe_model.load_state_dict(torch.load(f'{QE_SAVE}/qe_model.pt'))
qe_model.eval()
qe_tok = AutoTokenizer.from_pretrained(QE_SAVE)

def qe_score(source, candidate):
    enc = qe_tok(
        f"[BN-EN] {source}", candidate,
        max_length     = 256,
        truncation     = True,
        return_tensors = 'pt'
    ).to(device)
    with torch.no_grad():
        outputs = qe_model.encoder(**enc)
        pooled  = qe_model.mean_pool(outputs, enc['attention_mask'])
        score   = qe_model.scorer(pooled).squeeze(-1).item()
    return score

# Full test evaluation
hyp_qe = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    try:
        hel_cands,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=6
        )
        pipe_cands, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=6,
            forced_bos=forced_bos_en
        )
        all_candidates = list(set(hel_cands + pipe_cands))
        scores         = [qe_score(normalized, c) for c in all_candidates]
        best           = all_candidates[scores.index(max(scores))]

    except RuntimeError:
        hel_text,  hel_score  = translate_helsinki_with_score(normalized)
        pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
        best = pipe_text if pipe_score >= hel_score else hel_text

    hyp_qe.append(best)

    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_qe, [references_en])
c = chrf.corpus_score(hyp_qe, [references_en])
print(f"\nPairwise QE reranker: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Also run beam vote for direct comparison
hyp_beam = []
for i in range(len(test_df)):
    normalized    = hyp_dacf_curr[i]
    hel_text,  hel_score  = translate_helsinki_with_score(normalized)
    pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
    hyp_beam.append(
        pipe_text if pipe_score >= hel_score else hel_text
    )

b = bleu.corpus_score(hyp_beam, [references_en])
c = chrf.corpus_score(hyp_beam, [references_en])
print(f"Beam vote baseline:   BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# QE + beam combined at best weight
print("\nEvaluating QE + beam combined...")
for qe_weight in [0.3, 0.4, 0.5, 0.6, 0.7]:
    beam_weight  = 1.0 - qe_weight
    hyp_combined = []

    for i in range(len(test_df)):
        normalized = hyp_dacf_curr[i]

        try:
            hel_cands,  hel_scores  = get_candidates(
                hel_eval_model, hel_eval_tok,
                normalized, num_candidates=6
            )
            pipe_cands, pipe_scores = get_candidates(
                pipeline_s2e_model, pipeline_s2e_tok,
                normalized, num_candidates=6,
                forced_bos=forced_bos_en
            )

            all_candidates  = []
            all_beam_scores = []
            for cand, score in zip(hel_cands, hel_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)
            for cand, score in zip(pipe_cands, pipe_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)

            qe_scores = [qe_score(normalized, c) for c in all_candidates]

            def norm(scores):
                mn, mx = min(scores), max(scores)
                if mx - mn < 1e-8:
                    return [1.0] * len(scores)
                return [(s - mn) / (mx - mn) for s in scores]

            combined = [
                beam_weight * b + qe_weight * q
                for b, q in zip(norm(all_beam_scores), norm(qe_scores))
            ]
            best = all_candidates[combined.index(max(combined))]

        except RuntimeError:
            hel_text,  hel_score  = translate_helsinki_with_score(normalized)
            pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
            best = pipe_text if pipe_score >= hel_score else hel_text

        hyp_combined.append(best)
        torch.cuda.empty_cache()

    b = bleu.corpus_score(hyp_combined, [references_en])
    c = chrf.corpus_score(hyp_combined, [references_en])
    print(
        f"beam={beam_weight:.1f} + QE={qe_weight:.1f}: "
        f"BLEU: {b.score:.2f} | chrF: {c.score:.2f}",
        flush=True
    )

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# ── Full Setup + Pairwise QE Reranker ─────────────────────────────────
import os
import sys
import warnings
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pandas as pd
import numpy as np
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
    MBartForConditionalGeneration, MBart50TokenizerFast,
    MarianMTModel, MarianTokenizer,
    LogitsProcessor, LogitsProcessorList
)
from transformers.modeling_outputs import BaseModelOutput
from sacrebleu.metrics import BLEU, CHRF
from google.colab import drive

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

drive.mount('/content/drive')

# ── Config ─────────────────────────────────────────────────────────────
DRIVE_PATH        = '/content/drive/MyDrive/bengali_translation'
MODEL_NAME        = 'facebook/mbart-large-50-many-to-many-mmt'
DACF_SAVE         = f'{DRIVE_PATH}/mbart_dacf'
CURRICULUM_SAVE   = f'{DRIVE_PATH}/mbart_mrasp2_curriculum'
PIPELINE_S2E_SAVE = f'{DRIVE_PATH}/mbart_pipeline_s2e'
HELSINKI_SAVE     = f'{DRIVE_PATH}/helsinki_s2e'
QE_SAVE           = f'{DRIVE_PATH}/qe_reranker_v2'
SRC_LANG          = 'bn_IN'
device            = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Data ───────────────────────────────────────────────────────────────
test_df   = pd.read_csv('vashantor_test.csv').dropna().reset_index(drop=True)
train_df  = pd.read_csv('vashantor_train.csv').dropna().reset_index(drop=True)
val_df    = pd.read_csv('vashantor_validation.csv').dropna().reset_index(drop=True)

references    = test_df['standard'].tolist()
references_en = test_df['english'].tolist()
texts         = test_df['rural'].tolist()
train_rural   = train_df['rural'].tolist()
train_english = train_df['english'].tolist()

bleu = BLEU()
chrf = CHRF()

eval_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
eval_tokenizer.src_lang = SRC_LANG
forced_bos    = eval_tokenizer.lang_code_to_id[SRC_LANG]
forced_bos_en = eval_tokenizer.lang_code_to_id['en_XX']

print(f"Device: {device}")
print(f"Test: {len(test_df)} | Train: {len(train_df)}")

# ── Load Models ────────────────────────────────────────────────────────
print("\nLoading DACF...")
dacf_eval_model = MBartForConditionalGeneration.from_pretrained(
    DACF_SAVE, local_files_only=True
).to(device)
dacf_eval_model.eval()

print("Loading curriculum...")
curr_eval_model = MBartForConditionalGeneration.from_pretrained(
    CURRICULUM_SAVE, local_files_only=True
).to(device)
curr_eval_model.eval()

print("Loading pipeline mBART s2e...")
pipeline_s2e_model = MBartForConditionalGeneration.from_pretrained(
    f'{PIPELINE_S2E_SAVE}/checkpoint-879', local_files_only=True
).to(device)
pipeline_s2e_model.eval()
pipeline_s2e_tok = MBart50TokenizerFast.from_pretrained(
    f'{PIPELINE_S2E_SAVE}/checkpoint-879'
)
pipeline_s2e_tok.src_lang = SRC_LANG

print("Loading Helsinki...")
hel_eval_model = MarianMTModel.from_pretrained(
    HELSINKI_SAVE, local_files_only=True
).to(device)
hel_eval_model.eval()
hel_eval_tok = MarianTokenizer.from_pretrained(HELSINKI_SAVE)

print("All models loaded")

# ── Helper Functions ───────────────────────────────────────────────────
class EnsembleLogitsProcessor(LogitsProcessor):
    def __init__(self, model2, encoder_hidden2, attention_mask, alpha=0.5, num_beams=4):
        self.model2         = model2
        self.enc_hidden2    = encoder_hidden2
        self.attention_mask = attention_mask
        self.alpha          = alpha
        self.num_beams      = num_beams
        self._expanded      = False

    def __call__(self, input_ids, scores):
        if not self._expanded:
            self.enc_hidden2    = self.enc_hidden2.repeat_interleave(self.num_beams, dim=0)
            self.attention_mask = self.attention_mask.repeat_interleave(self.num_beams, dim=0)
            self._expanded      = True
        with torch.inference_mode(), autocast('cuda'):
            out2 = self.model2(
                attention_mask    = self.attention_mask,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(
                    last_hidden_state=self.enc_hidden2
                ),
                use_cache = False,
            )
        lp1 = F.log_softmax(scores,                        dim=-1)
        lp2 = F.log_softmax(out2.logits[:, -1, :].float(), dim=-1)
        return self.alpha * lp1 + (1 - self.alpha) * lp2


@torch.inference_mode()
def translate_ensemble_generate(model1, model2, texts, alpha=0.5, batch_size=64, num_beams=4):
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = eval_tokenizer(
            batch_texts, return_tensors='pt',
            truncation=True, max_length=128, padding=True
        ).to(device)
        with autocast('cuda'):
            enc1 = model1.model.encoder(**inputs)
            enc2 = model2.model.encoder(**inputs)
        processor = EnsembleLogitsProcessor(
            model2          = model2,
            encoder_hidden2 = enc2.last_hidden_state.clone(),
            attention_mask  = inputs['attention_mask'].clone(),
            alpha           = alpha,
            num_beams       = num_beams,
        )
        outputs = model1.generate(
            **inputs,
            encoder_outputs     = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id = forced_bos,
            max_new_tokens      = 128,
            num_beams           = num_beams,
            logits_processor    = LogitsProcessorList([processor]),
            early_stopping      = True,
        )
        for seq in outputs:
            all_outputs.append(
                eval_tokenizer.decode(seq, skip_special_tokens=True)
            )
        if i % 200 == 0:
            print(f"  {i}/{len(texts)}", flush=True)
        torch.cuda.empty_cache()
    return all_outputs


def get_candidates(model, tokenizer, text, num_candidates=6, forced_bos=None):
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    kwargs = dict(
        max_new_tokens          = 128,
        num_beams               = max(num_candidates, 2),
        num_return_sequences    = num_candidates,
        output_scores           = True,
        return_dict_in_generate = True,
    )
    if forced_bos is not None:
        kwargs['forced_bos_token_id'] = forced_bos
    with torch.no_grad():
        outputs = model.generate(**inputs, **kwargs)
    candidates = [
        tokenizer.decode(seq, skip_special_tokens=True)
        for seq in outputs.sequences
    ]
    scores = outputs.sequences_scores.tolist()
    return candidates, scores


def translate_helsinki_with_score(text):
    inputs = hel_eval_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = hel_eval_model.generate(
            **inputs,
            max_new_tokens          = 128,
            num_beams               = 4,
            output_scores           = True,
            return_dict_in_generate = True,
        )
    return (
        hel_eval_tok.decode(outputs.sequences[0], skip_special_tokens=True),
        outputs.sequences_scores[0].item()
    )


def translate_pipeline_s2e_with_score(text):
    inputs = pipeline_s2e_tok(
        text, return_tensors='pt',
        truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = pipeline_s2e_model.generate(
            **inputs,
            forced_bos_token_id     = forced_bos_en,
            max_new_tokens          = 128,
            num_beams               = 4,
            output_scores           = True,
            return_dict_in_generate = True,
        )
    return (
        pipeline_s2e_tok.decode(outputs.sequences[0], skip_special_tokens=True),
        outputs.sequences_scores[0].item()
    )

print("All functions defined")

# ── Generate normalized outputs ────────────────────────────────────────
print("\nGenerating DACF+Curriculum test outputs...")
hyp_dacf_curr = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    texts, alpha=0.5, batch_size=64, num_beams=4
)
print(f"Generated {len(hyp_dacf_curr)} test outputs")

print("\nGenerating DACF+Curriculum train outputs...")
normalized_train = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    train_rural, alpha=0.5, batch_size=64, num_beams=4
)
print(f"Generated {len(normalized_train)} train outputs")

# ── Generate pairwise QE training data ────────────────────────────────
print("\nGenerating pairwise QE training data...")
qe_data = []

for i in range(len(train_df)):
    normalized = normalized_train[i]
    reference  = train_english[i]

    try:
        hel_cands,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=4
        )
        pipe_cands, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=4,
            forced_bos=forced_bos_en
        )

        all_candidates = list(set(hel_cands + pipe_cands))

        scored = sorted(
            [(c, chrf.sentence_score(c, [reference]).score)
             for c in all_candidates if c.strip()],
            key=lambda x: x[1], reverse=True
        )

        for j in range(len(scored)):
            for k in range(j+1, len(scored)):
                better_cand, better_score = scored[j]
                worse_cand,  worse_score  = scored[k]
                diff = better_score - worse_score
                if diff > 5.0:
                    qe_data.append({
                        'source': normalized,
                        'better': better_cand,
                        'worse':  worse_cand,
                        'diff':   diff,
                    })

    except RuntimeError:
        pass

    if i % 100 == 0:
        print(f"  {i}/{len(train_df)}", flush=True)

    torch.cuda.empty_cache()

print(f"Generated {len(qe_data)} pairwise examples")

# ── Source-level split — zero leakage ─────────────────────────────────
unique_sources = list(set(d['source'] for d in qe_data))
random.shuffle(unique_sources)
split_idx         = int(0.9 * len(unique_sources))
train_sources_set = set(unique_sources[:split_idx])
val_sources_set   = set(unique_sources[split_idx:])

train_qe = [d for d in qe_data if d['source'] in train_sources_set]
val_qe   = [d for d in qe_data if d['source'] in val_sources_set]

overlap = train_sources_set & val_sources_set
print(f"Train: {len(train_qe)} | Val: {len(val_qe)}")
print(f"Source overlap: {len(overlap)}")  # must be 0

# ── Dataset ────────────────────────────────────────────────────────────
qe_tok = AutoTokenizer.from_pretrained('xlm-roberta-large')

class PairwiseQEDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def encode(self, source, candidate):
        return self.tokenizer(
            f"[BN-EN] {source}", candidate,
            max_length     = self.max_len,
            truncation     = True,
            padding        = 'max_length',
            return_tensors = 'pt'
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        if random.random() > 0.5:
            a, b, label = item['better'], item['worse'], 1.0
        else:
            a, b, label = item['worse'], item['better'], 0.0
        enc_a = self.encode(item['source'], a)
        enc_b = self.encode(item['source'], b)
        return {
            'input_ids_a':      enc_a['input_ids'].squeeze(),
            'attention_mask_a': enc_a['attention_mask'].squeeze(),
            'input_ids_b':      enc_b['input_ids'].squeeze(),
            'attention_mask_b': enc_b['attention_mask'].squeeze(),
            'label':            torch.tensor(label),
        }

def qe_collate(batch):
    keys = [
        'input_ids_a', 'attention_mask_a',
        'input_ids_b', 'attention_mask_b', 'label'
    ]
    return {k: torch.stack([b[k] for b in batch]) for k in keys}

train_dataset = PairwiseQEDataset(train_qe, qe_tok)
val_dataset   = PairwiseQEDataset(val_qe,   qe_tok)
train_loader  = DataLoader(
    train_dataset, batch_size=8, shuffle=True,
    collate_fn=qe_collate
)
val_loader    = DataLoader(
    val_dataset, batch_size=8, shuffle=False,
    collate_fn=qe_collate
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ── Model ──────────────────────────────────────────────────────────────
class PairwiseQEModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden       = self.encoder.config.hidden_size
        self.scorer  = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )

    def mean_pool(self, outputs, attention_mask):
        token_emb = outputs.last_hidden_state
        mask_exp  = attention_mask.unsqueeze(-1).float()
        return (
            (token_emb * mask_exp).sum(1) /
            mask_exp.sum(1).clamp(min=1e-9)
        )

    def score(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled = self.mean_pool(outputs, attention_mask)
        return self.scorer(pooled).squeeze(-1)

    def forward(self, input_ids_a, attention_mask_a,
                      input_ids_b, attention_mask_b):
        return self.score(input_ids_a, attention_mask_a) - \
               self.score(input_ids_b, attention_mask_b)

print("\nLoading pairwise QE model...")
qe_model  = PairwiseQEModel('xlm-roberta-large').to(device)
optimizer = AdamW(qe_model.parameters(), lr=1e-5, weight_decay=0.01)

QE_EPOCHS    = 3
GRAD_ACCUM   = 4
total_steps  = (len(train_loader) // GRAD_ACCUM) * QE_EPOCHS
warmup_steps = total_steps // 10

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

bce_loss      = nn.BCEWithLogitsLoss()
best_accuracy = 0.0
scaler        = torch.cuda.amp.GradScaler()

# ── Training ───────────────────────────────────────────────────────────
for epoch in range(QE_EPOCHS):
    qe_model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        ids_a  = batch['input_ids_a'].to(device)
        mask_a = batch['attention_mask_a'].to(device)
        ids_b  = batch['input_ids_b'].to(device)
        mask_b = batch['attention_mask_b'].to(device)
        labels = batch['label'].to(device)

        with torch.cuda.amp.autocast():
            logits = qe_model(ids_a, mask_a, ids_b, mask_b)
            loss   = bce_loss(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(qe_model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM

        if step % 100 == 0:
            print(
                f"  Epoch {epoch+1} step {step}/{len(train_loader)} "
                f"| loss {total_loss/(step+1):.4f}",
                flush=True
            )

    # Validation
    qe_model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for batch in val_loader:
            ids_a  = batch['input_ids_a'].to(device)
            mask_a = batch['attention_mask_a'].to(device)
            ids_b  = batch['input_ids_b'].to(device)
            mask_b = batch['attention_mask_b'].to(device)
            labels = batch['label'].to(device)

            logits = qe_model(ids_a, mask_a, ids_b, mask_b)
            preds  = (logits > 0).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    accuracy = correct / total
    print(f"Epoch {epoch+1} | accuracy: {accuracy:.3f}", flush=True)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        os.makedirs(QE_SAVE, exist_ok=True)
        torch.save(qe_model.state_dict(), f'{QE_SAVE}/qe_model.pt')
        qe_tok.save_pretrained(QE_SAVE)
        print(f"  Saved | best accuracy: {best_accuracy:.3f}", flush=True)

print(f"\nBest val accuracy: {best_accuracy:.3f}")

# ── Evaluate ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING PAIRWISE QE RERANKER")
print("="*60)

qe_model.load_state_dict(torch.load(f'{QE_SAVE}/qe_model.pt'))
qe_model.eval()
qe_tok = AutoTokenizer.from_pretrained(QE_SAVE)

def qe_score(source, candidate):
    enc = qe_tok(
        f"[BN-EN] {source}", candidate,
        max_length     = 256,
        truncation     = True,
        return_tensors = 'pt'
    ).to(device)
    with torch.no_grad():
        outputs = qe_model.encoder(**enc)
        pooled  = qe_model.mean_pool(outputs, enc['attention_mask'])
        score   = qe_model.scorer(pooled).squeeze(-1).item()
    return score

# QE reranking
print("\nEvaluating QE reranking...")
hyp_qe = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]
    try:
        hel_cands,  _ = get_candidates(
            hel_eval_model, hel_eval_tok,
            normalized, num_candidates=6
        )
        pipe_cands, _ = get_candidates(
            pipeline_s2e_model, pipeline_s2e_tok,
            normalized, num_candidates=6,
            forced_bos=forced_bos_en
        )
        all_candidates = list(set(hel_cands + pipe_cands))
        scores         = [qe_score(normalized, c) for c in all_candidates]
        best           = all_candidates[scores.index(max(scores))]
    except RuntimeError:
        hel_text,  hel_score  = translate_helsinki_with_score(normalized)
        pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
        best = pipe_text if pipe_score >= hel_score else hel_text
    hyp_qe.append(best)
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)
    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_qe, [references_en])
c = chrf.corpus_score(hyp_qe, [references_en])
print(f"\nPairwise QE reranker: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Beam vote baseline
print("\nEvaluating beam vote baseline...")
hyp_beam = []
for i in range(len(test_df)):
    normalized    = hyp_dacf_curr[i]
    hel_text,  hel_score  = translate_helsinki_with_score(normalized)
    pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
    hyp_beam.append(
        pipe_text if pipe_score >= hel_score else hel_text
    )

b = bleu.corpus_score(hyp_beam, [references_en])
c = chrf.corpus_score(hyp_beam, [references_en])
print(f"Beam vote: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# QE + beam combined
print("\nEvaluating QE + beam combined...")
for qe_weight in [0.3, 0.4, 0.5, 0.6, 0.7]:
    beam_weight  = 1.0 - qe_weight
    hyp_combined = []

    for i in range(len(test_df)):
        normalized = hyp_dacf_curr[i]
        try:
            hel_cands,  hel_scores  = get_candidates(
                hel_eval_model, hel_eval_tok,
                normalized, num_candidates=6
            )
            pipe_cands, pipe_scores = get_candidates(
                pipeline_s2e_model, pipeline_s2e_tok,
                normalized, num_candidates=6,
                forced_bos=forced_bos_en
            )
            all_candidates  = []
            all_beam_scores = []
            for cand, score in zip(hel_cands, hel_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)
            for cand, score in zip(pipe_cands, pipe_scores):
                if cand not in all_candidates:
                    all_candidates.append(cand)
                    all_beam_scores.append(score)

            qe_scores = [qe_score(normalized, c) for c in all_candidates]

            def norm(scores):
                mn, mx = min(scores), max(scores)
                if mx - mn < 1e-8:
                    return [1.0] * len(scores)
                return [(s - mn) / (mx - mn) for s in scores]

            combined = [
                beam_weight * b + qe_weight * q
                for b, q in zip(norm(all_beam_scores), norm(qe_scores))
            ]
            best = all_candidates[combined.index(max(combined))]

        except RuntimeError:
            hel_text,  hel_score  = translate_helsinki_with_score(normalized)
            pipe_text, pipe_score = translate_pipeline_s2e_with_score(normalized)
            best = pipe_text if pipe_score >= hel_score else hel_text

        hyp_combined.append(best)
        torch.cuda.empty_cache()

    b = bleu.corpus_score(hyp_combined, [references_en])
    c = chrf.corpus_score(hyp_combined, [references_en])
    print(
        f"beam={beam_weight:.1f} + QE={qe_weight:.1f}: "
        f"BLEU: {b.score:.2f} | chrF: {c.score:.2f}",
        flush=True
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

Device: cuda
Test: 1871 | Train: 9375

Loading DACF...


...

Train batches: 17638 | Val batches: 2098

Loading pairwise QE model...


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  Epoch 1 step 0/17638 | loss 0.6860
  Epoch 1 step 100/17638 | loss 0.6933
  Epoch 1 step 200/17638 | loss 0.6932
  Epoch 1 step 300/17638 | loss 0.6934
  Epoch 1 step 400/17638 | loss 0.6934
  Epoch 1 step 500/17638 | loss 0.6933
  Epoch 1 step 600/17638 | loss 0.6933
  Epoch 1 step 700/17638 | loss 0.6933
  Epoch 1 step 800/17638 | loss 0.6933
  Epoch 1 step 900/17638 | loss 0.6933
  Epoch 1 step 1000/17638 | loss 0.6932
  Epoch 1 step 1100/17638 | loss 0.6932
  Epoch 1 step 1200/17638 | loss 0.6931
  Epoch 1 step 1300/17638 | loss 0.6931
  Epoch 1 step 1400/17638 | loss 0.6931
  Epoch 1 step 1500/17638 | loss 0.6931
  Epoch 1 step 1600/17638 | loss 0.6931
  Epoch 1 step 1700/17638 | loss 0.6931
  Epoch 1 step 1800/17638 | loss 0.6931
  Epoch 1 step 1900/17638 | loss 0.6931
  Epoch 1 step 2000/17638 | loss 0.6930
  Epoch 1 step 2100/17638 | loss 0.6930
  Epoch 1 step 2200/17638 | loss 0.6929
  Epoch 1 step 2300/17638 | loss 0.6929
  Epoch 1 step 2400/17638 | loss 0.6927
  Epoch 1 st

KeyboardInterrupt: 

In [ ]:
# ── Ultra-Fast Candidate Generation + All Evaluations ─────────────────
import torch
import torch.nn.functional as F
from transformers.modeling_outputs import BaseModelOutput
from sacrebleu.metrics import BLEU, CHRF

bleu = BLEU()
chrf = CHRF()

# ── Batch candidate generation ─────────────────────────────────────────
@torch.inference_mode()
def batch_get_candidates(model, tokenizer, texts, num_candidates=6,
                          forced_bos=None, batch_size=32):
    all_candidates = []
    all_scores     = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt',
            truncation=True, max_length=128,
            padding=True
        ).to(device)

        kwargs = dict(
            max_new_tokens       = 128,
            num_beams            = num_candidates,
            num_return_sequences = num_candidates,
            output_scores        = True,
            return_dict_in_generate = True,
        )
        if forced_bos is not None:
            kwargs['forced_bos_token_id'] = forced_bos

        outputs = model.generate(**inputs, **kwargs)

        n = len(batch)
        for j in range(n):
            start = j * num_candidates
            end   = start + num_candidates
            cands = [
                tokenizer.decode(seq, skip_special_tokens=True)
                for seq in outputs.sequences[start:end]
            ]
            scores = outputs.sequences_scores[start:end].tolist()
            all_candidates.append(cands)
            all_scores.append(scores)

        if i % 200 == 0:
            print(f"  {i}/{len(texts)}", flush=True)
        torch.cuda.empty_cache()

    return all_candidates, all_scores


# ── Generate all candidates in batch ──────────────────────────────────
print("Generating Helsinki candidates (batched)...")
all_hel_cands, all_hel_scores = batch_get_candidates(
    hel_eval_model, hel_eval_tok,
    hyp_dacf_curr, num_candidates=6, batch_size=32
)

print("Generating pipeline mBART candidates (batched)...")
all_pipe_cands, all_pipe_scores = batch_get_candidates(
    pipeline_s2e_model, pipeline_s2e_tok,
    hyp_dacf_curr, num_candidates=6,
    forced_bos=forced_bos_en, batch_size=32
)

print("All candidates generated")

# ── Beam vote ──────────────────────────────────────────────────────────
print("\nBeam vote...")
hyp_beam = []
for i in range(len(test_df)):
    hel_score  = all_hel_scores[i][0]
    pipe_score = all_pipe_scores[i][0]
    hyp_beam.append(
        all_hel_cands[i][0] if hel_score >= pipe_score
        else all_pipe_cands[i][0]
    )

b = bleu.corpus_score(hyp_beam, [references_en])
c = chrf.corpus_score(hyp_beam, [references_en])
print(f"Beam vote: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# ── QE tournament ──────────────────────────────────────────────────────
print("\nQE tournament...")

# Batch score all pairs at once
def batch_qe_tournament(source, candidates):
    if len(candidates) == 1:
        return candidates[0]

    wins = [0] * len(candidates)
    pairs_a = []
    pairs_b = []
    pair_indices = []

    for j in range(len(candidates)):
        for k in range(j+1, len(candidates)):
            pairs_a.append(f"[BN-EN] {source} {candidates[j]}")
            pairs_b.append(f"[BN-EN] {source} {candidates[k]}")
            pair_indices.append((j, k))

    # Encode all pairs at once
    enc_a = qe_tok(
        [f"[BN-EN] {source}"] * len(pair_indices),
        [candidates[j] for j, k in pair_indices],
        max_length=256, truncation=True,
        padding=True, return_tensors='pt'
    ).to(device)
    enc_b = qe_tok(
        [f"[BN-EN] {source}"] * len(pair_indices),
        [candidates[k] for j, k in pair_indices],
        max_length=256, truncation=True,
        padding=True, return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        scores_a = qe_model.score(
            enc_a['input_ids'], enc_a['attention_mask']
        )
        scores_b = qe_model.score(
            enc_b['input_ids'], enc_b['attention_mask']
        )
        logits = scores_a - scores_b

    for idx, (j, k) in enumerate(pair_indices):
        if logits[idx].item() > 0:
            wins[j] += 1
        else:
            wins[k] += 1

    return candidates[wins.index(max(wins))]


hyp_tournament = []
for i in range(len(test_df)):
    normalized     = hyp_dacf_curr[i]
    all_candidates = list(set(all_hel_cands[i] + all_pipe_cands[i]))
    best           = batch_qe_tournament(normalized, all_candidates)
    hyp_tournament.append(best)
    if i % 50 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_tournament, [references_en])
c = chrf.corpus_score(hyp_tournament, [references_en])
print(f"QE tournament: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# ── QE pointwise ───────────────────────────────────────────────────────
print("\nQE pointwise...")
hyp_qe = []
for i in range(len(test_df)):
    normalized     = hyp_dacf_curr[i]
    all_candidates = list(set(all_hel_cands[i] + all_pipe_cands[i]))
    scores         = [qe_score(normalized, c) for c in all_candidates]
    hyp_qe.append(all_candidates[scores.index(max(scores))])

b = bleu.corpus_score(hyp_qe, [references_en])
c = chrf.corpus_score(hyp_qe, [references_en])
print(f"QE pointwise: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# ── Combined beam + QE ─────────────────────────────────────────────────
print("\nCombined beam + QE...")

def norm(scores):
    mn, mx = min(scores), max(scores)
    if mx - mn < 1e-8:
        return [1.0] * len(scores)
    return [(s - mn) / (mx - mn) for s in scores]

for qe_weight in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    beam_weight  = 1.0 - qe_weight
    hyp_combined = []

    for i in range(len(test_df)):
        normalized = hyp_dacf_curr[i]

        all_candidates  = []
        all_beam_scores = []
        for cand, score in zip(all_hel_cands[i], all_hel_scores[i]):
            if cand not in all_candidates:
                all_candidates.append(cand)
                all_beam_scores.append(score)
        for cand, score in zip(all_pipe_cands[i], all_pipe_scores[i]):
            if cand not in all_candidates:
                all_candidates.append(cand)
                all_beam_scores.append(score)

        qe_scores = [qe_score(normalized, c) for c in all_candidates]

        combined = [
            beam_weight * b + qe_weight * q
            for b, q in zip(norm(all_beam_scores), norm(qe_scores))
        ]
        hyp_combined.append(all_candidates[combined.index(max(combined))])

    b = bleu.corpus_score(hyp_combined, [references_en])
    c = chrf.corpus_score(hyp_combined, [references_en])
    print(
        f"beam={beam_weight:.1f} + QE={qe_weight:.1f}: "
        f"BLEU: {b.score:.2f} | chrF: {c.score:.2f}",
        flush=True
    )

# ── Oracle ceiling on these candidates ────────────────────────────────
print("\nOracle ceiling on stored candidates...")
hyp_oracle = []
for i in range(len(test_df)):
    all_candidates = list(set(all_hel_cands[i] + all_pipe_cands[i]))
    best_score     = -1
    best_cand      = all_candidates[0]
    for cand in all_candidates:
        score = chrf.sentence_score(cand, [references_en[i]]).score
        if score > best_score:
            best_score = score
            best_cand  = cand
    hyp_oracle.append(best_cand)

b = bleu.corpus_score(hyp_oracle, [references_en])
c = chrf.corpus_score(hyp_oracle, [references_en])
print(f"Oracle (6 candidates each): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Generating Helsinki candidates (batched)...
  0/1871
  800/1871
  1600/1871
Generating pipeline mBART candidates (batched)...
  0/1871
  800/1871
  1600/1871
All candidates generated

Beam vote...
Beam vote: BLEU: 38.37 | chrF: 57.72

QE tournament...
  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
QE tournament: BLEU: 36.38 | chrF: 57.01

QE pointwise...
QE pointwise: BLEU: 36.38 | chrF: 57.01

Combined beam + QE...


KeyboardInterrupt: 

In [ ]:
# Free r2s models — no longer needed for s2e evaluation
del dacf_eval_model
del curr_eval_model
torch.cuda.empty_cache()
import gc
gc.collect()
print(f"Memory freed")

Memory freed


In [ ]:
def qe_mbr_fast(sources, candidates_list):
    """Batch QE-MBR for all sentences at once"""
    results = []

    for i, (source, candidates) in enumerate(zip(sources, candidates_list)):
        if len(candidates) == 1:
            results.append(candidates[0])
            continue

        n = len(candidates)
        # Build all pairs at once
        src_repeated = [f"[BN-EN] {source}"] * (n * (n-1))
        cands_a, cands_b = [], []
        pair_idx = []  # (j, k) meaning j vs k

        for j in range(n):
            for k in range(n):
                if j != k:
                    cands_a.append(candidates[j])
                    cands_b.append(candidates[k])
                    pair_idx.append(j)

        # Single tokenization call for all pairs
        enc_a = qe_tok(
            src_repeated, cands_a,
            max_length=256, truncation=True,
            padding=True, return_tensors='pt'
        ).to(device)
        enc_b = qe_tok(
            src_repeated, cands_b,
            max_length=256, truncation=True,
            padding=True, return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            sa = qe_model.score(enc_a['input_ids'], enc_a['attention_mask'])
            sb = qe_model.score(enc_b['input_ids'], enc_b['attention_mask'])
            probs = torch.sigmoid(sa - sb)

        # Accumulate win probs per candidate
        mbr = torch.zeros(n, device=device)
        counts = torch.zeros(n, device=device)
        for idx, j in enumerate(pair_idx):
            mbr[j]    += probs[idx]
            counts[j] += 1
        mbr = mbr / counts.clamp(min=1)

        results.append(candidates[mbr.argmax().item()])

        if i % 50 == 0:
            print(f"  {i}/{len(sources)}", flush=True)

        torch.cuda.empty_cache()

    return results


# Build candidate lists
all_candidates_list = [
    list(set(all_hel_cands[i] + all_pipe_cands[i]))
    for i in range(len(test_df))
]

# QE-MBR only
print("Evaluating QE-MBR...")
hyp_mbr = qe_mbr_fast(hyp_dacf_curr, all_candidates_list)
b = bleu.corpus_score(hyp_mbr, [references_en])
c = chrf.corpus_score(hyp_mbr, [references_en])
print(f"QE-MBR: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# QE-MBR + beam combined
print("\nQE-MBR + beam combined...")
for qe_weight in [0.3, 0.4, 0.5, 0.6]:
    beam_weight  = 1.0 - qe_weight
    hyp_combined = []

    for i in range(len(test_df)):
        normalized  = hyp_dacf_curr[i]
        candidates  = all_candidates_list[i]
        n           = len(candidates)

        all_beam_sc = []
        for cand in candidates:
            if cand in all_hel_cands[i]:
                idx = all_hel_cands[i].index(cand)
                all_beam_sc.append(all_hel_scores[i][idx])
            else:
                idx = all_pipe_cands[i].index(cand)
                all_beam_sc.append(all_pipe_scores[i][idx])

        # MBR scores
        src_rep  = [f"[BN-EN] {normalized}"] * (n * (n-1))
        ca, cb   = [], []
        pidx     = []
        for j in range(n):
            for k in range(n):
                if j != k:
                    ca.append(candidates[j])
                    cb.append(candidates[k])
                    pidx.append(j)

        enc_a = qe_tok(
            src_rep, ca, max_length=256,
            truncation=True, padding=True, return_tensors='pt'
        ).to(device)
        enc_b = qe_tok(
            src_rep, cb, max_length=256,
            truncation=True, padding=True, return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            sa    = qe_model.score(enc_a['input_ids'], enc_a['attention_mask'])
            sb    = qe_model.score(enc_b['input_ids'], enc_b['attention_mask'])
            probs = torch.sigmoid(sa - sb)

        mbr    = torch.zeros(n, device=device)
        counts = torch.zeros(n, device=device)
        for idx, j in enumerate(pidx):
            mbr[j]    += probs[idx]
            counts[j] += 1
        mbr = (mbr / counts.clamp(min=1)).cpu().tolist()

        norm_mbr  = norm(mbr)
        norm_beam = norm(all_beam_sc)

        combined = [
            beam_weight * b + qe_weight * m
            for b, m in zip(norm_beam, norm_mbr)
        ]
        hyp_combined.append(candidates[combined.index(max(combined))])
        torch.cuda.empty_cache()

    b = bleu.corpus_score(hyp_combined, [references_en])
    c = chrf.corpus_score(hyp_combined, [references_en])
    print(
        f"beam={beam_weight:.1f} + MBR={qe_weight:.1f}: "
        f"BLEU: {b.score:.2f} | chrF: {c.score:.2f}",
        flush=True
    )

Evaluating QE-MBR...
  0/1871
  50/1871
  100/1871
  150/1871
  200/1871
  250/1871
  300/1871
  350/1871
  400/1871
  450/1871
  500/1871
  550/1871
  600/1871
  650/1871
  700/1871
  750/1871
  800/1871
  850/1871
  900/1871
  950/1871
  1000/1871
  1050/1871
  1100/1871
  1150/1871
  1200/1871
  1250/1871
  1300/1871
  1350/1871
  1400/1871
  1450/1871
  1500/1871
  1550/1871
  1600/1871
  1650/1871
  1700/1871
  1750/1871
  1800/1871
  1850/1871
QE-MBR: BLEU: 36.38 | chrF: 57.01

QE-MBR + beam combined...
beam=0.7 + MBR=0.3: BLEU: 38.27 | chrF: 58.16
beam=0.6 + MBR=0.4: BLEU: 38.37 | chrF: 58.37
beam=0.5 + MBR=0.5: BLEU: 38.39 | chrF: 58.51


KeyboardInterrupt: 

### 12f — Span-Level System Fusion (Best BLEU Result)
Iteratively replaces spans in the beam vote winner with better alternatives scored by model loss. 38.55 BLEU — best overall result.

In [ ]:
import difflib
import torch

def fast_fusion_beam_score(source, base, candidates,
                            base_beam_score, cand_beam_scores,
                            tokenizer, model, forced_bos=None):
    """
    Fast fusion using beam scores instead of QE forward passes.
    Score full candidate with beam score, not per-span QE.
    """
    current       = base
    current_score = base_beam_score

    for cand, cand_score in zip(candidates, cand_beam_scores):
        if cand == current:
            continue

        matcher = difflib.SequenceMatcher(
            None, current.split(), cand.split(),
            autojunk=False
        )

        for opcode, i1, i2, j1, j2 in matcher.get_opcodes():
            if opcode == 'equal':
                continue

            current_words     = current.split()
            cand_words        = cand.split()
            alternative_words = (
                current_words[:i1] +
                cand_words[j1:j2] +
                current_words[i2:]
            )
            alternative = ' '.join(alternative_words).strip()

            if not alternative or alternative == current:
                continue

            # Score the alternative with the model that produced
            # the contributing candidate
            inputs = tokenizer(
                source, return_tensors='pt',
                truncation=True, max_length=128
            ).to(device)
            tgt = tokenizer(
                text_target=alternative,
                return_tensors='pt',
                truncation=True, max_length=128
            ).to(device)
            labels = tgt['input_ids'].clone()
            labels[labels == tokenizer.pad_token_id] = -100

            with torch.no_grad():
                kwargs = dict(
                    input_ids      = inputs['input_ids'],
                    attention_mask = inputs['attention_mask'],
                    labels         = labels,
                )
                if forced_bos is not None:
                    kwargs['forced_bos_token_id'] = forced_bos
                out   = model(**kwargs)
                score = -out.loss.item()

            if score > current_score:
                current       = alternative
                current_score = score
                break

    return current


# ── Ultra-fast fusion — no QE forward passes ───────────────────────────
print("Ultra-fast QE-Fusion (beam score span selection)...")
hyp_fast_fusion = []

for i in range(len(test_df)):
    normalized     = hyp_dacf_curr[i]
    all_candidates = list(set(all_hel_cands[i] + all_pipe_cands[i]))

    # Base = beam vote winner
    hel_score  = all_hel_scores[i][0]
    pipe_score = all_pipe_scores[i][0]

    if hel_score >= pipe_score:
        base            = all_hel_cands[i][0]
        base_score      = hel_score
        other_cands     = all_pipe_cands[i]
        other_scores    = all_pipe_scores[i]
        score_tokenizer = hel_eval_tok
        score_model     = hel_eval_model
        forced_bos_arg  = None
    else:
        base            = all_pipe_cands[i][0]
        base_score      = pipe_score
        other_cands     = all_hel_cands[i]
        other_scores    = all_hel_scores[i]
        score_tokenizer = pipeline_s2e_tok
        score_model     = pipeline_s2e_model
        forced_bos_arg  = forced_bos_en

    try:
        fused = fast_fusion_beam_score(
            normalized, base, other_cands, base_score, other_scores,
            score_tokenizer, score_model, forced_bos_arg
        )
    except RuntimeError:
        fused = base

    hyp_fast_fusion.append(fused)

    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_fast_fusion, [references_en])
c = chrf.corpus_score(hyp_fast_fusion, [references_en])
print(f"Fast fusion (beam score spans): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

changed = sum(
    1 for i in range(len(test_df))
    if hyp_fast_fusion[i] != all_hel_cands[i][0]
    and hyp_fast_fusion[i] != all_pipe_cands[i][0]
)
print(f"Modified: {changed}/{len(test_df)} sentences")

Ultra-fast QE-Fusion (beam score span selection)...
  0/1871
  100/1871
  200/1871
  300/1871
  400/1871
  500/1871
  600/1871
  700/1871
  800/1871
  900/1871
  1000/1871
  1100/1871
  1200/1871
  1300/1871
  1400/1871
  1500/1871
  1600/1871
  1700/1871
  1800/1871
Fast fusion (beam score spans): BLEU: 38.55 | chrF: 57.83
Modified: 26/1871 sentences


### 12g — Word Lattice Decoding (Negative Result)
29.83 BLEU — incoherent outputs due to unaligned word positions across candidates.

In [ ]:
import numpy as np
from collections import defaultdict
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# Load GPT-2 once
print("Loading GPT-2...")
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
gpt2_tok   = GPT2TokenizerFast.from_pretrained('gpt2')
gpt2_tok.pad_token = gpt2_tok.eos_token
gpt2_model.eval()
print("Done")

def batch_lm_score(texts, model, tokenizer, batch_size=32):
    """Score a batch of texts with GPT-2 in one pass"""
    all_scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(
            batch, return_tensors='pt',
            truncation=True, max_length=64,
            padding=True
        ).to(device)
        with torch.no_grad():
            out      = gpt2_model(**enc, labels=enc['input_ids'])
            # Per sample loss
            logits   = out.logits
            loss_fn  = torch.nn.CrossEntropyLoss(
                ignore_index=tokenizer.pad_token_id,
                reduction='none'
            )
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = enc['input_ids'][..., 1:].contiguous()
            per_tok      = loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            ).view(shift_labels.size())
            mask         = (shift_labels != tokenizer.pad_token_id).float()
            per_sample   = (per_tok * mask).sum(-1) / mask.sum(-1).clamp(min=1)
            all_scores.extend((-per_sample).tolist())
    return all_scores


def fast_lattice(candidates, scores, lm_weight=0.4, beam_width=50):
    """
    Fast lattice: build word graph, enumerate top paths,
    batch rescore all with GPT-2 in one forward pass.
    """
    # Normalize scores
    sc    = np.array(scores, dtype=np.float32)
    sc   -= sc.max()
    probs = np.exp(sc)
    probs /= probs.sum()

    # Build position → set of (word, cum_prob) mappings
    # lattice[pos][word] = total probability mass
    lattice = defaultdict(lambda: defaultdict(float))
    max_len = 0

    for cand, prob in zip(candidates, probs):
        words = cand.split()
        max_len = max(max_len, len(words))
        for pos, word in enumerate(words):
            lattice[pos][word] += prob

    # For each position collect top-k words by probability
    top_k = 3  # top words per position
    pos_words = {}
    for pos in range(max_len):
        if pos in lattice:
            words_at_pos = sorted(
                lattice[pos].items(),
                key=lambda x: x[1], reverse=True
            )[:top_k]
            pos_words[pos] = words_at_pos
        else:
            pos_words[pos] = []

    # Generate candidate paths from top words at each position
    # Use beam search over the lattice
    beam = [(0.0, [])]  # (score, path)

    for pos in range(max_len):
        if not pos_words[pos]:
            continue
        new_beam = []
        for path_score, path in beam:
            for word, word_prob in pos_words[pos]:
                new_score = path_score + np.log(word_prob + 1e-10)
                new_beam.append((new_score, path + [word]))
        # Keep top beam_width paths
        new_beam.sort(key=lambda x: x[0], reverse=True)
        beam = new_beam[:beam_width]

    if not beam:
        return candidates[0]

    # Extract all path texts
    path_texts   = [' '.join(path).strip() for _, path in beam]
    path_scores  = [s for s, _ in beam]

    # Also include original candidates
    all_texts    = path_texts + candidates
    all_lat_sc   = path_scores + list(scores)

    # Deduplicate
    seen   = {}
    for text, lat_sc in zip(all_texts, all_lat_sc):
        if text and text not in seen:
            seen[text] = lat_sc
    unique_texts = list(seen.keys())
    unique_lat   = list(seen.values())

    # Batch LM score all at once
    lm_scores = batch_lm_score(unique_texts, gpt2_model, gpt2_tok)

    # Normalize lattice scores
    lat_arr  = np.array(unique_lat, dtype=np.float32)
    lat_arr -= lat_arr.max()
    lat_norm = lat_arr / (np.abs(lat_arr).max() + 1e-8)

    lm_arr   = np.array(lm_scores, dtype=np.float32)
    lm_norm  = (lm_arr - lm_arr.mean()) / (lm_arr.std() + 1e-8)

    combined = (1 - lm_weight) * lat_norm + lm_weight * lm_norm
    best_idx = combined.argmax()

    return unique_texts[best_idx]


# ── Evaluate ───────────────────────────────────────────────────────────
print("Evaluating fast lattice decoding...")
hyp_lattice = []

for i in range(len(test_df)):
    all_cands = []
    all_sc    = []
    for cand, score in zip(all_hel_cands[i], all_hel_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)
    for cand, score in zip(all_pipe_cands[i], all_pipe_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)

    try:
        best = fast_lattice(all_cands, all_sc, lm_weight=0.4, beam_width=50)
    except Exception:
        best = all_cands[0]

    hyp_lattice.append(best)

    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_lattice, [references_en])
c = chrf.corpus_score(hyp_lattice, [references_en])
print(f"\nFast lattice: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

novel = sum(
    1 for i in range(len(test_df))
    if hyp_lattice[i] not in all_hel_cands[i]
    and hyp_lattice[i] not in all_pipe_cands[i]
)
print(f"Novel paths: {novel}/{len(test_df)}")

# Tune LM weight
print("\nTuning LM weight...")
for lm_w in [0.2, 0.3, 0.4, 0.5, 0.6]:
    hyp_lw = []
    for i in range(len(test_df)):
        all_cands = []
        all_sc    = []
        for cand, score in zip(all_hel_cands[i], all_hel_scores[i]):
            if cand not in all_cands:
                all_cands.append(cand)
                all_sc.append(score)
        for cand, score in zip(all_pipe_cands[i], all_pipe_scores[i]):
            if cand not in all_cands:
                all_cands.append(cand)
                all_sc.append(score)
        try:
            best = fast_lattice(all_cands, all_sc, lm_weight=lm_w, beam_width=50)
        except Exception:
            best = all_cands[0]
        hyp_lw.append(best)

    b = bleu.corpus_score(hyp_lw, [references_en])
    c = chrf.corpus_score(hyp_lw, [references_en])
    print(f"lm_weight={lm_w:.1f}: BLEU: {b.score:.2f} | chrF: {c.score:.2f}", flush=True)

Loading GPT-2...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Done
Evaluating fast lattice decoding...
  0/1871
  100/1871
  200/1871
  300/1871
  400/1871
  500/1871
  600/1871
  700/1871
  800/1871
  900/1871
  1000/1871
  1100/1871
  1200/1871
  1300/1871
  1400/1871
  1500/1871
  1600/1871
  1700/1871
  1800/1871

Fast lattice: BLEU: 29.83 | chrF: 50.21
Novel paths: 13/1871

Tuning LM weight...
lm_weight=0.2: BLEU: 29.72 | chrF: 50.21
lm_weight=0.3: BLEU: 29.61 | chrF: 50.14


KeyboardInterrupt: 

In [ ]:
# ── ROVER with TER Alignment ───────────────────────────────────────────
print("\n" + "="*60)
print("ROVER + TER SYSTEM COMBINATION")
print("="*60)

!pip install sacrebleu --quiet  # already installed but ensures TER available

import re
from itertools import zip_longest
from collections import defaultdict
import numpy as np

# ── TER Alignment ─────────────────────────────────────────────────────
def ter_align(hypothesis, reference):
    """
    Align hypothesis to reference using TER-style edit operations.
    Returns aligned hypothesis with insertion markers.
    """
    hyp_words = hypothesis.split()
    ref_words  = reference.split()

    # Standard edit distance with backtrace
    n, m = len(ref_words), len(hyp_words)
    dp   = np.zeros((n+1, m+1), dtype=np.float32)

    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # deletion
                    dp[i][j-1],    # insertion
                    dp[i-1][j-1]   # substitution
                )

    # Backtrace to get alignment
    aligned = []
    i, j    = n, m

    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref_words[i-1] == hyp_words[j-1]:
            aligned.append(('match', ref_words[i-1], hyp_words[j-1]))
            i -= 1
            j -= 1
        elif j > 0 and (i == 0 or dp[i][j-1] <= dp[i-1][j] and dp[i][j-1] <= dp[i-1][j-1]):
            aligned.append(('insert', None, hyp_words[j-1]))
            j -= 1
        elif i > 0 and (j == 0 or dp[i-1][j] <= dp[i][j-1] and dp[i-1][j] <= dp[i-1][j-1]):
            aligned.append(('delete', ref_words[i-1], None))
            i -= 1
        else:
            aligned.append(('sub', ref_words[i-1], hyp_words[j-1]))
            i -= 1
            j -= 1

    aligned.reverse()
    return aligned


def build_confusion_network(backbone, hypotheses, hyp_scores):
    """
    Build confusion network by aligning all hypotheses to backbone.
    Each slot contains alternatives from different systems.
    """
    # Normalize scores to weights
    scores  = np.array(hyp_scores, dtype=np.float32)
    scores -= scores.max()
    weights = np.exp(scores)
    weights /= weights.sum()

    # Align each hypothesis to backbone
    backbone_words = backbone.split()

    # Initialize confusion network slots
    # Each slot: {word: weight}
    cn_slots = [{backbone_words[i]: weights[0]}
                for i in range(len(backbone_words))]

    for hyp, weight in zip(hypotheses[1:], weights[1:]):
        alignment = ter_align(hyp, backbone)

        ref_pos = 0  # position in backbone/CN
        for op, ref_word, hyp_word in alignment:
            if op == 'match':
                # Both agree — add weight to existing slot
                if ref_pos < len(cn_slots):
                    cn_slots[ref_pos][hyp_word] = (
                        cn_slots[ref_pos].get(hyp_word, 0) + weight
                    )
                ref_pos += 1

            elif op == 'sub':
                # Disagreement — add alternative to slot
                if ref_pos < len(cn_slots):
                    cn_slots[ref_pos][hyp_word] = (
                        cn_slots[ref_pos].get(hyp_word, 0) + weight
                    )
                ref_pos += 1

            elif op == 'insert':
                # Hypothesis has extra word — add epsilon slot
                if ref_pos < len(cn_slots):
                    eps_slot = {'': weight, **cn_slots[ref_pos]}
                    cn_slots.insert(ref_pos, eps_slot)
                    ref_pos += 1

            elif op == 'delete':
                # Hypothesis missing word — add NULL to slot
                if ref_pos < len(cn_slots):
                    cn_slots[ref_pos][''] = (
                        cn_slots[ref_pos].get('', 0) + weight
                    )
                ref_pos += 1

    return cn_slots


def decode_confusion_network(cn_slots, lm_model, lm_tok,
                              lm_weight=0.3, beam_width=20):
    """
    Decode confusion network using beam search + LM rescoring.
    """
    # Beam: list of (score, words_so_far)
    beam = [(0.0, [])]

    for slot in cn_slots:
        if not slot:
            continue

        new_beam = []
        for path_score, path in beam:
            for word, word_weight in slot.items():
                slot_score = np.log(word_weight + 1e-10)
                new_path   = path + ([word] if word else [])
                new_beam.append((
                    path_score + slot_score,
                    new_path
                ))

        # Keep top beam_width
        new_beam.sort(key=lambda x: x[0], reverse=True)
        beam = new_beam[:beam_width]

    if not beam:
        return None

    # Get unique complete paths
    path_texts  = list(set(
        ' '.join(p).strip() for _, p in beam if p
    ))

    if not path_texts:
        return None

    # Batch LM rescore all paths
    lm_scores = batch_lm_score(path_texts, lm_model, lm_tok)

    # Get CN scores for each unique path
    path_cn_scores = []
    for text in path_texts:
        for score, path in beam:
            if ' '.join(path).strip() == text:
                path_cn_scores.append(score)
                break
        else:
            path_cn_scores.append(-999)

    # Normalize and combine
    cn_arr  = np.array(path_cn_scores, dtype=np.float32)
    cn_arr -= cn_arr.max()
    cn_norm = cn_arr / (np.abs(cn_arr).max() + 1e-8)

    lm_arr  = np.array(lm_scores, dtype=np.float32)
    lm_norm = (lm_arr - lm_arr.mean()) / (lm_arr.std() + 1e-8)

    combined = (1 - lm_weight) * cn_norm + lm_weight * lm_norm
    best_idx = combined.argmax()

    return path_texts[best_idx]


def rover_combine(candidates, scores, lm_model, lm_tok,
                  lm_weight=0.3, beam_width=20):
    """
    Full ROVER pipeline:
    1. Pick backbone (highest beam score)
    2. Align all others to backbone via TER
    3. Build confusion network
    4. Decode with beam search + LM rescoring
    """
    if len(candidates) == 1:
        return candidates[0]

    # Sort by score — best first = backbone
    paired     = sorted(zip(scores, candidates), reverse=True)
    scores_s   = [p[0] for p in paired]
    cands_s    = [p[1] for p in paired]

    backbone    = cands_s[0]
    hypotheses  = cands_s
    hyp_scores  = scores_s

    # Build confusion network
    cn_slots = build_confusion_network(backbone, hypotheses, hyp_scores)

    # Decode
    result = decode_confusion_network(
        cn_slots, lm_model, lm_tok,
        lm_weight=lm_weight, beam_width=beam_width
    )

    return result if result else backbone


# ── Evaluate ROVER ─────────────────────────────────────────────────────
print("Evaluating ROVER + TER...")
hyp_rover = []

for i in range(len(test_df)):
    all_cands = []
    all_sc    = []
    for cand, score in zip(all_hel_cands[i], all_hel_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)
    for cand, score in zip(all_pipe_cands[i], all_pipe_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)

    try:
        best = rover_combine(
            all_cands, all_sc,
            gpt2_model, gpt2_tok,
            lm_weight=0.3, beam_width=20
        )
    except Exception as e:
        best = all_cands[0]

    hyp_rover.append(best)

    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_rover, [references_en])
c = chrf.corpus_score(hyp_rover, [references_en])
print(f"\nROVER + TER: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

novel = sum(
    1 for i in range(len(test_df))
    if hyp_rover[i] not in all_hel_cands[i]
    and hyp_rover[i] not in all_pipe_cands[i]
)
print(f"Novel outputs: {novel}/{len(test_df)}")

# Tune LM weight
print("\nTuning LM weight...")
for lm_w in [0.1, 0.2, 0.3, 0.4, 0.5]:
    hyp_lw = []
    for i in range(len(test_df)):
        all_cands = []
        all_sc    = []
        for cand, score in zip(all_hel_cands[i], all_hel_scores[i]):
            if cand not in all_cands:
                all_cands.append(cand)
                all_sc.append(score)
        for cand, score in zip(all_pipe_cands[i], all_pipe_scores[i]):
            if cand not in all_cands:
                all_cands.append(cand)
                all_sc.append(score)
        try:
            best = rover_combine(
                all_cands, all_sc,
                gpt2_model, gpt2_tok,
                lm_weight=lm_w, beam_width=20
            )
        except Exception:
            best = all_cands[0]
        hyp_lw.append(best)

    b = bleu.corpus_score(hyp_lw, [references_en])
    c = chrf.corpus_score(hyp_lw, [references_en])
    print(
        f"lm_weight={lm_w:.1f}: BLEU: {b.score:.2f} | chrF: {c.score:.2f}",
        flush=True
    )


ROVER + TER SYSTEM COMBINATION
Evaluating ROVER + TER...
  0/1871
  100/1871
  200/1871
  300/1871
  400/1871
  500/1871
  600/1871
  700/1871
  800/1871
  900/1871
  1000/1871
  1100/1871
  1200/1871
  1300/1871
  1400/1871
  1500/1871
  1600/1871
  1700/1871
  1800/1871

ROVER + TER: BLEU: 16.41 | chrF: 47.19
Novel outputs: 1554/1871

Tuning LM weight...


KeyboardInterrupt: 

### 12h — Reinforcement Learning Candidate Selection
Binary RL selector and enhanced RL with candidate features. Enhanced model achieved 67.4% val accuracy but did not beat beam vote on test — beam wins 66% of disagreements.

In [ ]:
# ── REINFORCE Candidate Selection ─────────────────────────────────────
print("\n" + "="*60)
print("REINFORCE CANDIDATE SELECTION")
print("="*60)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from transformers import AutoTokenizer, AutoModel
import numpy as np

RL_SAVE = f'{DRIVE_PATH}/rl_selector'

# ── Policy Network ─────────────────────────────────────────────────────
class CandidateSelector(nn.Module):
    """
    Takes normalized Bengali source as input.
    Outputs logits over N candidates.
    Uses XLM-RoBERTa encoder — already understands Bengali.
    """
    def __init__(self, encoder_name='xlm-roberta-base', n_candidates=12):
        super().__init__()
        self.encoder     = AutoModel.from_pretrained(encoder_name)
        hidden           = self.encoder.config.hidden_size
        self.policy_head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, n_candidates)
        )

    def mean_pool(self, outputs, attention_mask):
        token_emb = outputs.last_hidden_state
        mask_exp  = attention_mask.unsqueeze(-1).float()
        return (
            (token_emb * mask_exp).sum(1) /
            mask_exp.sum(1).clamp(min=1e-9)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled = self.mean_pool(outputs, attention_mask)
        logits = self.policy_head(pooled)
        return logits


# ── Training Data ──────────────────────────────────────────────────────
print("Building RL training data...")

# For each training sentence we need:
# 1. Normalized Bengali source
# 2. All candidates
# 3. chrF of each candidate against reference (reward signal)

rl_tok = AutoTokenizer.from_pretrained('xlm-roberta-base')

# Generate candidates for training set
print("Generating train candidates...")
train_hel_cands,  train_hel_scores  = batch_get_candidates(
    hel_eval_model, hel_eval_tok,
    normalized_train, num_candidates=6, batch_size=8
)
train_pipe_cands, train_pipe_scores = batch_get_candidates(
    pipeline_s2e_model, pipeline_s2e_tok,
    normalized_train, num_candidates=6,
    forced_bos=forced_bos_en, batch_size=8
)
print("Done")

# Build candidate pools and rewards
print("Computing rewards...")
train_rl_data = []
for i in range(len(train_df)):
    reference = train_english[i]
    normalized = normalized_train[i]

    all_cands = []
    all_sc    = []
    for cand, score in zip(train_hel_cands[i], train_hel_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)
    for cand, score in zip(train_pipe_cands[i], train_pipe_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)

    # Reward = chrF of each candidate
    rewards = [
        chrf.sentence_score(c, [reference]).score / 100.0
        for c in all_cands
    ]

    # Baseline = beam vote reward
    hel_sc   = train_hel_scores[i][0]
    pipe_sc  = train_pipe_scores[i][0]
    baseline_cand = (
        train_hel_cands[i][0] if hel_sc >= pipe_sc
        else train_pipe_cands[i][0]
    )
    baseline = chrf.sentence_score(baseline_cand, [reference]).score / 100.0

    # Pad candidates to fixed size (12)
    while len(all_cands) < 12:
        all_cands.append(all_cands[-1])
        rewards.append(rewards[-1])
        all_sc.append(all_sc[-1])

    train_rl_data.append({
        'source':    normalized,
        'candidates': all_cands[:12],
        'rewards':    rewards[:12],
        'baseline':   baseline,
    })

    if i % 500 == 0:
        print(f"  {i}/{len(train_df)}", flush=True)

print(f"RL training examples: {len(train_rl_data)}")

# Check reward distribution
all_rewards  = [max(d['rewards']) for d in train_rl_data]
all_baseline = [d['baseline'] for d in train_rl_data]
print(f"Mean max reward:  {np.mean(all_rewards):.3f}")
print(f"Mean baseline:    {np.mean(all_baseline):.3f}")
print(f"Mean advantage:   {np.mean([r-b for r,b in zip(all_rewards,all_baseline)]):.3f}")

# ── REINFORCE Training ─────────────────────────────────────────────────
N_CANDIDATES = 12
policy = CandidateSelector(
    encoder_name='xlm-roberta-base',
    n_candidates=N_CANDIDATES
).to(device)

optimizer  = Adam(policy.parameters(), lr=1e-5)
RL_EPOCHS  = 3
best_val_reward = 0.0

# Val data — use val_df
print("\nGenerating val candidates...")
val_rural     = val_df['rural'].tolist()
val_english   = val_df['english'].tolist()

normalized_val_rl = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    val_rural, alpha=0.5, batch_size=64, num_beams=4
)

val_hel_cands,  val_hel_scores  = batch_get_candidates(
    hel_eval_model, hel_eval_tok,
    normalized_val_rl, num_candidates=6, batch_size=8
)
val_pipe_cands, val_pipe_scores = batch_get_candidates(
    pipeline_s2e_model, pipeline_s2e_tok,
    normalized_val_rl, num_candidates=6,
    forced_bos=forced_bos_en, batch_size=8
)

val_rl_data = []
for i in range(len(val_df)):
    reference  = val_english[i]
    normalized = normalized_val_rl[i]

    all_cands = []
    all_sc    = []
    for cand, score in zip(val_hel_cands[i], val_hel_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)
    for cand, score in zip(val_pipe_cands[i], val_pipe_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)

    rewards = [
        chrf.sentence_score(c, [reference]).score / 100.0
        for c in all_cands
    ]

    while len(all_cands) < 12:
        all_cands.append(all_cands[-1])
        rewards.append(rewards[-1])

    val_rl_data.append({
        'source':     normalized,
        'candidates': all_cands[:12],
        'rewards':    rewards[:12],
    })

print(f"Val RL examples: {len(val_rl_data)}")

# ── REINFORCE Loop ─────────────────────────────────────────────────────
for epoch in range(RL_EPOCHS):
    policy.train()
    random.shuffle(train_rl_data)
    total_loss   = 0
    total_reward = 0

    for step, item in enumerate(train_rl_data):
        # Encode source
        enc = rl_tok(
            item['source'],
            return_tensors='pt',
            truncation=True,
            max_length=128,
            padding='max_length'
        ).to(device)

        # Get action logits
        logits = policy(enc['input_ids'], enc['attention_mask'])
        probs  = F.softmax(logits, dim=-1)
        dist   = torch.distributions.Categorical(probs)

        # Sample action
        action = dist.sample()
        log_prob = dist.log_prob(action)

        # Get reward and advantage
        reward   = torch.tensor(
            item['rewards'][action.item()],
            dtype=torch.float32, device=device
        )
        baseline = torch.tensor(
            item['baseline'],
            dtype=torch.float32, device=device
        )
        advantage = reward - baseline

        # REINFORCE loss
        loss = -log_prob * advantage

        loss.backward()

        # Gradient accumulation — update every 8 steps
        if (step + 1) % 8 == 0:
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss   += loss.item()
        total_reward += reward.item()

        if step % 500 == 0:
            print(
                f"  Epoch {epoch+1} step {step}/{len(train_rl_data)} "
                f"| loss {total_loss/(step+1):.4f} "
                f"| avg reward {total_reward/(step+1):.3f}",
                flush=True
            )

    # Validation — greedy policy (argmax)
    policy.eval()
    val_rewards = []

    with torch.no_grad():
        for item in val_rl_data:
            enc = rl_tok(
                item['source'],
                return_tensors='pt',
                truncation=True,
                max_length=128,
                padding='max_length'
            ).to(device)

            logits = policy(enc['input_ids'], enc['attention_mask'])
            action = logits.argmax(-1).item()
            reward = item['rewards'][action]
            val_rewards.append(reward)

    avg_val_reward = np.mean(val_rewards)
    print(
        f"Epoch {epoch+1} | val reward: {avg_val_reward:.4f}",
        flush=True
    )

    if avg_val_reward > best_val_reward:
        best_val_reward = avg_val_reward
        os.makedirs(RL_SAVE, exist_ok=True)
        torch.save(policy.state_dict(), f'{RL_SAVE}/policy.pt')
        print(f"  Saved | best reward: {best_val_reward:.4f}", flush=True)

print(f"\nBest val reward: {best_val_reward:.4f}")

# ── Evaluate RL Policy ─────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING RL POLICY")
print("="*60)

policy.load_state_dict(torch.load(f'{RL_SAVE}/policy.pt'))
policy.eval()

hyp_rl = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    all_cands = []
    all_sc    = []
    for cand, score in zip(all_hel_cands[i], all_hel_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)
    for cand, score in zip(all_pipe_cands[i], all_pipe_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)

    while len(all_cands) < 12:
        all_cands.append(all_cands[-1])

    enc = rl_tok(
        normalized,
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding='max_length'
    ).to(device)

    with torch.no_grad():
        logits = policy(enc['input_ids'], enc['attention_mask'])
        action = logits.argmax(-1).item()

    hyp_rl.append(all_cands[min(action, len(all_cands)-1)])

    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_rl, [references_en])
c = chrf.corpus_score(hyp_rl, [references_en])
print(f"\nRL policy: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Compare with beam vote
hyp_beam_rl = []
for i in range(len(test_df)):
    hel_score  = all_hel_scores[i][0]
    pipe_score = all_pipe_scores[i][0]
    hyp_beam_rl.append(
        all_hel_cands[i][0] if hel_score >= pipe_score
        else all_pipe_cands[i][0]
    )

b = bleu.corpus_score(hyp_beam_rl, [references_en])
c = chrf.corpus_score(hyp_beam_rl, [references_en])
print(f"Beam vote:  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Also try RL + fusion
print("\nRL policy + fast fusion...")
hyp_rl_fusion = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    all_cands = []
    all_sc


REINFORCE CANDIDATE SELECTION
Building RL training data...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Generating train candidates...
  0/9375
  200/9375
  400/9375
  600/9375
  800/9375
  1000/9375
  1200/9375
  1400/9375
  1600/9375
  1800/9375
  2000/9375
  2200/9375
  2400/9375
  2600/9375
  2800/9375
  3000/9375
  3200/9375
  3400/9375
  3600/9375
  3800/9375
  4000/9375
  4200/9375
  4400/9375
  4600/9375
  4800/9375
  5000/9375
  5200/9375
  5400/9375
  5600/9375
  5800/9375
  6000/9375
  6200/9375
  6400/9375
  6600/9375
  6800/9375
  7000/9375
  7200/9375
  7400/9375
  7600/9375
  7800/9375
  8000/9375
  8200/9375
  8400/9375
  8600/9375
  8800/9375
  9000/9375
  9200/9375
  0/9375
  200/9375
  400/9375
  600/9375
  800/9375
  1000/9375
  1200/9375
  1400/9375
  1600/9375
  1800/9375
  2000/9375
  2200/9375
  2400/9375
  2600/9375
  2800/9375
  3000/9375
  3200/9375
  3400/9375
  3600/9375
  3800/9375
  4000/9375
  4200/9375
  4400/9375
  4600/9375
  4800/9375
  5000/9375
  5200/9375
  5400/9375
  5600/9375
  5800/9375
  6000/9375
  6200/9375
  6400/9375
  6600/9375
  6800/9375

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Generating val candidates...


NameError: name 'dacf_eval_model' is not defined

In [ ]:
# ── Reload r2s models ──────────────────────────────────────────────────
print("Loading r2s models...")
dacf_eval_model = MBartForConditionalGeneration.from_pretrained(
    DACF_SAVE, local_files_only=True
).to(device)
dacf_eval_model.eval()

curr_eval_model = MBartForConditionalGeneration.from_pretrained(
    CURRICULUM_SAVE, local_files_only=True
).to(device)
curr_eval_model.eval()
print("Loaded")

# ── Generate val normalized outputs ───────────────────────────────────
print("Generating val normalized outputs...")
val_rural   = val_df['rural'].tolist()
val_english = val_df['english'].tolist()

normalized_val_rl = translate_ensemble_generate(
    dacf_eval_model, curr_eval_model,
    val_rural, alpha=0.5, batch_size=64, num_beams=4
)
print(f"Generated {len(normalized_val_rl)} val outputs")

# Free r2s models immediately
del dacf_eval_model, curr_eval_model
torch.cuda.empty_cache()
print("r2s models freed")

# ── Generate val candidates ────────────────────────────────────────────
print("Generating val Helsinki candidates...")
val_hel_cands,  val_hel_scores  = batch_get_candidates(
    hel_eval_model, hel_eval_tok,
    normalized_val_rl, num_candidates=6, batch_size=8
)

print("Generating val pipeline mBART candidates...")
val_pipe_cands, val_pipe_scores = batch_get_candidates(
    pipeline_s2e_model, pipeline_s2e_tok,
    normalized_val_rl, num_candidates=6,
    forced_bos=forced_bos_en, batch_size=8
)

# ── Build val RL data ──────────────────────────────────────────────────
val_rl_data = []
for i in range(len(val_df)):
    reference  = val_english[i]
    normalized = normalized_val_rl[i]

    all_cands = []
    all_sc    = []
    for cand, score in zip(val_hel_cands[i], val_hel_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)
    for cand, score in zip(val_pipe_cands[i], val_pipe_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)

    rewards = [
        chrf.sentence_score(c, [reference]).score / 100.0
        for c in all_cands
    ]

    while len(all_cands) < 12:
        all_cands.append(all_cands[-1])
        rewards.append(rewards[-1])

    val_rl_data.append({
        'source':     normalized,
        'candidates': all_cands[:12],
        'rewards':    rewards[:12],
    })

print(f"Val RL examples: {len(val_rl_data)}")
print(f"Mean val max reward: {np.mean([max(d['rewards']) for d in val_rl_data]):.3f}")

# ── Train RL Policy ────────────────────────────────────────────────────
print("\nTraining RL policy...")
RL_EPOCHS       = 3
best_val_reward = 0.0

for epoch in range(RL_EPOCHS):
    policy.train()
    random.shuffle(train_rl_data)
    total_loss   = 0
    total_reward = 0

    for step, item in enumerate(train_rl_data):
        enc = rl_tok(
            item['source'],
            return_tensors='pt',
            truncation=True,
            max_length=128,
            padding='max_length'
        ).to(device)

        logits   = policy(enc['input_ids'], enc['attention_mask'])
        probs    = F.softmax(logits, dim=-1)
        dist     = torch.distributions.Categorical(probs)
        action   = dist.sample()
        log_prob = dist.log_prob(action)

        reward   = torch.tensor(
            item['rewards'][action.item()],
            dtype=torch.float32, device=device
        )
        baseline = torch.tensor(
            item['baseline'],
            dtype=torch.float32, device=device
        )
        advantage = reward - baseline
        loss      = -log_prob * advantage

        loss.backward()

        if (step + 1) % 8 == 0:
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss   += loss.item()
        total_reward += reward.item()

        if step % 500 == 0:
            print(
                f"  Epoch {epoch+1} step {step}/{len(train_rl_data)} "
                f"| loss {total_loss/(step+1):.4f} "
                f"| avg reward {total_reward/(step+1):.3f}",
                flush=True
            )

    # Validation
    policy.eval()
    val_rewards = []
    with torch.no_grad():
        for item in val_rl_data:
            enc = rl_tok(
                item['source'],
                return_tensors='pt',
                truncation=True,
                max_length=128,
                padding='max_length'
            ).to(device)
            logits = policy(enc['input_ids'], enc['attention_mask'])
            action = logits.argmax(-1).item()
            reward = item['rewards'][min(action, len(item['rewards'])-1)]
            val_rewards.append(reward)

    avg_val_reward = np.mean(val_rewards)
    print(f"Epoch {epoch+1} | val reward: {avg_val_reward:.4f}", flush=True)

    if avg_val_reward > best_val_reward:
        best_val_reward = avg_val_reward
        os.makedirs(RL_SAVE, exist_ok=True)
        torch.save(policy.state_dict(), f'{RL_SAVE}/policy.pt')
        print(f"  Saved | best reward: {best_val_reward:.4f}", flush=True)

print(f"\nBest val reward: {best_val_reward:.4f}")

# ── Evaluate ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING RL POLICY")
print("="*60)

policy.load_state_dict(torch.load(f'{RL_SAVE}/policy.pt'))
policy.eval()

hyp_rl = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    all_cands = []
    for cand in all_hel_cands[i]:
        if cand not in all_cands:
            all_cands.append(cand)
    for cand in all_pipe_cands[i]:
        if cand not in all_cands:
            all_cands.append(cand)

    while len(all_cands) < 12:
        all_cands.append(all_cands[-1])

    enc = rl_tok(
        normalized,
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding='max_length'
    ).to(device)

    with torch.no_grad():
        logits = policy(enc['input_ids'], enc['attention_mask'])
        action = logits.argmax(-1).item()

    hyp_rl.append(all_cands[min(action, len(all_cands)-1)])

    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_rl, [references_en])
c = chrf.corpus_score(hyp_rl, [references_en])
print(f"\nRL policy: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Beam vote comparison
hyp_beam_rl = []
for i in range(len(test_df)):
    hel_score  = all_hel_scores[i][0]
    pipe_score = all_pipe_scores[i][0]
    hyp_beam_rl.append(
        all_hel_cands[i][0] if hel_score >= pipe_score
        else all_pipe_cands[i][0]
    )

b = bleu.corpus_score(hyp_beam_rl, [references_en])
c = chrf.corpus_score(hyp_beam_rl, [references_en])
print(f"Beam vote:  BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# RL + fusion
print("\nRL + fast fusion...")
hyp_rl_fusion = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    all_cands = []
    all_sc    = []
    for cand, score in zip(all_hel_cands[i], all_hel_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)
    for cand, score in zip(all_pipe_cands[i], all_pipe_scores[i]):
        if cand not in all_cands:
            all_cands.append(cand)
            all_sc.append(score)

    while len(all_cands) < 12:
        all_cands.append(all_cands[-1])
        all_sc.append(all_sc[-1])

    enc = rl_tok(
        normalized,
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding='max_length'
    ).to(device)

    with torch.no_grad():
        logits = policy(enc['input_ids'], enc['attention_mask'])
        action = logits.argmax(-1).item()

    base       = all_cands[min(action, len(all_cands)-1)]
    base_score = all_sc[min(action, len(all_cands)-1)]

    if base in all_hel_cands[i]:
        score_tokenizer = hel_eval_tok
        score_model_    = hel_eval_model
        forced_bos_arg  = None
    else:
        score_tokenizer = pipeline_s2e_tok
        score_model_    = pipeline_s2e_model
        forced_bos_arg  = forced_bos_en

    try:
        fused, _ = turbo_fusion(
            normalized, all_cands, all_sc,
            score_tokenizer, score_model_,
            forced_bos_arg, max_iter=3
        )
    except Exception:
        fused = base

    hyp_rl_fusion.append(fused)

    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

    torch.cuda.empty_cache()

b = bleu.corpus_score(hyp_rl_fusion, [references_en])
c = chrf.corpus_score(hyp_rl_fusion, [references_en])
print(f"RL + fusion: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Loading r2s models...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Loaded
Generating val normalized outputs...
  0/1250
Generated 1250 val outputs
r2s models freed
Generating val Helsinki candidates...
  0/1250
  200/1250
  400/1250
  600/1250
  800/1250
  1000/1250
  1200/1250
Generating val pipeline mBART candidates...
  0/1250
  200/1250
  400/1250
  600/1250
  800/1250
  1000/1250
  1200/1250
Val RL examples: 1250
Mean val max reward: 0.730

Training RL policy...
  Epoch 1 step 0/9375 | loss 1.0365 | avg reward 0.660
  Epoch 1 step 500/9375 | loss -0.5994 | avg reward 0.653
  Epoch 1 step 1000/9375 | loss -0.6099 | avg reward 0.651
  Epoch 1 step 1500/9375 | loss -0.6046 | avg reward 0.652
  Epoch 1 step 2000/9375 | loss -0.5747 | avg reward 0.663
  Epoch 1 step 2500/9375 | loss -0.5466 | avg reward 0.671
  Epoch 1 step 3000/9375 | loss -0.5192 | avg reward 0.677
  Epoch 1 step 3500/9375 | loss -0.4963 | avg reward 0.682
  Epoch 1 step 4000/9375 | loss -0.4708 | avg reward 0.685
  Epoch 1 step 4500/9375 | loss -0.4496 | avg reward 0.692
  Epoch 1 

KeyboardInterrupt: 

In [ ]:
# Binary policy — Helsinki vs pipeline mBART
class BinarySelector(nn.Module):
    def __init__(self, encoder_name='xlm-roberta-base'):
        super().__init__()
        self.encoder     = AutoModel.from_pretrained(encoder_name)
        hidden           = self.encoder.config.hidden_size
        self.policy_head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2)  # binary: Helsinki or mBART
        )

    def mean_pool(self, outputs, attention_mask):
        token_emb = outputs.last_hidden_state
        mask_exp  = attention_mask.unsqueeze(-1).float()
        return (
            (token_emb * mask_exp).sum(1) /
            mask_exp.sum(1).clamp(min=1e-9)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled = self.mean_pool(outputs, attention_mask)
        return self.policy_head(pooled)


# Rebuild training data for binary action
binary_train = []
for i in range(len(train_df)):
    hel_cand   = train_hel_cands[i][0]
    pipe_cand  = train_pipe_cands[i][0]
    reference  = train_english[i]
    normalized = normalized_train[i]

    hel_reward  = chrf.sentence_score(hel_cand,  [reference]).score / 100.0
    pipe_reward = chrf.sentence_score(pipe_cand, [reference]).score / 100.0

    # Baseline = beam vote
    hel_sc  = train_hel_scores[i][0]
    pipe_sc = train_pipe_scores[i][0]
    if hel_sc >= pipe_sc:
        baseline = hel_reward
    else:
        baseline = pipe_reward

    binary_train.append({
        'source':       normalized,
        'hel_reward':   hel_reward,
        'pipe_reward':  pipe_reward,
        'baseline':     baseline,
    })

binary_val = []
for i in range(len(val_df)):
    hel_cand   = val_hel_cands[i][0]
    pipe_cand  = val_pipe_cands[i][0]
    reference  = val_english[i]
    normalized = normalized_val_rl[i]

    hel_reward  = chrf.sentence_score(hel_cand,  [reference]).score / 100.0
    pipe_reward = chrf.sentence_score(pipe_cand, [reference]).score / 100.0

    binary_val.append({
        'source':      normalized,
        'hel_reward':  hel_reward,
        'pipe_reward': pipe_reward,
    })

# Check class balance
hel_better  = sum(1 for d in binary_train if d['hel_reward'] >= d['pipe_reward'])
pipe_better = len(binary_train) - hel_better
print(f"Helsinki better:  {hel_better} ({100*hel_better/len(binary_train):.1f}%)")
print(f"Pipeline better:  {pipe_better} ({100*pipe_better/len(binary_train):.1f}%)")

# Train binary selector
binary_policy = BinarySelector().to(device)
optimizer_bin = Adam(binary_policy.parameters(), lr=2e-5)

BINARY_EPOCHS   = 3
best_val_reward = 0.0

for epoch in range(BINARY_EPOCHS):
    binary_policy.train()
    random.shuffle(binary_train)
    total_loss   = 0
    total_reward = 0

    for step, item in enumerate(binary_train):
        enc = rl_tok(
            item['source'],
            return_tensors='pt',
            truncation=True,
            max_length=128,
            padding='max_length'
        ).to(device)

        logits   = binary_policy(enc['input_ids'], enc['attention_mask'])
        probs    = F.softmax(logits, dim=-1)
        dist     = torch.distributions.Categorical(probs)
        action   = dist.sample()
        log_prob = dist.log_prob(action)

        # Action 0 = Helsinki, Action 1 = pipeline mBART
        reward = torch.tensor(
            item['hel_reward'] if action.item() == 0 else item['pipe_reward'],
            dtype=torch.float32, device=device
        )
        baseline  = torch.tensor(item['baseline'], dtype=torch.float32, device=device)
        advantage = reward - baseline
        loss      = -log_prob * advantage

        loss.backward()

        if (step + 1) % 8 == 0:
            torch.nn.utils.clip_grad_norm_(binary_policy.parameters(), 1.0)
            optimizer_bin.step()
            optimizer_bin.zero_grad()

        total_loss   += loss.item()
        total_reward += reward.item()

        if step % 500 == 0:
            print(
                f"  Epoch {epoch+1} step {step}/{len(binary_train)} "
                f"| loss {total_loss/(step+1):.4f} "
                f"| avg reward {total_reward/(step+1):.3f}",
                flush=True
            )

    # Validation
    binary_policy.eval()
    val_rewards = []
    with torch.no_grad():
        for item in binary_val:
            enc = rl_tok(
                item['source'],
                return_tensors='pt',
                truncation=True,
                max_length=128,
                padding='max_length'
            ).to(device)
            logits = binary_policy(enc['input_ids'], enc['attention_mask'])
            action = logits.argmax(-1).item()
            reward = item['hel_reward'] if action == 0 else item['pipe_reward']
            val_rewards.append(reward)

    avg_val_reward = np.mean(val_rewards)
    print(f"Epoch {epoch+1} | val reward: {avg_val_reward:.4f}", flush=True)

    if avg_val_reward > best_val_reward:
        best_val_reward = avg_val_reward
        os.makedirs(RL_SAVE, exist_ok=True)
        torch.save(binary_policy.state_dict(), f'{RL_SAVE}/binary_policy.pt')
        print(f"  Saved | best reward: {best_val_reward:.4f}", flush=True)

print(f"\nBest val reward: {best_val_reward:.4f}")

# ── Evaluate binary selector ───────────────────────────────────────────
print("\nEvaluating binary selector...")
binary_policy.load_state_dict(
    torch.load(f'{RL_SAVE}/binary_policy.pt')
)
binary_policy.eval()

hyp_binary = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]

    enc = rl_tok(
        normalized,
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding='max_length'
    ).to(device)

    with torch.no_grad():
        logits = binary_policy(enc['input_ids'], enc['attention_mask'])
        action = logits.argmax(-1).item()

    hyp_binary.append(
        all_hel_cands[i][0] if action == 0
        else all_pipe_cands[i][0]
    )

    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_binary, [references_en])
c = chrf.corpus_score(hyp_binary, [references_en])
print(f"\nBinary RL selector: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Beam vote comparison
hyp_beam_bin = []
for i in range(len(test_df)):
    hel_score  = all_hel_scores[i][0]
    pipe_score = all_pipe_scores[i][0]
    hyp_beam_bin.append(
        all_hel_cands[i][0] if hel_score >= pipe_score
        else all_pipe_cands[i][0]
    )

b = bleu.corpus_score(hyp_beam_bin, [references_en])
c = chrf.corpus_score(hyp_beam_bin, [references_en])
print(f"Beam vote:          BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

Helsinki better:  4881 (52.1%)
Pipeline better:  4494 (47.9%)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Epoch 1 step 0/9375 | loss -0.3171 | avg reward 0.359
  Epoch 1 step 500/9375 | loss -0.0374 | avg reward 0.851
  Epoch 1 step 1000/9375 | loss -0.0259 | avg reward 0.878
  Epoch 1 step 1500/9375 | loss -0.0249 | avg reward 0.882
  Epoch 1 step 2000/9375 | loss -0.0196 | avg reward 0.881
  Epoch 1 step 2500/9375 | loss -0.0163 | avg reward 0.886
  Epoch 1 step 3000/9375 | loss -0.0147 | avg reward 0.885
  Epoch 1 step 3500/9375 | loss -0.0178 | avg reward 0.882
  Epoch 1 step 4000/9375 | loss -0.0182 | avg reward 0.884
  Epoch 1 step 4500/9375 | loss -0.0162 | avg reward 0.886
  Epoch 1 step 5000/9375 | loss -0.0146 | avg reward 0.886
  Epoch 1 step 5500/9375 | loss -0.0146 | avg reward 0.887
  Epoch 1 step 6000/9375 | loss -0.0137 | avg reward 0.887
  Epoch 1 step 6500/9375 | loss -0.0126 | avg reward 0.887


KeyboardInterrupt: 

In [ ]:
# Save current state
torch.save(binary_policy.state_dict(), f'{RL_SAVE}/binary_policy.pt')
rl_tok.save_pretrained(RL_SAVE)
print("Saved")

Saved


In [ ]:
binary_policy.eval()
hyp_binary = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]
    enc = rl_tok(
        normalized, return_tensors='pt',
        truncation=True, max_length=128,
        padding='max_length'
    ).to(device)
    with torch.no_grad():
        logits = binary_policy(enc['input_ids'], enc['attention_mask'])
        action = logits.argmax(-1).item()
    hyp_binary.append(
        all_hel_cands[i][0] if action == 0
        else all_pipe_cands[i][0]
    )
    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_binary, [references_en])
c = chrf.corpus_score(hyp_binary, [references_en])
print(f"\nBinary RL: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

  0/1871
  100/1871
  200/1871
  300/1871
  400/1871
  500/1871
  600/1871
  700/1871
  800/1871
  900/1871
  1000/1871
  1100/1871
  1200/1871
  1300/1871
  1400/1871
  1500/1871
  1600/1871
  1700/1871
  1800/1871

Binary RL: BLEU: 37.13 | chrF: 55.93


In [ ]:
# ── Enhanced Binary RL with Candidate Features ────────────────────────
print("\n" + "="*60)
print("ENHANCED BINARY RL — SOURCE + CANDIDATES")
print("="*60)


# Build training data
print("Building enhanced training data...")
enhanced_train = []

for i in range(len(train_df)):
    hel_cand   = train_hel_cands[i][0]
    pipe_cand  = train_pipe_cands[i][0]
    reference  = train_english[i]
    normalized = normalized_train[i]

    hel_reward  = chrf.sentence_score(hel_cand,  [reference]).score / 100.0
    pipe_reward = chrf.sentence_score(pipe_cand, [reference]).score / 100.0

    hel_sc  = train_hel_scores[i][0]
    pipe_sc = train_pipe_scores[i][0]
    baseline = (
        hel_reward if hel_sc >= pipe_sc else pipe_reward
    )

    # Input = source + both candidates
    input_text = (
        f"[BN-EN] {normalized} "
        f"[HEL] {hel_cand} "
        f"[PIPE] {pipe_cand}"
    )

    enhanced_train.append({
        'input':       input_text,
        'hel_reward':  hel_reward,
        'pipe_reward': pipe_reward,
        'baseline':    baseline,
        'label':       0 if hel_reward >= pipe_reward else 1,
    })

enhanced_val = []
for i in range(len(val_df)):
    hel_cand   = val_hel_cands[i][0]
    pipe_cand  = val_pipe_cands[i][0]
    reference  = val_df['english'].tolist()[i]
    normalized = normalized_val_rl[i]

    hel_reward  = chrf.sentence_score(hel_cand,  [reference]).score / 100.0
    pipe_reward = chrf.sentence_score(pipe_cand, [reference]).score / 100.0

    input_text = (
        f"[BN-EN] {normalized} "
        f"[HEL] {hel_cand} "
        f"[PIPE] {pipe_cand}"
    )

    enhanced_val.append({
        'input':       input_text,
        'hel_reward':  hel_reward,
        'pipe_reward': pipe_reward,
        'label':       0 if hel_reward >= pipe_reward else 1,
    })

print(f"Train: {len(enhanced_train)} | Val: {len(enhanced_val)}")

# Check label balance
hel_better = sum(1 for d in enhanced_train if d['label'] == 0)
print(f"Helsinki better: {hel_better} ({100*hel_better/len(enhanced_train):.1f}%)")
print(f"Pipeline better: {len(enhanced_train)-hel_better} ({100*(len(enhanced_train)-hel_better)/len(enhanced_train):.1f}%)")

# ── Model — source + candidates as input ──────────────────────────────
class EnhancedSelector(nn.Module):
    def __init__(self, encoder_name='xlm-roberta-base'):
        super().__init__()
        self.encoder     = AutoModel.from_pretrained(encoder_name)
        hidden           = self.encoder.config.hidden_size
        self.policy_head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 2)
        )

    def mean_pool(self, outputs, attention_mask):
        token_emb = outputs.last_hidden_state
        mask_exp  = attention_mask.unsqueeze(-1).float()
        return (
            (token_emb * mask_exp).sum(1) /
            mask_exp.sum(1).clamp(min=1e-9)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled = self.mean_pool(outputs, attention_mask)
        return self.policy_head(pooled)


# ── Step 1: Supervised pretraining ────────────────────────────────────
print("\nStep 1: Supervised pretraining...")

enhanced_tok   = AutoTokenizer.from_pretrained('xlm-roberta-base')
enhanced_model = EnhancedSelector().to(device)
optimizer_enh  = AdamW(enhanced_model.parameters(), lr=2e-5, weight_decay=0.01)

# Add special tokens
special_tokens = {'additional_special_tokens': ['[BN-EN]', '[HEL]', '[PIPE]']}
enhanced_tok.add_special_tokens(special_tokens)
enhanced_model.encoder.resize_token_embeddings(len(enhanced_tok))

PRETRAIN_EPOCHS = 2
ce_loss         = nn.CrossEntropyLoss()
best_val_acc    = 0.0

total_steps  = (len(enhanced_train) // 8) * PRETRAIN_EPOCHS
warmup_steps = total_steps // 10
scheduler_enh = get_cosine_schedule_with_warmup(
    optimizer_enh,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

for epoch in range(PRETRAIN_EPOCHS):
    enhanced_model.train()
    random.shuffle(enhanced_train)
    total_loss = 0
    optimizer_enh.zero_grad()

    for step, item in enumerate(enhanced_train):
        enc = enhanced_tok(
            item['input'],
            return_tensors='pt',
            truncation=True,
            max_length=256,
            padding='max_length'
        ).to(device)

        logits = enhanced_model(enc['input_ids'], enc['attention_mask'])
        label  = torch.tensor([item['label']], device=device)
        loss   = ce_loss(logits, label) / 8

        loss.backward()

        if (step + 1) % 8 == 0:
            torch.nn.utils.clip_grad_norm_(enhanced_model.parameters(), 1.0)
            optimizer_enh.step()
            scheduler_enh.step()
            optimizer_enh.zero_grad()

        total_loss += loss.item() * 8

        if step % 500 == 0:
            print(
                f"  Pretrain Epoch {epoch+1} step {step}/{len(enhanced_train)} "
                f"| loss {total_loss/(step+1):.4f}",
                flush=True
            )

    # Val accuracy
    enhanced_model.eval()
    correct = 0
    with torch.no_grad():
        for item in enhanced_val:
            enc = enhanced_tok(
                item['input'],
                return_tensors='pt',
                truncation=True,
                max_length=256,
                padding='max_length'
            ).to(device)
            logits = enhanced_model(enc['input_ids'], enc['attention_mask'])
            pred   = logits.argmax(-1).item()
            if pred == item['label']:
                correct += 1

    acc = correct / len(enhanced_val)
    print(f"Pretrain Epoch {epoch+1} | val accuracy: {acc:.3f}", flush=True)

    if acc > best_val_acc:
        best_val_acc = acc
        torch.save(enhanced_model.state_dict(), f'{RL_SAVE}/enhanced_pretrained.pt')
        print(f"  Saved | best acc: {best_val_acc:.3f}", flush=True)

print(f"Best pretrain val accuracy: {best_val_acc:.3f}")

# ── Step 2: REINFORCE fine-tuning ─────────────────────────────────────
print("\nStep 2: REINFORCE fine-tuning...")

# Load best pretrained
enhanced_model.load_state_dict(
    torch.load(f'{RL_SAVE}/enhanced_pretrained.pt')
)
optimizer_rl  = Adam(enhanced_model.parameters(), lr=5e-6)
RL_EPOCHS     = 2
best_val_rew  = 0.0

for epoch in range(RL_EPOCHS):
    enhanced_model.train()
    random.shuffle(enhanced_train)
    total_loss   = 0
    total_reward = 0

    for step, item in enumerate(enhanced_train):
        enc = enhanced_tok(
            item['input'],
            return_tensors='pt',
            truncation=True,
            max_length=256,
            padding='max_length'
        ).to(device)

        logits   = enhanced_model(enc['input_ids'], enc['attention_mask'])
        probs    = F.softmax(logits, dim=-1)
        dist     = torch.distributions.Categorical(probs)
        action   = dist.sample()
        log_prob = dist.log_prob(action)

        reward = torch.tensor(
            item['hel_reward'] if action.item() == 0 else item['pipe_reward'],
            dtype=torch.float32, device=device
        )
        baseline  = torch.tensor(item['baseline'], dtype=torch.float32, device=device)
        advantage = reward - baseline
        loss      = -log_prob * advantage

        loss.backward()

        if (step + 1) % 8 == 0:
            torch.nn.utils.clip_grad_norm_(enhanced_model.parameters(), 1.0)
            optimizer_rl.step()
            optimizer_rl.zero_grad()

        total_loss   += loss.item()
        total_reward += reward.item()

        if step % 500 == 0:
            print(
                f"  RL Epoch {epoch+1} step {step}/{len(enhanced_train)} "
                f"| loss {total_loss/(step+1):.4f} "
                f"| avg reward {total_reward/(step+1):.3f}",
                flush=True
            )

    # Val reward
    enhanced_model.eval()
    val_rewards = []
    with torch.no_grad():
        for item in enhanced_val:
            enc = enhanced_tok(
                item['input'],
                return_tensors='pt',
                truncation=True,
                max_length=256,
                padding='max_length'
            ).to(device)
            logits = enhanced_model(enc['input_ids'], enc['attention_mask'])
            action = logits.argmax(-1).item()
            reward = item['hel_reward'] if action == 0 else item['pipe_reward']
            val_rewards.append(reward)

    avg_val_rew = np.mean(val_rewards)
    print(f"RL Epoch {epoch+1} | val reward: {avg_val_rew:.4f}", flush=True)

    if avg_val_rew > best_val_rew:
        best_val_rew = avg_val_rew
        torch.save(enhanced_model.state_dict(), f'{RL_SAVE}/enhanced_rl.pt')
        print(f"  Saved | best reward: {best_val_rew:.4f}", flush=True)

print(f"Best RL val reward: {best_val_rew:.4f}")

# ── Evaluate ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVALUATING ENHANCED RL SELECTOR")
print("="*60)

enhanced_model.load_state_dict(torch.load(f'{RL_SAVE}/enhanced_rl.pt'))
enhanced_model.eval()

hyp_enhanced = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]
    hel_cand   = all_hel_cands[i][0]
    pipe_cand  = all_pipe_cands[i][0]

    input_text = (
        f"[BN-EN] {normalized} "
        f"[HEL] {hel_cand} "
        f"[PIPE] {pipe_cand}"
    )

    enc = enhanced_tok(
        input_text,
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding='max_length'
    ).to(device)

    with torch.no_grad():
        logits = enhanced_model(enc['input_ids'], enc['attention_mask'])
        action = logits.argmax(-1).item()

    hyp_enhanced.append(
        hel_cand if action == 0 else pipe_cand
    )

    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_enhanced, [references_en])
c = chrf.corpus_score(hyp_enhanced, [references_en])
print(f"\nEnhanced RL (pretrain + REINFORCE): BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Beam vote comparison
hyp_beam_enh = []
for i in range(len(test_df)):
    hel_score  = all_hel_scores[i][0]
    pipe_score = all_pipe_scores[i][0]
    hyp_beam_enh.append(
        all_hel_cands[i][0] if hel_score >= pipe_score
        else all_pipe_cands[i][0]
    )

b = bleu.corpus_score(hyp_beam_enh, [references_en])
c = chrf.corpus_score(hyp_beam_enh, [references_en])
print(f"Beam vote:                          BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Also evaluate supervised-only (no REINFORCE) for comparison
print("\nEvaluating supervised-only (no REINFORCE)...")
enhanced_model.load_state_dict(torch.load(f'{RL_SAVE}/enhanced_pretrained.pt'))
enhanced_model.eval()

hyp_supervised = []
for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]
    hel_cand   = all_hel_cands[i][0]
    pipe_cand  = all_pipe_cands[i][0]

    input_text = (
        f"[BN-EN] {normalized} "
        f"[HEL] {hel_cand} "
        f"[PIPE] {pipe_cand}"
    )

    enc = enhanced_tok(
        input_text,
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding='max_length'
    ).to(device)

    with torch.no_grad():
        logits = enhanced_model(enc['input_ids'], enc['attention_mask'])
        action = logits.argmax(-1).item()

    hyp_supervised.append(
        hel_cand if action == 0 else pipe_cand
    )

b = bleu.corpus_score(hyp_supervised, [references_en])
c = chrf.corpus_score(hyp_supervised, [references_en])
print(f"Supervised only: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Show examples
print("\nExample selections:")
for i in range(5):
    normalized = hyp_dacf_curr[i]
    hel_cand   = all_hel_cands[i][0]
    pipe_cand  = all_pipe_cands[i][0]
    input_text = (
        f"[BN-EN] {normalized} "
        f"[HEL] {hel_cand} "
        f"[PIPE] {pipe_cand}"
    )
    enc = enhanced_tok(
        input_text, return_tensors='pt',
        truncation=True, max_length=256,
        padding='max_length'
    ).to(device)
    enhanced_model.load_state_dict(torch.load(f'{RL_SAVE}/enhanced_rl.pt'))
    enhanced_model.eval()
    with torch.no_grad():
        logits = enhanced_model(enc['input_ids'], enc['attention_mask'])
        action = logits.argmax(-1).item()
        probs  = F.softmax(logits, dim=-1).squeeze()

    print(f"\n[{i}] Rural:    {test_df.iloc[i]['rural']}")
    print(f"     Ref:      {references_en[i]}")
    print(f"     Helsinki: {hel_cand}")
    print(f"     Pipeline: {pipe_cand}")
    print(f"     Picks:    {'Helsinki' if action==0 else 'Pipeline'} "
          f"(conf: {probs[action].item():.3f})")
    print(f"     Ref chrF Helsinki: {chrf.sentence_score(hel_cand, [references_en[i]]).score:.1f}")
    print(f"     Ref chrF Pipeline: {chrf.sentence_score(pipe_cand, [references_en[i]]).score:.1f}")


ENHANCED BINARY RL — SOURCE + CANDIDATES
Building enhanced training data...
Train: 9375 | Val: 1250
Helsinki better: 4881 (52.1%)
Pipeline better: 4494 (47.9%)

Step 1: Supervised pretraining...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Pretrain Epoch 1 step 0/9375 | loss 0.7362
  Pretrain Epoch 1 step 500/9375 | loss 0.6953
  Pretrain Epoch 1 step 1000/9375 | loss 0.6876
  Pretrain Epoch 1 step 1500/9375 | loss 0.6512
  Pretrain Epoch 1 step 2000/9375 | loss 0.5988
  Pretrain Epoch 1 step 2500/9375 | loss 0.5757
  Pretrain Epoch 1 step 3000/9375 | loss 0.5451
  Pretrain Epoch 1 step 3500/9375 | loss 0.5219
  Pretrain Epoch 1 step 4000/9375 | loss 0.5195
  Pretrain Epoch 1 step 4500/9375 | loss 0.5097
  Pretrain Epoch 1 step 5000/9375 | loss 0.5123
  Pretrain Epoch 1 step 5500/9375 | loss 0.4997
  Pretrain Epoch 1 step 6000/9375 | loss 0.5052
  Pretrain Epoch 1 step 6500/9375 | loss 0.5048
  Pretrain Epoch 1 step 7000/9375 | loss 0.5018
  Pretrain Epoch 1 step 7500/9375 | loss 0.5008
  Pretrain Epoch 1 step 8000/9375 | loss 0.4952
  Pretrain Epoch 1 step 8500/9375 | loss 0.4871
  Pretrain Epoch 1 step 9000/9375 | loss 0.4810
Pretrain Epoch 1 | val accuracy: 0.660
  Saved | best acc: 0.660
  Pretrain Epoch 2 step 0/9

KeyboardInterrupt: 

In [ ]:
torch.save(enhanced_model.state_dict(), f'{RL_SAVE}/enhanced_rl.pt')
enhanced_tok.save_pretrained(RL_SAVE)
print("Saved")

Saved


In [ ]:
enhanced_model.eval()
hyp_enhanced = []

for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]
    hel_cand   = all_hel_cands[i][0]
    pipe_cand  = all_pipe_cands[i][0]

    input_text = (
        f"[BN-EN] {normalized} "
        f"[HEL] {hel_cand} "
        f"[PIPE] {pipe_cand}"
    )

    enc = enhanced_tok(
        input_text, return_tensors='pt',
        truncation=True, max_length=256,
        padding='max_length'
    ).to(device)

    with torch.no_grad():
        logits = enhanced_model(enc['input_ids'], enc['attention_mask'])
        action = logits.argmax(-1).item()

    hyp_enhanced.append(
        hel_cand if action == 0 else pipe_cand
    )

    if i % 100 == 0:
        print(f"  {i}/{len(test_df)}", flush=True)

b = bleu.corpus_score(hyp_enhanced, [references_en])
c = chrf.corpus_score(hyp_enhanced, [references_en])
print(f"\nEnhanced RL: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

# Beam vote
hyp_beam_enh = []
for i in range(len(test_df)):
    hel_score  = all_hel_scores[i][0]
    pipe_score = all_pipe_scores[i][0]
    hyp_beam_enh.append(
        all_hel_cands[i][0] if hel_score >= pipe_score
        else all_pipe_cands[i][0]
    )

b = bleu.corpus_score(hyp_beam_enh, [references_en])
c = chrf.corpus_score(hyp_beam_enh, [references_en])
print(f"Beam vote:   BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

  0/1871
  100/1871
  200/1871
  300/1871
  400/1871
  500/1871
  600/1871
  700/1871
  800/1871
  900/1871
  1000/1871
  1100/1871
  1200/1871
  1300/1871
  1400/1871
  1500/1871
  1600/1871
  1700/1871
  1800/1871

Enhanced RL: BLEU: 37.26 | chrF: 56.15
Beam vote:   BLEU: 38.37 | chrF: 57.72


In [ ]:
# How often does enhanced RL agree with beam vote?
enhanced_actions = []
beam_actions_enh = []

for i in range(len(test_df)):
    normalized = hyp_dacf_curr[i]
    hel_cand   = all_hel_cands[i][0]
    pipe_cand  = all_pipe_cands[i][0]

    input_text = (
        f"[BN-EN] {normalized} "
        f"[HEL] {hel_cand} "
        f"[PIPE] {pipe_cand}"
    )
    enc = enhanced_tok(
        input_text, return_tensors='pt',
        truncation=True, max_length=256,
        padding='max_length'
    ).to(device)
    with torch.no_grad():
        logits = enhanced_model(enc['input_ids'], enc['attention_mask'])
        probs  = F.softmax(logits, dim=-1).squeeze()
        action = logits.argmax(-1).item()
    enhanced_actions.append((action, probs[action].item()))

    hel_score  = all_hel_scores[i][0]
    pipe_score = all_pipe_scores[i][0]
    beam_actions_enh.append(0 if hel_score >= pipe_score else 1)

agreements    = sum(1 for e, b in zip(enhanced_actions, beam_actions_enh) if e[0] == b)
disagreements = len(test_df) - agreements
print(f"Agrees with beam:    {agreements} ({100*agreements/len(test_df):.1f}%)")
print(f"Disagrees with beam: {disagreements} ({100*disagreements/len(test_df):.1f}%)")

# When they disagree, who is right?
disagree_enhanced_wins = 0
disagree_beam_wins     = 0
for i in range(len(test_df)):
    if enhanced_actions[i][0] != beam_actions_enh[i]:
        enh_pick  = all_hel_cands[i][0] if enhanced_actions[i][0] == 0 else all_pipe_cands[i][0]
        beam_pick = all_hel_cands[i][0] if beam_actions_enh[i]    == 0 else all_pipe_cands[i][0]
        enh_chrf  = chrf.sentence_score(enh_pick,  [references_en[i]]).score
        beam_chrf = chrf.sentence_score(beam_pick, [references_en[i]]).score
        if enh_chrf > beam_chrf:
            disagree_enhanced_wins += 1
        else:
            disagree_beam_wins += 1

print(f"\nOn disagreements:")
print(f"  Enhanced wins: {disagree_enhanced_wins}")
print(f"  Beam wins:     {disagree_beam_wins}")

Agrees with beam:    992 (53.0%)
Disagrees with beam: 879 (47.0%)

On disagreements:
  Enhanced wins: 299
  Beam wins:     580


In [ ]:
# Only use enhanced model when confidence is high
print("Confidence-gated combination...")

for conf_threshold in [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]:
    hyp_gated = []
    used_enhanced = 0

    for i in range(len(test_df)):
        action, confidence = enhanced_actions[i]
        beam_action        = beam_actions_enh[i]

        if confidence >= conf_threshold and action != beam_action:
            # Enhanced model disagrees with high confidence — trust it
            hyp_gated.append(
                all_hel_cands[i][0] if action == 0
                else all_pipe_cands[i][0]
            )
            used_enhanced += 1
        else:
            # Low confidence or agrees — use beam vote
            hyp_gated.append(
                all_hel_cands[i][0] if beam_action == 0
                else all_pipe_cands[i][0]
            )

    b = bleu.corpus_score(hyp_gated, [references_en])
    c = chrf.corpus_score(hyp_gated, [references_en])
    print(
        f"conf>={conf_threshold:.2f}: BLEU: {b.score:.2f} | chrF: {c.score:.2f} "
        f"| overrides: {used_enhanced}/{len(test_df)}",
        flush=True
    )

Confidence-gated combination...
conf>=0.60: BLEU: 37.27 | chrF: 56.16 | overrides: 878/1871
conf>=0.65: BLEU: 37.27 | chrF: 56.16 | overrides: 878/1871
conf>=0.70: BLEU: 37.27 | chrF: 56.16 | overrides: 877/1871
conf>=0.75: BLEU: 37.27 | chrF: 56.16 | overrides: 877/1871
conf>=0.80: BLEU: 37.27 | chrF: 56.16 | overrides: 873/1871
conf>=0.85: BLEU: 37.27 | chrF: 56.16 | overrides: 873/1871
conf>=0.90: BLEU: 37.14 | chrF: 56.07 | overrides: 871/1871


## RUN EVERYTHING

In [ ]:
import os
import torch
import torch.nn.functional as F
import difflib
import pandas as pd
from transformers import (
    MBartForConditionalGeneration, MBart50TokenizerFast,
    MarianMTModel, MarianTokenizer,
    LogitsProcessor, LogitsProcessorList
)
from transformers.modeling_outputs import BaseModelOutput
from torch.amp import autocast
from sacrebleu.metrics import BLEU, CHRF
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PATH        = '/content/drive/MyDrive/bengali_translation'
MODEL_NAME        = 'facebook/mbart-large-50-many-to-many-mmt'
DACF_SAVE         = f'{DRIVE_PATH}/mbart_dacf'
CURRICULUM_SAVE   = f'{DRIVE_PATH}/mbart_mrasp2_curriculum'
PIPELINE_S2E_SAVE = f'{DRIVE_PATH}/mbart_pipeline_s2e'
HELSINKI_SAVE     = f'{DRIVE_PATH}/helsinki_s2e'
SRC_LANG          = 'bn_IN'
device            = 'cuda' if torch.cuda.is_available() else 'cpu'

test_df       = pd.read_csv('vashantor_test.csv').dropna().reset_index(drop=True)
references_en = test_df['english'].tolist()
texts         = test_df['rural'].tolist()

bleu = BLEU()
chrf = CHRF()

eval_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
eval_tokenizer.src_lang = SRC_LANG
forced_bos    = eval_tokenizer.lang_code_to_id[SRC_LANG]
forced_bos_en = eval_tokenizer.lang_code_to_id['en_XX']

print("Loading models...")
dacf_model = MBartForConditionalGeneration.from_pretrained(
    DACF_SAVE, local_files_only=True
).to(device)
dacf_model.eval()

curr_model = MBartForConditionalGeneration.from_pretrained(
    CURRICULUM_SAVE, local_files_only=True
).to(device)
curr_model.eval()

pipeline_s2e_model = MBartForConditionalGeneration.from_pretrained(
    f'{PIPELINE_S2E_SAVE}/checkpoint-879', local_files_only=True
).to(device)
pipeline_s2e_model.eval()
pipeline_s2e_tok = MBart50TokenizerFast.from_pretrained(
    f'{PIPELINE_S2E_SAVE}/checkpoint-879'
)
pipeline_s2e_tok.src_lang = SRC_LANG

hel_model = MarianMTModel.from_pretrained(
    HELSINKI_SAVE, local_files_only=True
).to(device)
hel_model.eval()
hel_tok = MarianTokenizer.from_pretrained(HELSINKI_SAVE)
print("All models loaded")


class EnsembleLogitsProcessor(LogitsProcessor):
    def __init__(self, model2, encoder_hidden2, attention_mask, alpha=0.5, num_beams=4):
        self.model2         = model2
        self.enc_hidden2    = encoder_hidden2
        self.attention_mask = attention_mask
        self.alpha          = alpha
        self.num_beams      = num_beams
        self._expanded      = False

    def __call__(self, input_ids, scores):
        if not self._expanded:
            self.enc_hidden2    = self.enc_hidden2.repeat_interleave(self.num_beams, dim=0)
            self.attention_mask = self.attention_mask.repeat_interleave(self.num_beams, dim=0)
            self._expanded      = True
        with torch.inference_mode(), autocast('cuda'):
            out2 = self.model2(
                attention_mask    = self.attention_mask,
                decoder_input_ids = input_ids,
                encoder_outputs   = BaseModelOutput(
                    last_hidden_state=self.enc_hidden2
                ),
                use_cache = False,
            )
        lp1 = F.log_softmax(scores,                        dim=-1)
        lp2 = F.log_softmax(out2.logits[:, -1, :].float(), dim=-1)
        return self.alpha * lp1 + (1 - self.alpha) * lp2


@torch.inference_mode()
def translate_ensemble(model1, model2, texts, alpha=0.5, batch_size=64, num_beams=4):
    all_outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = eval_tokenizer(
            batch, return_tensors='pt',
            truncation=True, max_length=128, padding=True
        ).to(device)
        with autocast('cuda'):
            enc1 = model1.model.encoder(**inputs)
            enc2 = model2.model.encoder(**inputs)
        processor = EnsembleLogitsProcessor(
            model2          = model2,
            encoder_hidden2 = enc2.last_hidden_state.clone(),
            attention_mask  = inputs['attention_mask'].clone(),
            alpha           = alpha,
            num_beams       = num_beams,
        )
        outputs = model1.generate(
            **inputs,
            encoder_outputs     = BaseModelOutput(
                last_hidden_state=enc1.last_hidden_state
            ),
            forced_bos_token_id = forced_bos,
            max_new_tokens      = 128,
            num_beams           = num_beams,
            logits_processor    = LogitsProcessorList([processor]),
            early_stopping      = True,
        )
        for seq in outputs:
            all_outputs.append(
                eval_tokenizer.decode(seq, skip_special_tokens=True)
            )
        if i % 200 == 0:
            print(f"  r2s {i}/{len(texts)}", flush=True)
        torch.cuda.empty_cache()
    return all_outputs


@torch.inference_mode()
def get_candidates(model, tokenizer, texts, num_candidates=6,
                   forced_bos=None, batch_size=8):
    all_cands  = []
    all_scores = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt',
            truncation=True, max_length=128, padding=True
        ).to(device)
        kwargs = dict(
            max_new_tokens          = 128,
            num_beams               = max(num_candidates, 2),
            num_return_sequences    = num_candidates,
            output_scores           = True,
            return_dict_in_generate = True,
        )
        if forced_bos is not None:
            kwargs['forced_bos_token_id'] = forced_bos
        outputs = model.generate(**inputs, **kwargs)
        n = len(batch)
        for j in range(n):
            start = j * num_candidates
            end   = start + num_candidates
            cands = [
                tokenizer.decode(seq, skip_special_tokens=True)
                for seq in outputs.sequences[start:end]
            ]
            scores = outputs.sequences_scores[start:end].tolist()
            all_cands.append(cands)
            all_scores.append(scores)
        if i % 200 == 0:
            print(f"  candidates {i}/{len(texts)}", flush=True)
        torch.cuda.empty_cache()
    return all_cands, all_scores


def fast_fusion(source, base, base_score, candidates, cand_scores,
                score_tokenizer, score_model, forced_bos_arg=None):
    current       = base
    current_score = base_score
    for cand, cand_score in zip(candidates, cand_scores):
        if cand == current:
            continue
        matcher = difflib.SequenceMatcher(
            None, current.split(), cand.split(), autojunk=False
        )
        for opcode, i1, i2, j1, j2 in matcher.get_opcodes():
            if opcode == 'equal':
                continue
            alt = ' '.join(
                current.split()[:i1] +
                cand.split()[j1:j2] +
                current.split()[i2:]
            ).strip()
            if not alt or alt == current:
                continue
            inputs = score_tokenizer(
                source, return_tensors='pt',
                truncation=True, max_length=128
            ).to(device)
            tgt = score_tokenizer(
                text_target=alt, return_tensors='pt',
                truncation=True, max_length=128
            ).to(device)
            labels = tgt['input_ids'].clone()
            labels[labels == score_tokenizer.pad_token_id] = -100
            with torch.no_grad():
                kwargs = dict(
                    input_ids      = inputs['input_ids'],
                    attention_mask = inputs['attention_mask'],
                    labels         = labels,
                )
                if forced_bos_arg is not None:
                    kwargs['forced_bos_token_id'] = forced_bos_arg
                score = -score_model(**kwargs).loss.item()
            if score > current_score:
                current       = alt
                current_score = score
                break
    return current


def translate_rural_to_english(rural_texts):
    """
    Full pipeline: rural Bengali → English
    Returns list of English translations
    """
    print("Step 1: r2s normalization...")
    normalized = translate_ensemble(
        dacf_model, curr_model,
        rural_texts, alpha=0.5, batch_size=64, num_beams=4
    )

    print("Step 2: generating s2e candidates...")
    hel_cands,  hel_scores  = get_candidates(
        hel_model, hel_tok,
        normalized, num_candidates=6, batch_size=8
    )
    pipe_cands, pipe_scores = get_candidates(
        pipeline_s2e_model, pipeline_s2e_tok,
        normalized, num_candidates=6,
        forced_bos=forced_bos_en, batch_size=8
    )

    print("Step 3: beam vote + span fusion...")
    results = []
    for i in range(len(rural_texts)):
        norm       = normalized[i]
        hel_score  = hel_scores[i][0]
        pipe_score = pipe_scores[i][0]

        if hel_score >= pipe_score:
            base            = hel_cands[i][0]
            base_sc         = hel_score
            other_cands     = pipe_cands[i]
            other_scores    = pipe_scores[i]
            score_tokenizer = hel_tok
            score_model_    = hel_model
            forced_bos_arg  = None
        else:
            base            = pipe_cands[i][0]
            base_sc         = pipe_score
            other_cands     = hel_cands[i]
            other_scores    = hel_scores[i]
            score_tokenizer = pipeline_s2e_tok
            score_model_    = pipeline_s2e_model
            forced_bos_arg  = forced_bos_en

        try:
            fused = fast_fusion(
                norm, base, base_sc,
                other_cands, other_scores,
                score_tokenizer, score_model_,
                forced_bos_arg
            )
        except RuntimeError:
            fused = base

        results.append(fused)

        if i % 100 == 0:
            print(f"  fusion {i}/{len(rural_texts)}", flush=True)
        torch.cuda.empty_cache()

    return results


print("\nRunning full pipeline on test set...")
predictions = translate_rural_to_english(texts)

b = bleu.corpus_score(predictions, [references_en])
c = chrf.corpus_score(predictions, [references_en])
print(f"\nFull pipeline: BLEU: {b.score:.2f} | chrF: {c.score:.2f}")

print("\nExample translations:")
for i in range(5):
    print(f"\n[{i}] Rural:  {test_df.iloc[i]['rural']}")
    print(f"     English: {predictions[i]}")
    print(f"     Ref:     {references_en[i]}")

In [ ]:
# EXAMPLE TRANSLATIONS:

my_sentences = [
    "তোমার আব্বায় ক্যামন আছে?",
    "মোর বড় বুইনের আইজগো মন ভালো নাই",
]
translations = translate_rural_to_english(my_sentences)
for rural, english in zip(my_sentences, translations):
    print(f"{rural} → {english}")